# Proposed Quantum Transformer HAR Demo

This notebook contains the cleaned proposed-model demonstration based on the original 40-epoch HAR experiment notebook. It compares a Quantum Transformer model against several 1D neural-network baselines across HAR datasets. Datasets are not included; configure `DATA_DIR = "../data"` locally before running.


In [ ]:
#!/usr/bin/env python3
"""
WISDM HAR with Quantum Transformer (QTModel):

- Stronger 1D CNN backbone (64-d features)
- Stacked quantum blocks:
    * Linear -> angles for Q/K/V
    * TorchQuantum QFM for Q, K, V
    * Self-attention after each QFM block (element-wise Q·K gating of V)
    * Transformer-style FFN + LayerNorm with residuals
- Final quantum gate:
    * extra QFM on top of last features
    * fused [features + gate] -> MLP classifier

Two levels of quantum interaction:
  (1) Inside each block
  (2) Final gate

Runs 6-fold CV on balanced WISDM.
"""

import os
import urllib.request
from pathlib import Path

import psutil
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR

from ptflops import get_model_complexity_info

from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve
)
from sklearn.preprocessing import label_binarize

import matplotlib.pyplot as plt
from tqdm import trange

import torchquantum as tq

# -----------------------------------------------------------------------------
# GLOBAL CONFIG
# -----------------------------------------------------------------------------

QSize = 4                # number of quantum wires
Q_LAYERS = 4             # depth of quantum feature map
Q_BLOCKS = 4             # number of quantum blocks
EPOCHS = 40              # training epochs per fold
BATCH_SIZE = 256         # batch size
N_SPLITS = 6             # K-fold CV

WINDOW_SIZE = 200        # WISDM window length
STEP_SIZE = 100          # WISDM step size

DATA_DIR = Path(os.environ.get("DATA_DIR", "../data"))
DATA_DIR.mkdir(exist_ok=True)

WISDM_URL = (
    "https://raw.githubusercontent.com/soham97/WISD_HAR_files/master/"
    "WISDM_ar_v1.1_raw.txt"
)
WISDM_PATH = DATA_DIR / "WISDM_ar_v1.1_raw.txt"

# -----------------------------------------------------------------------------
# 1) DEVICE SELECTION (fixed cuda:0 if available)
# -----------------------------------------------------------------------------

if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
    print("Using device: cuda:0")
else:
    DEVICE = torch.device("cpu")
    print("CUDA not available, using CPU")

ram = psutil.virtual_memory()
print(f"RAM Usage: {(ram.total-ram.available)/1e9:.1f}/{ram.total/1e9:.1f} GiB ({ram.percent}%)\n")

# -----------------------------------------------------------------------------
# 2) WISDM LOADING & PREPROCESSING
# -----------------------------------------------------------------------------

def download_wisdm():
    if WISDM_PATH.exists():
        print(f"[INFO] Found WISDM file at {WISDM_PATH}")
        return
    print(f"[INFO] Downloading WISDM from {WISDM_URL} ...")
    urllib.request.urlretrieve(WISDM_URL, WISDM_PATH)
    print("[INFO] Download complete.")

def load_wisdm_dataframe(path: Path) -> pd.DataFrame:
    """Load and clean WISDM dataset."""
    print("[INFO] Loading WISDM data into DataFrame...")
    columns = ["user", "activity", "timestamp", "x", "y", "z"]
    df = pd.read_csv(
        path,
        header=None,
        names=columns,
        engine="python",
        on_bad_lines="skip",
    )

    df = df.dropna()
    df["z"] = df["z"].astype(str).str.replace(";", "", regex=False)
    df["z"] = df["z"].astype(float)
    df = df[df["timestamp"] != 0]
    df = df.sort_values(by=["user", "timestamp"], ignore_index=True)

    print("[INFO] Cleaned WISDM shape:", df.shape)
    return df

def segment_wisdm(df: pd.DataFrame,
                  window_size: int = WINDOW_SIZE,
                  step_size: int = STEP_SIZE):
    """Segment accelerometer data into windows and compute magnitude."""
    print("[INFO] Segmenting time series into windows...")
    X_segments = []
    y_labels = []

    users = df["user"].unique()
    for user in users:
        df_user = df[df["user"] == user]
        activities = df_user["activity"].unique()

        for act in activities:
            df_ua = df_user[df_user["activity"] == act]
            signal = df_ua[["x", "y", "z"]].values  # [T, 3]

            for start in range(0, len(signal) - window_size, step_size):
                end = start + window_size
                window = signal[start:end]  # [window_size, 3]
                mag = np.sqrt((window ** 2).sum(axis=1))  # [window_size]
                X_segments.append(mag)
                y_labels.append(act)

    X_segments = np.array(X_segments, dtype=np.float32)
    y_labels = np.array(y_labels)
    print("[INFO] Segmented X shape:", X_segments.shape)
    print("[INFO] Label distribution (raw):\n", pd.Series(y_labels).value_counts())
    return X_segments, y_labels

def balance_by_min_class(X: np.ndarray, y_int: np.ndarray, seed: int = 42):
    """
    Downsample all classes to the smallest class size.
    """
    rng = np.random.default_rng(seed)
    unique_classes = np.unique(y_int)
    indices_per_class = {c: np.where(y_int == c)[0] for c in unique_classes}
    class_sizes = {c: len(idx) for c, idx in indices_per_class.items()}
    min_count = min(class_sizes.values())

    print("[INFO] Class sizes before balancing:", class_sizes)
    print("[INFO] Using min_count =", min_count, "samples per class")

    balanced_indices_list = []
    for c in unique_classes:
        idxs = indices_per_class[c]
        if len(idxs) > min_count:
            chosen = rng.choice(idxs, size=min_count, replace=False)
        else:
            chosen = idxs
        balanced_indices_list.append(chosen)

    balanced_indices = np.concatenate(balanced_indices_list)
    rng.shuffle(balanced_indices)

    balanced_counts = np.bincount(y_int[balanced_indices])
    print("[INFO] Class sizes after balancing:",
          dict(zip(unique_classes, balanced_counts)))

    return X[balanced_indices], y_int[balanced_indices]

# -----------------------------------------------------------------------------
# 3) DATASET
# -----------------------------------------------------------------------------

class WISDMDataset(Dataset):
    """
    WISDM window-level dataset for time-series magnitude signals.
    Each sample: x -> [1, WINDOW_SIZE], y -> integer label.
    """

    def __init__(self, X_windows: np.ndarray, y_int: np.ndarray):
        X_tensor = torch.tensor(X_windows, dtype=torch.float32)
        y_tensor = torch.tensor(y_int, dtype=torch.long)

        # Per-window normalization
        means = X_tensor.mean(dim=1, keepdim=True)
        stds = X_tensor.std(dim=1, keepdim=True) + 1e-8
        X_tensor = (X_tensor - means) / stds

        self.X = X_tensor
        self.y = y_tensor

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx].unsqueeze(0)  # [1, L]
        y = self.y[idx]
        return x, y

# -----------------------------------------------------------------------------
# 4) STRONGER CNN BACKBONE
# -----------------------------------------------------------------------------

class EnhancedCNN1D(nn.Module):
    """
    Strong 1D CNN backbone for QTModel.
    Input:  [B, 1, L]
    Output: [B, 64]
    """
    def __init__(self, in_ch: int = 1):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv1d(in_ch, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2)   # L -> L/2
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2)   # L/2 -> L/4
        )
        self.block3 = nn.Sequential(
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)  # [B,64,1]
        )
        self.dropout = nn.Dropout(p=0.2)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.dropout(x)
        return x.flatten(1)  # [B, 64]

# -----------------------------------------------------------------------------
# 5) TORCHQUANTUM FEATURE MAP
# -----------------------------------------------------------------------------

class QuantumFeatureMapTQ(tq.QuantumModule):
    """
    TorchQuantum-based feature map:
     - Angle embedding via RY
     - Ring entangling CNOT chain
     - Measure all Z
    """
    def __init__(self, n_wires=4, n_layers=4):
        super().__init__()
        self.n_wires  = n_wires
        self.n_layers = n_layers
        self.measure  = tq.MeasureAll(tq.PauliZ)

    def forward(self, qdev, angles):
        # angles: (batch, n_wires)
        for w in range(self.n_wires):
            qdev.ry(wires=w, params=angles[:, w], static=self.static_mode)
        for _ in range(self.n_layers):
            for i in range(self.n_wires - 1):
                qdev.cnot(wires=[i, i+1])
            qdev.cnot(wires=[self.n_wires-1, 0])
        return self.measure(qdev)  # (batch, n_wires)

# -----------------------------------------------------------------------------
# 6) QUANTUM BLOCK WITH SELF-ATTENTION + FFN
# -----------------------------------------------------------------------------

class QBlockQT(tq.QuantumModule):
    """
    Single quantum block for QTModel:

      Input:  x ∈ R^{B×D}
      Steps:
        - Linear -> angles for Q, K, V (each in R^{B×n_wires})
        - QFM for Q, K, V (TorchQuantum)
        - Self-attention:
            q = Wq(q_out), k = Wk(k_out), v = Wv(v_out) ∈ R^{B×D}
            scores = softmax( (q * k) / sqrt(D) )   # element-wise
            attn_out = scores * v
        - Residual: z = x + attn_out
        - LayerNorm + Transformer-style FFN + residual + LayerNorm
    """
    def __init__(self, dim=64, n_wires=4, n_layers=4, ff_mult=2, dropout=0.1):
        super().__init__()
        self.dim     = dim
        self.n_wires = n_wires

        # QFM for Q, K, V
        self.q_fm = QuantumFeatureMapTQ(n_wires, n_layers)
        self.k_fm = QuantumFeatureMapTQ(n_wires, n_layers)
        self.v_fm = QuantumFeatureMapTQ(n_wires, n_layers)

        # Map input features -> Q/K/V angles
        self.reduce = nn.Linear(dim, 3 * n_wires)

        # Project Q/K/V outputs up to feature dim
        self.q_proj = nn.Linear(n_wires, dim)
        self.k_proj = nn.Linear(n_wires, dim)
        self.v_proj = nn.Linear(n_wires, dim)

        # Transformer-style FFN
        self.ff = nn.Sequential(
            nn.Linear(dim, ff_mult * dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_mult * dim, dim),
        )

        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        # x: [B, D]
        B, D = x.shape

        # Classical to quantum angles
        r = self.reduce(x)                      # [B, 3 * n_wires]
        q_ang, k_ang, v_ang = r.chunk(3, dim=1) # each [B, n_wires]

        # Quantum device
        qdev = tq.QuantumDevice(
            n_wires=self.n_wires,
            bsz=B,
            device=x.device
        )

        # QFM for Q, K, V
        q_out = self.q_fm(qdev, q_ang)  # [B, n_wires]
        k_out = self.k_fm(qdev, k_ang)
        v_out = self.v_fm(qdev, v_ang)

        # Project to feature dimension
        q = self.q_proj(q_out)  # [B, D]
        k = self.k_proj(k_out)  # [B, D]
        v = self.v_proj(v_out)  # [B, D]

        # Self-attention (element-wise gating)
        scale = float(D) ** 0.5
        scores = torch.softmax((q * k) / scale, dim=-1)  # [B, D]
        attn_out = scores * v                            # [B, D]

        # Residual + FFN + norms
        z = x + attn_out          # first residual
        z_norm = self.norm1(z)
        ff_out = self.ff(z_norm)
        y = self.norm2(z_norm + ff_out)  # second residual
        return y

# -----------------------------------------------------------------------------
# 7) QTModel: CNN -> stacked QBlocks -> quantum gate -> classifier
# -----------------------------------------------------------------------------

class QTModel(nn.Module):
    """
    QTModel:
      - EnhancedCNN1D -> 64-d feature
      - Multiple QBlockQT (each with QFM+SelfAttention+FFN)
      - Final quantum gate:
          angles = W_gate(e)
          q_gate = QFM(angles)
          fused = [e, q_gate]
          MLP classifier
    """
    def __init__(self, num_cls, in_ch=1,
                 n_blocks=Q_BLOCKS, n_wires=QSize, n_layers=Q_LAYERS,
                 dim=64, ff_mult=2, dropout=0.1):
        super().__init__()

        self.dim      = dim
        self.n_wires  = n_wires
        self.cnn      = EnhancedCNN1D(in_ch)

        # Stacked quantum blocks
        self.blocks = nn.ModuleList([
            QBlockQT(dim=dim, n_wires=n_wires, n_layers=n_layers,
                     ff_mult=ff_mult, dropout=dropout)
            for _ in range(n_blocks)
        ])

        # Final quantum gate
        self.gate_linear = nn.Linear(dim, n_wires)
        self.gate_qfm    = QuantumFeatureMapTQ(n_wires, n_layers)

        # Classifier on fused [features + quantum gate]
        self.cls_head = nn.Sequential(
            nn.LayerNorm(dim + n_wires),
            nn.Linear(dim + n_wires, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, num_cls),
        )

    def forward(self, x):
        # CNN backbone
        e = self.cnn(x)  # [B, 64]

        # Quantum blocks (each has QFM + self-attn + FFN)
        for block in self.blocks:
            e = block(e)  # [B, 64]

        # Final quantum gate
        angles = self.gate_linear(e)  # [B, n_wires]
        qdev = tq.QuantumDevice(
            n_wires=self.n_wires,
            bsz=angles.size(0),
            device=angles.device
        )
        q_gate = self.gate_qfm(qdev, angles)  # [B, n_wires]

        fused = torch.cat([e, q_gate], dim=1)  # [B, 64 + n_wires]
        logits = self.cls_head(fused)
        return F.log_softmax(logits, dim=1)

# -----------------------------------------------------------------------------
# 8) TRAIN / EVAL HELPERS
# -----------------------------------------------------------------------------

def process_target(y: torch.Tensor) -> torch.Tensor:
    if y.dim() > 1:
        if y.size(1) > 1:
            y = y.argmax(dim=1)
        else:
            y = y.squeeze(1)
    return y.long()

def train_one(model, loader, opt):
    model.train()
    losses = []
    for x, y in loader:
        x = x.to(DEVICE)
        y = process_target(y.to(DEVICE))

        out = model(x)
        loss = F.nll_loss(out, y)

        opt.zero_grad()
        loss.backward()
        opt.step()

        losses.append(loss.item())
    return float(np.mean(losses)) if losses else 0.0

def eval_one(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = process_target(y.to(DEVICE))

            out  = model(x)
            prob = out.exp().cpu().numpy()

            ys.append(y.cpu().numpy())
            ps.append(prob)

    ys = np.concatenate(ys)
    ps = np.concatenate(ps)

    preds = ps.argmax(axis=1)
    acc  = accuracy_score(ys, preds)
    f1   = f1_score(ys, preds, average="weighted")
    prec = precision_score(ys, preds, average="weighted")
    rec  = recall_score(ys, preds, average="weighted")

    if ps.shape[1] == 2:
        auc = roc_auc_score(ys, ps[:, 1])
        fpr, tpr, _ = roc_curve(ys, ps[:, 1], pos_label=1)
    else:
        ys_bin = label_binarize(ys, classes=list(range(ps.shape[1])))
        auc    = roc_auc_score(ys_bin, ps, average="micro", multi_class="ovo")
        fpr, tpr, _ = roc_curve(ys_bin.ravel(), ps.ravel())

    return acc, f1, prec, rec, auc, (fpr, tpr)

# -----------------------------------------------------------------------------
# 9) 6-FOLD CV FOR QTModel ONLY
# -----------------------------------------------------------------------------

def run_cv_qt(num_cls, dataset):
    N = len(dataset)
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=0)

    metrics = {"acc": [], "f1": [], "prec": [], "rec": [], "auc": []}
    all_rocs = []
    loss_hist = []

    print(f"\n====== Running 6-fold CV for QTModel ======")

    for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(N)), start=1):
        print(f"\n--- QTModel Fold {fold}/{N_SPLITS} ---")

        train_loader = DataLoader(
            Subset(dataset, train_idx),
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=4,
            pin_memory=True
        )
        val_loader = DataLoader(
            Subset(dataset, val_idx),
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=4,
            pin_memory=True
        )

        model = QTModel(num_cls=num_cls, in_ch=1).to(DEVICE)
        opt   = optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-4)
        sch   = CosineAnnealingLR(opt, T_max=EPOCHS)

        fold_losses = []
        for _ in trange(EPOCHS, desc=f"QTModel-F{fold}", leave=False):
            loss = train_one(model, train_loader, opt)
            sch.step()
            fold_losses.append(loss)
        loss_hist.append(fold_losses)

        a, f, p, r, auc_val, roc_pts = eval_one(model, val_loader)
        print(f" Fold {fold}: Acc={a:.4f} F1={f:.4f} Prec={p:.4f} Rec={r:.4f} AUC={auc_val:.4f}")
        for k, v in zip(("acc", "f1", "prec", "rec", "auc"),
                        (a, f, p, r, auc_val)):
            metrics[k].append(v)
        all_rocs.append(roc_pts)

    return metrics, all_rocs, loss_hist

# -----------------------------------------------------------------------------
# 10) MAIN
# -----------------------------------------------------------------------------

def main():
    # ---- WISDM data ----
    download_wisdm()
    df = load_wisdm_dataframe(WISDM_PATH)
    X_segments, y_labels = segment_wisdm(df)

    labels_cat = pd.Series(y_labels).astype("category")
    class_names = list(labels_cat.cat.categories)
    y_int = labels_cat.cat.codes.to_numpy().astype(np.int64)

    print("[INFO] WISDM classes:", class_names)
    print("[INFO] Raw class counts:", np.bincount(y_int))

    # Balance dataset
    X_bal, y_bal = balance_by_min_class(X_segments, y_int, seed=42)
    dataset = WISDMDataset(X_bal, y_bal)
    N = len(dataset)
    print(f"[INFO] Balanced dataset size: {N}")

    num_cls = len(class_names)

    # Complexity estimate for QTModel
    dims = (1, WINDOW_SIZE)
    tmp_qt = QTModel(num_cls=num_cls, in_ch=1).to(DEVICE)
    try:
        macs_qt, params_qt = get_model_complexity_info(
            tmp_qt, dims,
            as_strings=True,
            print_per_layer_stat=False
        )
    except Exception as e:
        print(f"[WARN] ptflops failed on QTModel: {e}")
        macs_qt, params_qt = "N/A", "N/A"

    print(f"[INFO] QTModel params: {params_qt}, FLOPs: {macs_qt}")

    # 6-fold CV
    metrics, all_rocs, loss_hist = run_cv_qt(num_cls, dataset)

    # Summary
    gpu_mem = None
    if DEVICE.type == "cuda":
        gpu_mem = (torch.cuda.memory_allocated()/1e6,
                   torch.cuda.memory_reserved()/1e6)

    summary = {
        "model":        "QTModel",
        "dataset":      "WISDM",
        "acc_mean±sd":  f"{np.mean(metrics['acc']):.4f}±{np.std(metrics['acc']):.4f}",
        "f1_mean±sd":   f"{np.mean(metrics['f1']):.4f}±{np.std(metrics['f1']):.4f}",
        "prec_mean±sd": f"{np.mean(metrics['prec']):.4f}±{np.std(metrics['prec']):.4f}",
        "rec_mean±sd":  f"{np.mean(metrics['rec']):.4f}±{np.std(metrics['rec']):.4f}",
        "auc_mean±sd":  f"{np.mean(metrics['auc']):.4f}±{np.std(metrics['auc']):.4f}",
        "params":       params_qt,
        "FLOPs":        macs_qt,
        "GPU_mem_MB":   gpu_mem,
    }

    df_sum = pd.DataFrame([summary])
    print("\n[SUMMARY QTModel]")
    print(df_sum.to_markdown(index=False))

    # Optional plots
    pd.DataFrame(metrics).boxplot(column=["acc", "f1", "prec", "rec", "auc"])
    plt.title("QTModel 6-Fold CV Metrics")
    plt.ylabel("Score")
    plt.grid(True, alpha=0.3)
    plt.show()

    for curve in loss_hist:
        plt.plot(curve, alpha=0.5)
    plt.title("QTModel Training Loss per Fold")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True, alpha=0.3)
    plt.show()

    plt.figure()
    for fpr, tpr in all_rocs:
        plt.plot(fpr, tpr, alpha=0.6)
    plt.title("QTModel ROC Curves")
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.grid(True, alpha=0.3)
    plt.show()

if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
"""
WISDM HAR with QTModel vs Baseline 1D CNN models.

QTModel:
 - Stronger 1D CNN backbone (64-d features)
 - Stacked quantum blocks:
     * Linear -> angles for Q/K/V
     * TorchQuantum QFM for Q, K, V
     * Self-attention after each QFM block (element-wise Q·K gating of V)
     * Transformer-style FFN + LayerNorm with residuals
 - Final quantum gate (extra QFM) on top of last features
 - Fused [features + gate] -> MLP classifier

Baselines:
 - ResNet1D (DeepILS)
 - EfficientNetB0_1D
 - MnasNet_1D
 - MobileNet_1D
 - MobileNetV2_1D
 - TSLANet

Outputs:
 - 6-fold CV metrics for each model
 - Training Loss Curves (mean over folds)
 - ROC curves (Testing Accuracy Curve with AUC)
 - Classification HeatMaps (confusion matrices in %, per model)
"""

import os
import urllib.request
from pathlib import Path

import psutil
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR

from ptflops import get_model_complexity_info

from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, confusion_matrix
)
from sklearn.preprocessing import label_binarize

import matplotlib.pyplot as plt
from tqdm import trange

import torchquantum as tq

# -----------------------------------------------------------------------------
# GLOBAL CONFIG
# -----------------------------------------------------------------------------

QSize = 4                # number of quantum wires
Q_LAYERS = 4             # depth of quantum feature map
Q_BLOCKS = 4             # number of quantum blocks
EPOCHS = 40              # training epochs per fold
BATCH_SIZE = 256         # batch size
N_SPLITS = 6             # K-fold CV

WINDOW_SIZE = 200        # WISDM window length
STEP_SIZE = 100          # WISDM step size

DATA_DIR = Path(os.environ.get("DATA_DIR", "../data"))
DATA_DIR.mkdir(exist_ok=True)

WISDM_URL = (
    "https://raw.githubusercontent.com/soham97/WISD_HAR_files/master/"
    "WISDM_ar_v1.1_raw.txt"
)
WISDM_PATH = DATA_DIR / "WISDM_ar_v1.1_raw.txt"

# -----------------------------------------------------------------------------
# 1) DEVICE SELECTION (fixed cuda:0 if available)
# -----------------------------------------------------------------------------

if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
    print("Using device: cuda:0")
else:
    DEVICE = torch.device("cpu")
    print("CUDA not available, using CPU")

ram = psutil.virtual_memory()
print(f"RAM Usage: {(ram.total-ram.available)/1e9:.1f}/{ram.total/1e9:.1f} GiB ({ram.percent}%)\n")

# -----------------------------------------------------------------------------
# 2) WISDM LOADING & PREPROCESSING
# -----------------------------------------------------------------------------

def download_wisdm():
    if WISDM_PATH.exists():
        print(f"[INFO] Found WISDM file at {WISDM_PATH}")
        return
    print(f"[INFO] Downloading WISDM from {WISDM_URL} ...")
    urllib.request.urlretrieve(WISDM_URL, WISDM_PATH)
    print("[INFO] Download complete.")

def load_wisdm_dataframe(path: Path) -> pd.DataFrame:
    """Load and clean WISDM dataset."""
    print("[INFO] Loading WISDM data into DataFrame...")
    columns = ["user", "activity", "timestamp", "x", "y", "z"]
    df = pd.read_csv(
        path,
        header=None,
        names=columns,
        engine="python",
        on_bad_lines="skip",
    )

    df = df.dropna()
    df["z"] = df["z"].astype(str).str.replace(";", "", regex=False)
    df["z"] = df["z"].astype(float)
    df = df[df["timestamp"] != 0]
    df = df.sort_values(by=["user", "timestamp"], ignore_index=True)

    print("[INFO] Cleaned WISDM shape:", df.shape)
    return df

def segment_wisdm(df: pd.DataFrame,
                  window_size: int = WINDOW_SIZE,
                  step_size: int = STEP_SIZE):
    """Segment accelerometer data into windows and compute magnitude."""
    print("[INFO] Segmenting time series into windows...")
    X_segments = []
    y_labels = []

    users = df["user"].unique()
    for user in users:
        df_user = df[df["user"] == user]
        activities = df_user["activity"].unique()

        for act in activities:
            df_ua = df_user[df_user["activity"] == act]
            signal = df_ua[["x", "y", "z"]].values  # [T, 3]

            for start in range(0, len(signal) - window_size, step_size):
                end = start + window_size
                window = signal[start:end]  # [window_size, 3]
                mag = np.sqrt((window ** 2).sum(axis=1))  # [window_size]
                X_segments.append(mag)
                y_labels.append(act)

    X_segments = np.array(X_segments, dtype=np.float32)
    y_labels = np.array(y_labels)
    print("[INFO] Segmented X shape:", X_segments.shape)
    print("[INFO] Label distribution (raw):\n", pd.Series(y_labels).value_counts())
    return X_segments, y_labels

def balance_by_min_class(X: np.ndarray, y_int: np.ndarray, seed: int = 42):
    """
    Downsample all classes to the smallest class size.
    """
    rng = np.random.default_rng(seed)
    unique_classes = np.unique(y_int)
    indices_per_class = {c: np.where(y_int == c)[0] for c in unique_classes}
    class_sizes = {c: len(idx) for c, idx in indices_per_class.items()}
    min_count = min(class_sizes.values())

    print("[INFO] Class sizes before balancing:", class_sizes)
    print("[INFO] Using min_count =", min_count, "samples per class")

    balanced_indices_list = []
    for c in unique_classes:
        idxs = indices_per_class[c]
        if len(idxs) > min_count:
            chosen = rng.choice(idxs, size=min_count, replace=False)
        else:
            chosen = idxs
        balanced_indices_list.append(chosen)

    balanced_indices = np.concatenate(balanced_indices_list)
    rng.shuffle(balanced_indices)

    balanced_counts = np.bincount(y_int[balanced_indices])
    print("[INFO] Class sizes after balancing:",
          dict(zip(unique_classes, balanced_counts)))

    return X[balanced_indices], y_int[balanced_indices]

# -----------------------------------------------------------------------------
# 3) DATASET
# -----------------------------------------------------------------------------

class WISDMDataset(Dataset):
    """
    WISDM window-level dataset for time-series magnitude signals.
    Each sample: x -> [1, WINDOW_SIZE], y -> integer label.
    """

    def __init__(self, X_windows: np.ndarray, y_int: np.ndarray):
        X_tensor = torch.tensor(X_windows, dtype=torch.float32)
        y_tensor = torch.tensor(y_int, dtype=torch.long)

        # Per-window normalization
        means = X_tensor.mean(dim=1, keepdim=True)
        stds = X_tensor.std(dim=1, keepdim=True) + 1e-8
        X_tensor = (X_tensor - means) / stds

        self.X = X_tensor
        self.y = y_tensor

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx].unsqueeze(0)  # [1, L]
        y = self.y[idx]
        return x, y

# -----------------------------------------------------------------------------
# 4) STRONGER CNN BACKBONE
# -----------------------------------------------------------------------------

class EnhancedCNN1D(nn.Module):
    """
    Strong 1D CNN backbone for QTModel.
    Input:  [B, 1, L]
    Output: [B, 64]
    """
    def __init__(self, in_ch: int = 1):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv1d(in_ch, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2)   # L -> L/2
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2)   # L/2 -> L/4
        )
        self.block3 = nn.Sequential(
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)  # [B,64,1]
        )
        self.dropout = nn.Dropout(p=0.2)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.dropout(x)
        return x.flatten(1)  # [B, 64]

# -----------------------------------------------------------------------------
# 5) TORCHQUANTUM FEATURE MAP
# -----------------------------------------------------------------------------

class QuantumFeatureMapTQ(tq.QuantumModule):
    """
    TorchQuantum-based feature map:
     - Angle embedding via RY
     - Ring entangling CNOT chain
     - Measure all Z
    """
    def __init__(self, n_wires=4, n_layers=4):
        super().__init__()
        self.n_wires  = n_wires
        self.n_layers = n_layers
        self.measure  = tq.MeasureAll(tq.PauliZ)

    def forward(self, qdev, angles):
        # angles: (batch, n_wires)
        for w in range(self.n_wires):
            qdev.ry(wires=w, params=angles[:, w], static=self.static_mode)
        for _ in range(self.n_layers):
            for i in range(self.n_wires - 1):
                qdev.cnot(wires=[i, i+1])
            qdev.cnot(wires=[self.n_wires-1, 0])
        return self.measure(qdev)  # (batch, n_wires)

# -----------------------------------------------------------------------------
# 6) QUANTUM BLOCK WITH SELF-ATTENTION + FFN
# -----------------------------------------------------------------------------

class QBlockQT(tq.QuantumModule):
    """
    Single quantum block for QTModel:

      Input:  x ∈ R^{B×D}
      Steps:
        - Linear -> angles for Q, K, V (each in R^{B×n_wires})
        - QFM for Q, K, V (TorchQuantum)
        - Self-attention:
            q = Wq(q_out), k = Wk(k_out), v = Wv(v_out) ∈ R^{B×D}
            scores = softmax( (q * k) / sqrt(D) )   # element-wise
            attn_out = scores * v
        - Residual: z = x + attn_out
        - LayerNorm + Transformer-style FFN + residual + LayerNorm
    """
    def __init__(self, dim=64, n_wires=4, n_layers=4, ff_mult=2, dropout=0.1):
        super().__init__()
        self.dim     = dim
        self.n_wires = n_wires

        # QFM for Q, K, V
        self.q_fm = QuantumFeatureMapTQ(n_wires, n_layers)
        self.k_fm = QuantumFeatureMapTQ(n_wires, n_layers)
        self.v_fm = QuantumFeatureMapTQ(n_wires, n_layers)

        # Map input features -> Q/K/V angles
        self.reduce = nn.Linear(dim, 3 * n_wires)

        # Project Q/K/V outputs up to feature dim
        self.q_proj = nn.Linear(n_wires, dim)
        self.k_proj = nn.Linear(n_wires, dim)
        self.v_proj = nn.Linear(n_wires, dim)

        # Transformer-style FFN
        self.ff = nn.Sequential(
            nn.Linear(dim, ff_mult * dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_mult * dim, dim),
        )

        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        # x: [B, D]
        B, D = x.shape

        # Classical to quantum angles
        r = self.reduce(x)                      # [B, 3 * n_wires]
        q_ang, k_ang, v_ang = r.chunk(3, dim=1) # each [B, n_wires]

        # Quantum device
        qdev = tq.QuantumDevice(
            n_wires=self.n_wires,
            bsz=B,
            device=x.device
        )

        # QFM for Q, K, V
        q_out = self.q_fm(qdev, q_ang)  # [B, n_wires]
        k_out = self.k_fm(qdev, k_ang)
        v_out = self.v_fm(qdev, v_ang)

        # Project to feature dimension
        q = self.q_proj(q_out)  # [B, D]
        k = self.k_proj(k_out)  # [B, D]
        v = self.v_proj(v_out)  # [B, D]

        # Self-attention (element-wise gating)
        scale = float(D) ** 0.5
        scores = torch.softmax((q * k) / scale, dim=-1)  # [B, D]
        attn_out = scores * v                            # [B, D]

        # Residual + FFN + norms
        z = x + attn_out          # first residual
        z_norm = self.norm1(z)
        ff_out = self.ff(z_norm)
        y = self.norm2(z_norm + ff_out)  # second residual
        return y

# -----------------------------------------------------------------------------
# 7) QTModel: CNN -> stacked QBlocks -> quantum gate -> classifier
# -----------------------------------------------------------------------------

class QTModel(nn.Module):
    """
    QTModel:
      - EnhancedCNN1D -> 64-d feature
      - Multiple QBlockQT (each with QFM+SelfAttention+FFN)
      - Final quantum gate:
          angles = W_gate(e)
          q_gate = QFM(angles)
          fused = [e, q_gate]
          MLP classifier
    """
    def __init__(self, num_cls, in_ch=1,
                 n_blocks=Q_BLOCKS, n_wires=QSize, n_layers=Q_LAYERS,
                 dim=64, ff_mult=2, dropout=0.1):
        super().__init__()

        self.dim      = dim
        self.n_wires  = n_wires
        self.cnn      = EnhancedCNN1D(in_ch)

        # Stacked quantum blocks
        self.blocks = nn.ModuleList([
            QBlockQT(dim=dim, n_wires=n_wires, n_layers=n_layers,
                     ff_mult=ff_mult, dropout=dropout)
            for _ in range(n_blocks)
        ])

        # Final quantum gate
        self.gate_linear = nn.Linear(dim, n_wires)
        self.gate_qfm    = QuantumFeatureMapTQ(n_wires, n_layers)

        # Classifier on fused [features + quantum gate]
        self.cls_head = nn.Sequential(
            nn.LayerNorm(dim + n_wires),
            nn.Linear(dim + n_wires, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, num_cls),
        )

    def forward(self, x):
        # CNN backbone
        e = self.cnn(x)  # [B, 64]

        # Quantum blocks (each has QFM + self-attn + FFN)
        for block in self.blocks:
            e = block(e)  # [B, 64]

        # Final quantum gate
        angles = self.gate_linear(e)  # [B, n_wires]
        qdev = tq.QuantumDevice(
            n_wires=self.n_wires,
            bsz=angles.size(0),
            device=angles.device
        )
        q_gate = self.gate_qfm(qdev, angles)  # [B, n_wires]

        fused = torch.cat([e, q_gate], dim=1)  # [B, 64 + n_wires]
        logits = self.cls_head(fused)          # raw logits
        return logits

# -----------------------------------------------------------------------------
# 8) BASELINE MODELS
# -----------------------------------------------------------------------------

def conv_dw(in_planes, out_planes, kernel_size, stride=1, dilation=1):
    return nn.Sequential(
        nn.Conv1d(in_planes, in_planes, kernel_size=kernel_size,
                  stride=stride, padding=(kernel_size//2)*dilation, dilation=dilation,
                  groups=in_planes, bias=False),
        nn.Conv1d(in_planes, out_planes, kernel_size=1, bias=False)
    )

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool1d(1)
        self.max = nn.AdaptiveMaxPool1d(1)
        red = max(1, in_planes // ratio)
        self.fc = nn.Sequential(
            nn.Conv1d(in_planes, red, 1, bias=False), nn.ReLU(),
            nn.Conv1d(red, in_planes, 1, bias=False)
        )
        self.sig = nn.Sigmoid()
    def forward(self, x):
        return self.sig(self.fc(self.avg(x)) + self.fc(self.max(x)))

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv1 = nn.Conv1d(2, 1, k, padding=k//2, bias=False)
        self.sig = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sig(self.conv1(torch.cat([avg_out, max_out], dim=1)))

class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, out_planes, k, stride=1, dilation=1, downsample=None):
        super().__init__()
        self.conv1 = conv_dw(in_planes, out_planes, k, stride, dilation)
        self.bn1 = nn.BatchNorm1d(out_planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv_dw(out_planes, out_planes, k)
        self.bn2 = nn.BatchNorm1d(out_planes)
        self.ca = ChannelAttention(out_planes)
        self.sa = SpatialAttention()
        self.downsample = downsample
    def forward(self, x):
        res = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.ca(out)*out
        out = self.sa(out)*out
        if self.downsample is not None:
            res = self.downsample(x)
        return self.relu(out + res)

class DeepILS(nn.Module):
    def __init__(self, num_inputs=1, num_classes=6, block=BasicBlock1D, group_sizes=(2,2,2,2), base=64, k=3):
        super().__init__()
        self.inp = nn.Sequential(
            nn.Conv1d(num_inputs, base, 5, 2, 3, bias=False),
            nn.BatchNorm1d(base),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(3,2,1)
        )
        self.inplanes = base
        self.groups = nn.ModuleList()
        planes = [base*(2**i) for i in range(len(group_sizes))]
        strides = [1] + [2]*(len(group_sizes)-1)
        for i, blocks in enumerate(group_sizes):
            pl = planes[i]
            st = strides[i]
            down = None
            if st != 1 or self.inplanes != pl*block.expansion:
                down = nn.Sequential(
                    nn.Conv1d(self.inplanes, pl*block.expansion, 1, st, bias=False),
                    nn.BatchNorm1d(pl*block.expansion)
                )
            layers = [block(self.inplanes, pl, k, stride=st, downsample=down)]
            self.inplanes = pl*block.expansion
            for _ in range(1, blocks):
                layers.append(block(self.inplanes, pl, k))
            self.groups.append(nn.Sequential(*layers))
        self.groups = nn.Sequential(*self.groups)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(planes[-1]*block.expansion, num_classes)
        )
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
    def forward(self, x):
        x = self.inp(x)
        x = self.groups(x)
        return self.head(x)

class ResNet1D(DeepILS):
    pass

class Swish(nn.Module):
    def forward(self, x): return x*torch.sigmoid(x)

def Conv_3x3(inp, oup, s):
    return nn.Sequential(
        nn.Conv1d(inp,oup,3,s,1,bias=False),
        nn.BatchNorm1d(oup),
        Swish()
    )

def Conv_1x1(inp, oup):
    return nn.Sequential(
        nn.Conv1d(inp,oup,1,1,0,bias=False),
        nn.BatchNorm1d(oup),
        Swish()
    )

class InvertedResidual_Eff(nn.Module):
    def __init__(self, inp, oup, s, t, k):
        super().__init__()
        self.use=(s==1 and inp==oup)
        hid=inp*t
        self.conv=nn.Sequential(
            nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), Swish(),
            nn.Conv1d(hid,hid,k,s,k//2,groups=hid,bias=False), nn.BatchNorm1d(hid), Swish(),
            nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
        )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

class EfficientNetB0_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        setting=[[1,16,1,1,3],[6,24,2,2,3],[6,40,2,2,5],[6,80,3,2,3],[6,112,3,1,5],[6,192,4,2,5],[6,320,1,1,3]]
        in_ch=int(32*wm)
        last=1280 if wm<=1 else int(1280*wm)
        feats=[Conv_3x3(1,in_ch,2)]
        ch=in_ch
        for t,c,n,s,k in setting:
            oc=int(c*wm)
            for i in range(n):
                feats.append(InvertedResidual_Eff(ch, oc, s if i==0 else 1, t, k))
                ch=oc
        feats += [Conv_1x1(ch,last), nn.AdaptiveAvgPool1d(1)]
        self.features=nn.Sequential(*feats)
        self.fc=nn.Linear(last,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        return self.fc(self.features(x).squeeze(-1))

class InvertedResidual_Mnas(nn.Module):
    def __init__(self, inp, oup, s, t, k):
        super().__init__()
        self.use=(s==1 and inp==oup)
        hid=inp*t
        self.conv=nn.Sequential(
            nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
            nn.Conv1d(hid,hid,k,s,k//2,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
            nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
        )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

def SepConv_3x3(inp,oup):
    return nn.Sequential(
        nn.Conv1d(inp,inp,3,1,1,groups=inp,bias=False), nn.BatchNorm1d(inp), nn.ReLU6(inplace=True),
        nn.Conv1d(inp,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
    )

def Conv3_3(inp,oup,s):
    return nn.Sequential(
        nn.Conv1d(inp,oup,3,s,1,bias=False), nn.BatchNorm1d(oup), nn.ReLU6(inplace=True)
    )

def Conv1_1(inp,oup):
    return nn.Sequential(
        nn.Conv1d(inp,oup,1,1,0,bias=False), nn.BatchNorm1d(oup), nn.ReLU6(inplace=True)
    )

class MnasNet_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        setting=[[3,24,3,2,3],[3,40,3,2,5],[6,80,3,2,5],[6,96,2,1,3],[6,192,4,2,5],[6,320,1,1,3]]
        in_ch=int(32*wm)
        last=1280 if wm<=1 else int(1280*wm)
        feats=[Conv3_3(1,in_ch,2), SepConv_3x3(in_ch,16)]
        ch=16
        for t,c,n,s,k in setting:
            oc=int(c*wm)
            for i in range(n):
                feats.append(InvertedResidual_Mnas(ch, oc, s if i==0 else 1, t, k))
                ch=oc
        feats += [Conv1_1(ch,last), nn.AdaptiveAvgPool1d(1)]
        self.features=nn.Sequential(*feats)
        self.fc=nn.Linear(last,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        return self.fc(self.features(x).squeeze(-1))

class DSConv_Mobile(nn.Module):
    def __init__(self, f3, f1, s=1, p=1):
        super().__init__()
        self.seq=nn.Sequential(
            nn.Conv1d(f3,f3,3,s,p,groups=f3,bias=False), nn.BatchNorm1d(f3), nn.ReLU(inplace=True),
            nn.Conv1d(f3,f1,1,1,0,bias=False), nn.BatchNorm1d(f1), nn.ReLU(inplace=True)
        )
    def forward(self,x): return self.seq(x)

class MobileNet_1D(nn.Module):
    def __init__(self, channels=[32,64,128,256,512,1024], wm=1.0, n_classes=6):
        super().__init__()
        ch=[int(c*wm) for c in channels]
        self.conv=nn.Sequential(
            nn.Conv1d(1,ch[0],3,2,1,bias=False), nn.BatchNorm1d(ch[0]), nn.ReLU(inplace=True)
        )
        self.features=nn.Sequential(
            DSConv_Mobile(ch[0],ch[1],1,1),
            DSConv_Mobile(ch[1],ch[2],2,1),
            DSConv_Mobile(ch[2],ch[2],1,1),
            DSConv_Mobile(ch[2],ch[3],2,1),
            DSConv_Mobile(ch[3],ch[3],1,1),
            DSConv_Mobile(ch[3],ch[4],2,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[5],2,1),
            DSConv_Mobile(ch[5],ch[5],1,1),
        )
        self.avg=nn.AdaptiveAvgPool1d(1)
        self.fc=nn.Linear(ch[5],n_classes)
    def forward(self,x):
        x=self.conv(x)
        x=self.features(x)
        x=self.avg(x).squeeze(-1)
        return self.fc(x)

class InvertedResidual_V2(nn.Module):
    def __init__(self, inp, oup, s, t):
        super().__init__()
        hid=int(inp*t)
        self.use=(s==1 and inp==oup)
        if t==1:
            self.conv=nn.Sequential(
                nn.Conv1d(hid,hid,3,s,1,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
            )
        else:
            self.conv=nn.Sequential(
                nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,hid,3,s,1,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
            )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

def make_divisible(x, d=8):
    return int((x + d - 1) // d * d)

class MobileNetV2_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        input_channel=32
        last=1280
        setting=[[1,16,1,1],[6,24,2,2],[6,32,3,2],[6,64,4,2],
                 [6,96,3,1],[6,160,3,2],[6,320,1,1]]
        self.last_ch=make_divisible(last*wm) if wm>1 else last
        feats=[nn.Conv1d(1,input_channel,3,2,1,bias=False), nn.BatchNorm1d(input_channel), nn.ReLU6(inplace=True)]
        ch=input_channel
        for t,c,n,s in setting:
            oc=make_divisible(c*wm) if t>1 else c
            for i in range(n):
                feats.append(InvertedResidual_V2(ch, oc, s if i==0 else 1, t))
                ch=oc
        feats += [nn.Conv1d(ch,self.last_ch,1,1,0,bias=False), nn.BatchNorm1d(self.last_ch), nn.ReLU6(inplace=True)]
        self.features=nn.Sequential(*feats)
        self.pool=nn.AdaptiveAvgPool1d(1)
        self.fc=nn.Linear(self.last_ch,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        x=self.features(x)
        x=self.pool(x).squeeze(-1)
        return self.fc(x)

# --- TSLANet (compact) ---
class TSLANet(nn.Module):
    def __init__(self, C_in, T_in, n_classes=6, emb=128, depth=3, patch=8, drop=0.10):
        super().__init__()
        stride=max(1, patch//2)
        self.proj=nn.Conv1d(C_in,emb,patch,stride)
        N=int((T_in-patch)/stride+1)
        self.pos=nn.Parameter(torch.zeros(1,N,emb))
        self.blocks=nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(emb),
                nn.Identity(),  # spectral placeholder
                nn.LayerNorm(emb),
                nn.Sequential(
                    nn.Linear(emb,int(emb*3)), nn.GELU(),
                    nn.Linear(int(emb*3),emb)
                )
            ) for _ in range(depth)
        ])
        self.drop=drop
        self.head=nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(emb,emb//2),
            nn.GELU(),
            nn.Linear(emb//2,n_classes)
        )
    def forward(self,x):
        x=self.proj(x).flatten(2).transpose(1,2)
        x=x+self.pos
        x=F.dropout(x,p=self.drop,training=self.training)
        for b in self.blocks:
            x=x + b[3](b[1](b[0](x)))
        return self.head(x.mean(1))

# -----------------------------------------------------------------------------
# 9) TRAIN / EVAL HELPERS
# -----------------------------------------------------------------------------

def process_target(y: torch.Tensor) -> torch.Tensor:
    if y.dim() > 1:
        if y.size(1) > 1:
            y = y.argmax(dim=1)
        else:
            y = y.squeeze(1)
    return y.long()

def train_one(model, loader, opt):
    model.train()
    losses = []
    for x, y in loader:
        x = x.to(DEVICE)
        y = process_target(y.to(DEVICE))

        out = model(x)               # logits
        loss = F.cross_entropy(out, y)

        opt.zero_grad()
        loss.backward()
        opt.step()

        losses.append(loss.item())
    return float(np.mean(losses)) if losses else 0.0

def eval_fold(model, loader):
    """
    Evaluate on one fold: return y_true and probabilities.
    """
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = process_target(y.to(DEVICE))

            out  = model(x)  # logits
            prob = F.softmax(out, dim=1).cpu().numpy()

            ys.append(y.cpu().numpy())
            ps.append(prob)

    ys = np.concatenate(ys)
    ps = np.concatenate(ps)
    return ys, ps

# -----------------------------------------------------------------------------
# 10) K-FOLD CV RUNNER FOR ANY MODEL
# -----------------------------------------------------------------------------

def run_cv_for_model(model_name, model_ctor, num_cls, dataset):
    N = len(dataset)
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=0)

    metrics = {"acc": [], "f1": [], "prec": [], "rec": [], "auc": []}
    loss_hist = []
    all_y = []
    all_probs = []

    print(f"\n====== Running 6-fold CV for {model_name} ======")

    for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(N)), start=1):
        print(f"\n--- {model_name} Fold {fold}/{N_SPLITS} ---")

        train_loader = DataLoader(
            Subset(dataset, train_idx),
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=4,
            pin_memory=True
        )
        val_loader = DataLoader(
            Subset(dataset, val_idx),
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=4,
            pin_memory=True
        )

        model = model_ctor().to(DEVICE)
        opt   = optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-4)
        sch   = CosineAnnealingLR(opt, T_max=EPOCHS)

        fold_losses = []
        for _ in trange(EPOCHS, desc=f"{model_name}-F{fold}", leave=False):
            loss = train_one(model, train_loader, opt)
            sch.step()
            fold_losses.append(loss)
        loss_hist.append(fold_losses)

        y_fold, p_fold = eval_fold(model, val_loader)
        preds = p_fold.argmax(axis=1)

        acc  = accuracy_score(y_fold, preds)
        f1   = f1_score(y_fold, preds, average="weighted")
        prec = precision_score(y_fold, preds, average="weighted")
        rec  = recall_score(y_fold, preds, average="weighted")

        # per-fold AUC (micro / multi-class)
        if num_cls == 2:
            auc_val = roc_auc_score(y_fold, p_fold[:,1])
        else:
            y_bin = label_binarize(y_fold, classes=list(range(num_cls)))
            auc_val = roc_auc_score(y_bin, p_fold, average="micro", multi_class="ovo")

        print(f" Fold {fold}: Acc={acc:.4f} F1={f1:.4f} Prec={prec:.4f} Rec={rec:.4f} AUC={auc_val:.4f}")
        for k,v in zip(("acc","f1","prec","rec","auc"), (acc,f1,prec,rec,auc_val)):
            metrics[k].append(v)

        all_y.append(y_fold)
        all_probs.append(p_fold)

    all_y = np.concatenate(all_y)
    all_probs = np.concatenate(all_probs, axis=0)

    # Global ROC for the model
    if num_cls == 2:
        fpr, tpr, _ = roc_curve(all_y, all_probs[:,1], pos_label=1)
        auc_global = roc_auc_score(all_y, all_probs[:,1])
    else:
        y_bin = label_binarize(all_y, classes=list(range(num_cls)))
        fpr, tpr, _ = roc_curve(y_bin.ravel(), all_probs.ravel())
        auc_global = roc_auc_score(y_bin, all_probs, average="micro", multi_class="ovo")

    return metrics, loss_hist, all_y, all_probs, (fpr, tpr, auc_global)

# -----------------------------------------------------------------------------
# 11) MAIN: WISDM + MODELS + PLOTS
# -----------------------------------------------------------------------------

def main():
    # ---- WISDM data ----
    download_wisdm()
    df = load_wisdm_dataframe(WISDM_PATH)
    X_segments, y_labels = segment_wisdm(df)

    labels_cat = pd.Series(y_labels).astype("category")
    class_names = list(labels_cat.cat.categories)
    y_int = labels_cat.cat.codes.to_numpy().astype(np.int64)

    print("[INFO] WISDM classes:", class_names)
    print("[INFO] Raw class counts:", np.bincount(y_int))

    # Balance dataset
    X_bal, y_bal = balance_by_min_class(X_segments, y_int, seed=42)
    dataset = WISDMDataset(X_bal, y_bal)
    N = len(dataset)
    print(f"[INFO] Balanced dataset size: {N}")

    num_cls = len(class_names)
    dims = (1, WINDOW_SIZE)

    # Complexity estimate for QTModel only
    tmp_qt = QTModel(num_cls=num_cls, in_ch=1).to(DEVICE)
    try:
        macs_qt, params_qt = get_model_complexity_info(
            tmp_qt, dims,
            as_strings=True,
            print_per_layer_stat=False
        )
    except Exception as e:
        print(f"[WARN] ptflops failed on QTModel: {e}")
        macs_qt, params_qt = "N/A", "N/A"
    print(f"[INFO] QTModel params: {params_qt}, FLOPs: {macs_qt}")

    # Model factories
    model_factories = {
        "QTModel":           lambda: QTModel(num_cls=num_cls, in_ch=1),
        "ResNet1D":          lambda: ResNet1D(num_inputs=1, num_classes=num_cls),
        "EfficientNetB0_1D": lambda: EfficientNetB0_1D(n_classes=num_cls),
        "MnasNet_1D":        lambda: MnasNet_1D(n_classes=num_cls),
        "MobileNet_1D":      lambda: MobileNet_1D(n_classes=num_cls),
        "MobileNetV2_1D":    lambda: MobileNetV2_1D(n_classes=num_cls),
        "TSLANet":           lambda: TSLANet(C_in=1, T_in=WINDOW_SIZE, n_classes=num_cls),
    }

    all_metrics  = {}
    all_losses   = {}
    all_yprobs   = {}
    all_rocs     = {}

    for name, ctor in model_factories.items():
        metrics, loss_hist, y_all, p_all, roc_info = run_cv_for_model(
            name, ctor, num_cls, dataset
        )
        all_metrics[name] = metrics
        all_losses[name]  = loss_hist
        all_yprobs[name]  = (y_all, p_all)
        all_rocs[name]    = roc_info

    # Summary table
    gpu_mem = None
    if DEVICE.type == "cuda":
        gpu_mem = (torch.cuda.memory_allocated()/1e6,
                   torch.cuda.memory_reserved()/1e6)

    summary_rows = []
    for name in model_factories.keys():
        m = all_metrics[name]
        row = {
            "model":        name,
            "dataset":      "WISDM",
            "acc_mean±sd":  f"{np.mean(m['acc']):.4f}±{np.std(m['acc']):.4f}",
            "f1_mean±sd":   f"{np.mean(m['f1']):.4f}±{np.std(m['f1']):.4f}",
            "prec_mean±sd": f"{np.mean(m['prec']):.4f}±{np.std(m['prec']):.4f}",
            "rec_mean±sd":  f"{np.mean(m['rec']):.4f}±{np.std(m['rec']):.4f}",
            "auc_mean±sd":  f"{np.mean(m['auc']):.4f}±{np.std(m['auc']):.4f}",
            "GPU_mem_MB":   gpu_mem,
        }
        if name == "QTModel":
            row["params"] = params_qt
            row["FLOPs"]  = macs_qt
        else:
            row["params"] = "N/A"
            row["FLOPs"]  = "N/A"
        summary_rows.append(row)

    df_sum = pd.DataFrame(summary_rows)
    print("\n[SUMMARY COMPARISON]")
    print(df_sum.to_markdown(index=False))

    # -------------------------------------------------
    # PLOTS
    # -------------------------------------------------

    # 1) Training Loss Curves (mean over folds) per model
    plt.figure()
    for name, loss_hist in all_losses.items():
        # loss_hist: list of [EPOCHS] per fold
        loss_arr = np.array(loss_hist)  # [n_folds, epochs]
        mean_loss = loss_arr.mean(axis=0)
        plt.plot(range(1, EPOCHS+1), mean_loss, label=name)
    plt.xlabel("Epoch")
    plt.ylabel("Training Loss")
    plt.title("Training Loss Curves (mean over folds)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # 2) ROC curves per model (Testing Accuracy Curve with AUC)
    plt.figure()
    for name, (fpr, tpr, auc_val) in all_rocs.items():
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})")
    plt.plot([0,1],[0,1],'k--',alpha=0.5)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curves (Validation, all folds aggregated)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # 3) Classification HeatMaps (Confusion Matrices in %)
    for name, (y_all, p_all) in all_yprobs.items():
        preds = p_all.argmax(axis=1)
        cm = confusion_matrix(y_all, preds, labels=list(range(num_cls)))
        cm_sum = cm.sum(axis=1, keepdims=True) + 1e-8
        cm_percent = cm.astype(float) / cm_sum * 100.0

        plt.figure(figsize=(6,5))
        im = plt.imshow(cm_percent, interpolation='nearest', cmap="Blues", vmin=0, vmax=100)
        plt.title(f"Confusion Matrix (%) - {name}")
        plt.colorbar(im, fraction=0.046, pad=0.04)
        tick_marks = np.arange(num_cls)
        plt.xticks(tick_marks, class_names, rotation=45, ha="right")
        plt.yticks(tick_marks, class_names)

        # annotate
        thresh = cm_percent.max() / 2.
        for i in range(num_cls):
            for j in range(num_cls):
                plt.text(j, i, f"{cm_percent[i, j]:.1f}",
                         ha="center", va="center",
                         color="white" if cm_percent[i, j] > thresh else "black",
                         fontsize=8)
        plt.ylabel("True label")
        plt.xlabel("Predicted label")
        plt.tight_layout()
        plt.show()

if __name__ == "__main__":
    main()


### On WISDM Dataset

In [ ]:
#!/usr/bin/env python3
"""
HAR with QTModel vs Baseline 1D CNN models on WISDM / UCI HAR.

QTModel:
 - Strong 1D CNN backbone (64-d features)
 - Stacked quantum blocks:
     * Linear -> angles for Q/K/V
     * TorchQuantum QFM for Q, K, V
     * Self-attention after each QFM block (element-wise Q·K gating of V)
     * Transformer-style FFN + LayerNorm with residuals
 - Final quantum gate (extra QFM) on top of last features
 - Fused [features + gate] -> MLP classifier

Baselines:
 - ResNet1D (DeepILS)
 - EfficientNetB0_1D
 - MnasNet_1D
 - MobileNet_1D
 - MobileNetV2_1D
 - TSLANet

Outputs:
 - 6-fold CV metrics for each model
 - Training Loss Curves (mean over folds)
 - ROC curves (Validation, AUC)
 - Classification HeatMaps (confusion matrices in %, per model)
"""

import os
import urllib.request
from pathlib import Path
import zipfile

import psutil
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR

from ptflops import get_model_complexity_info

from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, confusion_matrix
)
from sklearn.preprocessing import label_binarize

import matplotlib.pyplot as plt
from tqdm import trange

import torchquantum as tq

# -----------------------------------------------------------------------------
# GLOBAL CONFIG
# -----------------------------------------------------------------------------

DATASET = "WISDM"   # "WISDM" or "UCIHAR"

QSize = 4                # number of quantum wires
Q_LAYERS = 4             # depth of quantum feature map
Q_BLOCKS = 4             # number of quantum blocks
EPOCHS = 40              # training epochs per fold
BATCH_SIZE = 256         # batch size
N_SPLITS = 6             # K-fold CV

WINDOW_SIZE = 200        # default for WISDM, overwritten for UCIHAR if needed
STEP_SIZE = 100          # WISDM step size

DATA_DIR = Path(os.environ.get("DATA_DIR", "../data"))
DATA_DIR.mkdir(exist_ok=True)

# WISDM
WISDM_URL = (
    "https://raw.githubusercontent.com/soham97/WISD_HAR_files/master/"
    "WISDM_ar_v1.1_raw.txt"
)
WISDM_PATH = DATA_DIR / "WISDM_ar_v1.1_raw.txt"

# UCI HAR
UCIHAR_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "00240/UCI%20HAR%20Dataset.zip"
)
UCIHAR_ZIP_PATH = DATA_DIR / "UCI_HAR_Dataset.zip"
UCIHAR_EXTRACT_DIR = DATA_DIR / "UCI_HAR_Dataset"

# -----------------------------------------------------------------------------
# 1) DEVICE SELECTION (fixed cuda:0 if available)
# -----------------------------------------------------------------------------

if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
    print("Using device: cuda:0")
else:
    DEVICE = torch.device("cpu")
    print("CUDA not available, using CPU")

ram = psutil.virtual_memory()
print(f"RAM Usage: {(ram.total-ram.available)/1e9:.1f}/{ram.total/1e9:.1f} GiB ({ram.percent}%)\n")

# -----------------------------------------------------------------------------
# 2) WISDM & UCI HAR LOADING / PREPROCESSING
# -----------------------------------------------------------------------------

def download_wisdm():
    if WISDM_PATH.exists():
        print(f"[INFO] Found WISDM file at {WISDM_PATH}")
        return
    print(f"[INFO] Downloading WISDM from {WISDM_URL} ...")
    urllib.request.urlretrieve(WISDM_URL, WISDM_PATH)
    print("[INFO] Download complete.")

def load_wisdm_dataframe(path: Path) -> pd.DataFrame:
    """Load and clean WISDM dataset."""
    print("[INFO] Loading WISDM data into DataFrame...")
    columns = ["user", "activity", "timestamp", "x", "y", "z"]
    df = pd.read_csv(
        path,
        header=None,
        names=columns,
        engine="python",
        on_bad_lines="skip",
    )

    df = df.dropna()
    df["z"] = df["z"].astype(str).str.replace(";", "", regex=False)
    df["z"] = df["z"].astype(float)
    df = df[df["timestamp"] != 0]
    df = df.sort_values(by=["user", "timestamp"], ignore_index=True)

    print("[INFO] Cleaned WISDM shape:", df.shape)
    return df

def segment_wisdm(df: pd.DataFrame,
                  window_size: int = WINDOW_SIZE,
                  step_size: int = STEP_SIZE):
    """Segment accelerometer data into windows and compute magnitude."""
    print("[INFO] Segmenting WISDM time series into windows...")
    X_segments = []
    y_labels = []

    users = df["user"].unique()
    for user in users:
        df_user = df[df["user"] == user]
        activities = df_user["activity"].unique()

        for act in activities:
            df_ua = df_user[df_user["activity"] == act]
            signal = df_ua[["x", "y", "z"]].values  # [T, 3]

            for start in range(0, len(signal) - window_size, step_size):
                end = start + window_size
                window = signal[start:end]  # [window_size, 3]
                mag = np.sqrt((window ** 2).sum(axis=1))  # [window_size]
                X_segments.append(mag)
                y_labels.append(act)

    X_segments = np.array(X_segments, dtype=np.float32)
    y_labels = np.array(y_labels)
    print("[INFO] WISDM Segmented X shape:", X_segments.shape)
    print("[INFO] WISDM Label distribution (raw):\n", pd.Series(y_labels).value_counts())
    return X_segments, y_labels

def download_uci_har():
    if UCIHAR_EXTRACT_DIR.exists():
        print(f"[INFO] Found UCI HAR at {UCIHAR_EXTRACT_DIR}")
        return
    if not UCIHAR_ZIP_PATH.exists():
        print(f"[INFO] Downloading UCI HAR from {UCIHAR_URL} ...")
        urllib.request.urlretrieve(UCIHAR_URL, UCIHAR_ZIP_PATH)
        print("[INFO] Download complete.")
    print(f"[INFO] Extracting UCI HAR zip to {UCIHAR_EXTRACT_DIR} ...")
    with zipfile.ZipFile(UCIHAR_ZIP_PATH, "r") as zf:
        zf.extractall(UCIHAR_EXTRACT_DIR)
    print("[INFO] Extraction complete.")

def load_uci_har_magnitude():
    """
    Use body_acc_x/y/z from UCI HAR to build magnitude windows:
      - Each window length = 128
      - Combine train + test
      - Labels: 6 activities (0..5)
    """
    base = UCIHAR_EXTRACT_DIR / "UCI HAR Dataset"

    def load_split(split: str):
        split_dir = base / split
        sig_dir = split_dir / "Inertial Signals"

        # Each: [N_windows, 128]
        x = np.loadtxt(sig_dir / f"body_acc_x_{split}.txt")
        y = np.loadtxt(sig_dir / f"body_acc_y_{split}.txt")
        z = np.loadtxt(sig_dir / f"body_acc_z_{split}.txt")

        mag = np.sqrt(x**2 + y**2 + z**2).astype(np.float32)  # [N, 128]

        # Labels are 1..6 -> 0..5
        y_lab = np.loadtxt(split_dir / f"y_{split}.txt").astype(int) - 1
        return mag, y_lab

    X_train, y_train = load_split("train")
    X_test,  y_test  = load_split("test")

    X_all = np.concatenate([X_train, X_test], axis=0)
    y_all = np.concatenate([y_train, y_test], axis=0)

    print("[INFO] UCI HAR magnitude shape:", X_all.shape)
    print("[INFO] UCI HAR label distribution:",
          dict(zip(*np.unique(y_all, return_counts=True))))
    return X_all, y_all

def balance_by_min_class(X: np.ndarray, y_int: np.ndarray, seed: int = 42):
    """
    Downsample all classes to the smallest class size.
    """
    rng = np.random.default_rng(seed)
    unique_classes = np.unique(y_int)
    indices_per_class = {c: np.where(y_int == c)[0] for c in unique_classes}
    class_sizes = {c: len(idx) for c, idx in indices_per_class.items()}
    min_count = min(class_sizes.values())

    print("[INFO] Class sizes before balancing:", class_sizes)
    print("[INFO] Using min_count =", min_count, "samples per class")

    balanced_indices_list = []
    for c in unique_classes:
        idxs = indices_per_class[c]
        if len(idxs) > min_count:
            chosen = rng.choice(idxs, size=min_count, replace=False)
        else:
            chosen = idxs
        balanced_indices_list.append(chosen)

    balanced_indices = np.concatenate(balanced_indices_list)
    rng.shuffle(balanced_indices)

    balanced_counts = np.bincount(y_int[balanced_indices])
    print("[INFO] Class sizes after balancing:",
          dict(zip(unique_classes, balanced_counts)))
    return X[balanced_indices], y_int[balanced_indices]

# -----------------------------------------------------------------------------
# 3) DATASET CLASS
# -----------------------------------------------------------------------------

class WISDMDataset(Dataset):
    """
    Generic window-level dataset for time-series magnitude signals.
    Each sample: x -> [1, L], y -> integer label.
    """

    def __init__(self, X_windows: np.ndarray, y_int: np.ndarray):
        X_tensor = torch.tensor(X_windows, dtype=torch.float32)
        y_tensor = torch.tensor(y_int, dtype=torch.long)

        # Per-window normalization
        means = X_tensor.mean(dim=1, keepdim=True)
        stds = X_tensor.std(dim=1, keepdim=True) + 1e-8
        X_tensor = (X_tensor - means) / stds

        self.X = X_tensor
        self.y = y_tensor

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx].unsqueeze(0)  # [1, L]
        y = self.y[idx]
        return x, y

# -----------------------------------------------------------------------------
# 4) STRONGER CNN BACKBONE (for QTModel)
# -----------------------------------------------------------------------------

class EnhancedCNN1D(nn.Module):
    """
    Strong 1D CNN backbone for QTModel.
    Input:  [B, 1, L]
    Output: [B, 64]
    """
    def __init__(self, in_ch: int = 1):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv1d(in_ch, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2)   # L -> L/2
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2)   # L/2 -> L/4
        )
        self.block3 = nn.Sequential(
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)  # [B,64,1]
        )
        self.dropout = nn.Dropout(p=0.2)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.dropout(x)
        return x.flatten(1)  # [B, 64]

# -----------------------------------------------------------------------------
# 5) TORCHQUANTUM FEATURE MAP
# -----------------------------------------------------------------------------

class QuantumFeatureMapTQ(tq.QuantumModule):
    """
    TorchQuantum-based feature map:
     - Angle embedding via RY
     - Ring entangling CNOT chain
     - Measure all Z
    """
    def __init__(self, n_wires=4, n_layers=4):
        super().__init__()
        self.n_wires  = n_wires
        self.n_layers = n_layers
        self.measure  = tq.MeasureAll(tq.PauliZ)

    def forward(self, qdev, angles):
        # angles: (batch, n_wires)
        for w in range(self.n_wires):
            qdev.ry(wires=w, params=angles[:, w], static=self.static_mode)
        for _ in range(self.n_layers):
            for i in range(self.n_wires - 1):
                qdev.cnot(wires=[i, i+1])
            qdev.cnot(wires=[self.n_wires-1, 0])
        return self.measure(qdev)  # (batch, n_wires)

# -----------------------------------------------------------------------------
# 6) QUANTUM BLOCK WITH SELF-ATTENTION + FFN
# -----------------------------------------------------------------------------

class QBlockQT(tq.QuantumModule):
    """
    Single quantum block for QTModel:

      Input:  x ∈ R^{B×D}
      Steps:
        - Linear -> angles for Q, K, V (each in R^{B×n_wires})
        - QFM for Q, K, V (TorchQuantum)
        - Self-attention:
            q = Wq(q_out), k = Wk(k_out), v = Wv(v_out) ∈ R^{B×D}
            scores = softmax( (q * k) / sqrt(D) )
            attn_out = scores * v
        - Residual: z = x + attn_out
        - LayerNorm + Transformer-style FFN + residual + LayerNorm
    """
    def __init__(self, dim=64, n_wires=4, n_layers=4, ff_mult=2, dropout=0.1):
        super().__init__()
        self.dim     = dim
        self.n_wires = n_wires

        # QFM for Q, K, V
        self.q_fm = QuantumFeatureMapTQ(n_wires, n_layers)
        self.k_fm = QuantumFeatureMapTQ(n_wires, n_layers)
        self.v_fm = QuantumFeatureMapTQ(n_wires, n_layers)

        # Map input features -> Q/K/V angles
        self.reduce = nn.Linear(dim, 3 * n_wires)

        # Project Q/K/V outputs up to feature dim
        self.q_proj = nn.Linear(n_wires, dim)
        self.k_proj = nn.Linear(n_wires, dim)
        self.v_proj = nn.Linear(n_wires, dim)

        # Transformer-style FFN
        self.ff = nn.Sequential(
            nn.Linear(dim, ff_mult * dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_mult * dim, dim),
        )

        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        # x: [B, D]
        B, D = x.shape

        # Classical to quantum angles
        r = self.reduce(x)                      # [B, 3 * n_wires]
        q_ang, k_ang, v_ang = r.chunk(3, dim=1) # each [B, n_wires]

        # Quantum device
        qdev = tq.QuantumDevice(
            n_wires=self.n_wires,
            bsz=B,
            device=x.device
        )

        # QFM for Q, K, V
        q_out = self.q_fm(qdev, q_ang)  # [B, n_wires]
        k_out = self.k_fm(qdev, k_ang)
        v_out = self.v_fm(qdev, v_ang)

        # Project to feature dimension
        q = self.q_proj(q_out)  # [B, D]
        k = self.k_proj(k_out)  # [B, D]
        v = self.v_proj(v_out)  # [B, D]

        # Self-attention (element-wise gating)
        scale = float(D) ** 0.5
        scores = torch.softmax((q * k) / scale, dim=-1)  # [B, D]
        attn_out = scores * v                            # [B, D]

        # Residual + FFN + norms
        z = x + attn_out          # first residual
        z_norm = self.norm1(z)
        ff_out = self.ff(z_norm)
        y = self.norm2(z_norm + ff_out)  # second residual
        return y

# -----------------------------------------------------------------------------
# 7) QTModel: CNN -> stacked QBlocks -> quantum gate -> classifier
# -----------------------------------------------------------------------------

class QTModel(nn.Module):
    """
    QTModel:
      - EnhancedCNN1D -> 64-d feature
      - Multiple QBlockQT (each with QFM+SelfAttention+FFN)
      - Final quantum gate:
          angles = W_gate(e)
          q_gate = QFM(angles)
          fused = [e, q_gate]
          MLP classifier
    """
    def __init__(self, num_cls, in_ch=1,
                 n_blocks=Q_BLOCKS, n_wires=QSize, n_layers=Q_LAYERS,
                 dim=64, ff_mult=2, dropout=0.1):
        super().__init__()

        self.dim      = dim
        self.n_wires  = n_wires
        self.cnn      = EnhancedCNN1D(in_ch)

        # Stacked quantum blocks
        self.blocks = nn.ModuleList([
            QBlockQT(dim=dim, n_wires=n_wires, n_layers=n_layers,
                     ff_mult=ff_mult, dropout=dropout)
            for _ in range(n_blocks)
        ])

        # Final quantum gate
        self.gate_linear = nn.Linear(dim, n_wires)
        self.gate_qfm    = QuantumFeatureMapTQ(n_wires, n_layers)

        # Classifier on fused [features + quantum gate]
        self.cls_head = nn.Sequential(
            nn.LayerNorm(dim + n_wires),
            nn.Linear(dim + n_wires, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, num_cls),
        )

    def forward(self, x):
        # CNN backbone
        e = self.cnn(x)  # [B, 64]

        # Quantum blocks
        for block in self.blocks:
            e = block(e)  # [B, 64]

        # Final quantum gate
        angles = self.gate_linear(e)  # [B, n_wires]
        qdev = tq.QuantumDevice(
            n_wires=self.n_wires,
            bsz=angles.size(0),
            device=angles.device
        )
        q_gate = self.gate_qfm(qdev, angles)  # [B, n_wires]

        fused = torch.cat([e, q_gate], dim=1)  # [B, 64 + n_wires]
        logits = self.cls_head(fused)          # raw logits
        return logits

# -----------------------------------------------------------------------------
# 8) BASELINE MODELS
# -----------------------------------------------------------------------------

def conv_dw(in_planes, out_planes, kernel_size, stride=1, dilation=1):
    return nn.Sequential(
        nn.Conv1d(in_planes, in_planes, kernel_size=kernel_size,
                  stride=stride, padding=(kernel_size//2)*dilation, dilation=dilation,
                  groups=in_planes, bias=False),
        nn.Conv1d(in_planes, out_planes, kernel_size=1, bias=False)
    )

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool1d(1)
        self.max = nn.AdaptiveMaxPool1d(1)
        red = max(1, in_planes // ratio)
        self.fc = nn.Sequential(
            nn.Conv1d(in_planes, red, 1, bias=False), nn.ReLU(),
            nn.Conv1d(red, in_planes, 1, bias=False)
        )
        self.sig = nn.Sigmoid()
    def forward(self, x):
        return self.sig(self.fc(self.avg(x)) + self.fc(self.max(x)))

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv1 = nn.Conv1d(2, 1, k, padding=k//2, bias=False)
        self.sig = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sig(self.conv1(torch.cat([avg_out, max_out], dim=1)))

class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, out_planes, k, stride=1, dilation=1, downsample=None):
        super().__init__()
        self.conv1 = conv_dw(in_planes, out_planes, k, stride, dilation)
        self.bn1 = nn.BatchNorm1d(out_planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv_dw(out_planes, out_planes, k)
        self.bn2 = nn.BatchNorm1d(out_planes)
        self.ca = ChannelAttention(out_planes)
        self.sa = SpatialAttention()
        self.downsample = downsample
    def forward(self, x):
        res = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.ca(out)*out
        out = self.sa(out)*out
        if self.downsample is not None:
            res = self.downsample(x)
        return self.relu(out + res)

class DeepILS(nn.Module):
    def __init__(self, num_inputs=1, num_classes=6, block=BasicBlock1D, group_sizes=(2,2,2,2), base=64, k=3):
        super().__init__()
        self.inp = nn.Sequential(
            nn.Conv1d(num_inputs, base, 5, 2, 3, bias=False),
            nn.BatchNorm1d(base),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(3,2,1)
        )
        self.inplanes = base
        self.groups = nn.ModuleList()
        planes = [base*(2**i) for i in range(len(group_sizes))]
        strides = [1] + [2]*(len(group_sizes)-1)
        for i, blocks in enumerate(group_sizes):
            pl = planes[i]
            st = strides[i]
            down = None
            if st != 1 or self.inplanes != pl*block.expansion:
                down = nn.Sequential(
                    nn.Conv1d(self.inplanes, pl*block.expansion, 1, st, bias=False),
                    nn.BatchNorm1d(pl*block.expansion)
                )
            layers = [block(self.inplanes, pl, k, stride=st, downsample=down)]
            self.inplanes = pl*block.expansion
            for _ in range(1, blocks):
                layers.append(block(self.inplanes, pl, k))
            self.groups.append(nn.Sequential(*layers))
        self.groups = nn.Sequential(*self.groups)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(planes[-1]*block.expansion, num_classes)
        )
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
    def forward(self, x):
        x = self.inp(x)
        x = self.groups(x)
        return self.head(x)

class ResNet1D(DeepILS):
    pass

class Swish(nn.Module):
    def forward(self, x): return x*torch.sigmoid(x)

def Conv_3x3(inp, oup, s):
    return nn.Sequential(
        nn.Conv1d(inp,oup,3,s,1,bias=False),
        nn.BatchNorm1d(oup),
        Swish()
    )

def Conv_1x1(inp, oup):
    return nn.Sequential(
        nn.Conv1d(inp,oup,1,1,0,bias=False),
        nn.BatchNorm1d(oup),
        Swish()
    )

class InvertedResidual_Eff(nn.Module):
    def __init__(self, inp, oup, s, t, k):
        super().__init__()
        self.use=(s==1 and inp==oup)
        hid=inp*t
        self.conv=nn.Sequential(
            nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), Swish(),
            nn.Conv1d(hid,hid,k,s,k//2,groups=hid,bias=False), nn.BatchNorm1d(hid), Swish(),
            nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
        )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

class EfficientNetB0_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        setting=[[1,16,1,1,3],[6,24,2,2,3],[6,40,2,2,5],
                 [6,80,3,2,3],[6,112,3,1,5],[6,192,4,2,5],[6,320,1,1,3]]
        in_ch=int(32*wm)
        last=1280 if wm<=1 else int(1280*wm)
        feats=[Conv_3x3(1,in_ch,2)]
        ch=in_ch
        for t,c,n,s,k in setting:
            oc=int(c*wm)
            for i in range(n):
                feats.append(InvertedResidual_Eff(ch, oc, s if i==0 else 1, t, k))
                ch=oc
        feats += [Conv_1x1(ch,last), nn.AdaptiveAvgPool1d(1)]
        self.features=nn.Sequential(*feats)
        self.fc=nn.Linear(last,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        return self.fc(self.features(x).squeeze(-1))

class InvertedResidual_Mnas(nn.Module):
    def __init__(self, inp, oup, s, t, k):
        super().__init__()
        self.use=(s==1 and inp==oup)
        hid=inp*t
        self.conv=nn.Sequential(
            nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
            nn.Conv1d(hid,hid,k,s,k//2,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
            nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
        )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

def SepConv_3x3(inp,oup):
    return nn.Sequential(
        nn.Conv1d(inp,inp,3,1,1,groups=inp,bias=False), nn.BatchNorm1d(inp), nn.ReLU6(inplace=True),
        nn.Conv1d(inp,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
    )

def Conv3_3(inp,oup,s):
    return nn.Sequential(
        nn.Conv1d(inp,oup,3,s,1,bias=False), nn.BatchNorm1d(oup), nn.ReLU6(inplace=True)
    )

def Conv1_1(inp,oup):
    return nn.Sequential(
        nn.Conv1d(inp,oup,1,1,0,bias=False), nn.BatchNorm1d(oup), nn.ReLU6(inplace=True)
    )

class MnasNet_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        setting=[[3,24,3,2,3],[3,40,3,2,5],[6,80,3,2,5],
                 [6,96,2,1,3],[6,192,4,2,5],[6,320,1,1,3]]
        in_ch=int(32*wm)
        last=1280 if wm<=1 else int(1280*wm)
        feats=[Conv3_3(1,in_ch,2), SepConv_3x3(in_ch,16)]
        ch=16
        for t,c,n,s,k in setting:
            oc=int(c*wm)
            for i in range(n):
                feats.append(InvertedResidual_Mnas(ch, oc, s if i==0 else 1, t, k))
                ch=oc
        feats += [Conv1_1(ch,last), nn.AdaptiveAvgPool1d(1)]
        self.features=nn.Sequential(*feats)
        self.fc=nn.Linear(last,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        return self.fc(self.features(x).squeeze(-1))

class DSConv_Mobile(nn.Module):
    def __init__(self, f3, f1, s=1, p=1):
        super().__init__()
        self.seq=nn.Sequential(
            nn.Conv1d(f3,f3,3,s,p,groups=f3,bias=False), nn.BatchNorm1d(f3), nn.ReLU(inplace=True),
            nn.Conv1d(f3,f1,1,1,0,bias=False), nn.BatchNorm1d(f1), nn.ReLU(inplace=True)
        )
    def forward(self,x): return self.seq(x)

class MobileNet_1D(nn.Module):
    def __init__(self, channels=[32,64,128,256,512,1024], wm=1.0, n_classes=6):
        super().__init__()
        ch=[int(c*wm) for c in channels]
        self.conv=nn.Sequential(
            nn.Conv1d(1,ch[0],3,2,1,bias=False), nn.BatchNorm1d(ch[0]), nn.ReLU(inplace=True)
        )
        self.features=nn.Sequential(
            DSConv_Mobile(ch[0],ch[1],1,1),
            DSConv_Mobile(ch[1],ch[2],2,1),
            DSConv_Mobile(ch[2],ch[2],1,1),
            DSConv_Mobile(ch[2],ch[3],2,1),
            DSConv_Mobile(ch[3],ch[3],1,1),
            DSConv_Mobile(ch[3],ch[4],2,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[5],2,1),
            DSConv_Mobile(ch[5],ch[5],1,1),
        )
        self.avg=nn.AdaptiveAvgPool1d(1)
        self.fc=nn.Linear(ch[5],n_classes)
    def forward(self,x):
        x=self.conv(x)
        x=self.features(x)
        x=self.avg(x).squeeze(-1)
        return self.fc(x)

class InvertedResidual_V2(nn.Module):
    def __init__(self, inp, oup, s, t):
        super().__init__()
        hid=int(inp*t)
        self.use=(s==1 and inp==oup)
        if t==1:
            self.conv=nn.Sequential(
                nn.Conv1d(hid,hid,3,s,1,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
            )
        else:
            self.conv=nn.Sequential(
                nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,hid,3,s,1,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
            )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

def make_divisible(x, d=8):
    return int((x + d - 1) // d * d)

class MobileNetV2_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        input_channel=32
        last=1280
        setting=[[1,16,1,1],[6,24,2,2],[6,32,3,2],[6,64,4,2],
                 [6,96,3,1],[6,160,3,2],[6,320,1,1]]
        self.last_ch=make_divisible(last*wm) if wm>1 else last
        feats=[nn.Conv1d(1,input_channel,3,2,1,bias=False), nn.BatchNorm1d(input_channel), nn.ReLU6(inplace=True)]
        ch=input_channel
        for t,c,n,s in setting:
            oc=make_divisible(c*wm) if t>1 else c
            for i in range(n):
                feats.append(InvertedResidual_V2(ch, oc, s if i==0 else 1, t))
                ch=oc
        feats += [nn.Conv1d(ch,self.last_ch,1,1,0,bias=False), nn.BatchNorm1d(self.last_ch), nn.ReLU6(inplace=True)]
        self.features=nn.Sequential(*feats)
        self.pool=nn.AdaptiveAvgPool1d(1)
        self.fc=nn.Linear(self.last_ch,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        x=self.features(x)
        x=self.pool(x).squeeze(-1)
        return self.fc(x)

# --- TSLANet (compact) ---
class TSLANet(nn.Module):
    def __init__(self, C_in, T_in, n_classes=6, emb=128, depth=3, patch=8, drop=0.10):
        super().__init__()
        stride=max(1, patch//2)
        self.proj=nn.Conv1d(C_in,emb,patch,stride)
        N=int((T_in-patch)/stride+1)
        self.pos=nn.Parameter(torch.zeros(1,N,emb))
        self.blocks=nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(emb),
                nn.Identity(),  # spectral placeholder
                nn.LayerNorm(emb),
                nn.Sequential(
                    nn.Linear(emb,int(emb*3)), nn.GELU(),
                    nn.Linear(int(emb*3),emb)
                )
            ) for _ in range(depth)
        ])
        self.drop=drop
        self.head=nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(emb,emb//2),
            nn.GELU(),
            nn.Linear(emb//2,n_classes)
        )
    def forward(self,x):
        x=self.proj(x).flatten(2).transpose(1,2)
        x=x+self.pos
        x=F.dropout(x,p=self.drop,training=self.training)
        for b in self.blocks:
            x=x + b[3](b[1](b[0](x)))
        return self.head(x.mean(1))

# -----------------------------------------------------------------------------
# 9) TRAIN / EVAL HELPERS
# -----------------------------------------------------------------------------

def process_target(y: torch.Tensor) -> torch.Tensor:
    if y.dim() > 1:
        if y.size(1) > 1:
            y = y.argmax(dim=1)
        else:
            y = y.squeeze(1)
    return y.long()

def train_one(model, loader, opt):
    model.train()
    losses = []
    for x, y in loader:
        x = x.to(DEVICE)
        y = process_target(y.to(DEVICE))

        out = model(x)               # logits
        loss = F.cross_entropy(out, y)

        opt.zero_grad()
        loss.backward()
        opt.step()

        losses.append(loss.item())
    return float(np.mean(losses)) if losses else 0.0

def eval_fold(model, loader, num_cls):
    """Evaluate on one fold: return y_true and probabilities."""
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = process_target(y.to(DEVICE))
            out  = model(x)  # logits
            prob = F.softmax(out, dim=1).cpu().numpy()
            ys.append(y.cpu().numpy())
            ps.append(prob)
    ys = np.concatenate(ys)
    ps = np.concatenate(ps)
    return ys, ps

def run_cv_for_model(model_name, model_ctor, num_cls, dataset):
    N = len(dataset)
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=0)

    metrics = {"acc": [], "f1": [], "prec": [], "rec": [], "auc": []}
    loss_hist = []
    all_y = []
    all_probs = []

    print(f"\n====== Running 6-fold CV for {model_name} ======")

    for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(N)), start=1):
        print(f"\n--- {model_name} Fold {fold}/{N_SPLITS} ---")

        train_loader = DataLoader(
            Subset(dataset, train_idx),
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=4,
            pin_memory=True
        )
        val_loader = DataLoader(
            Subset(dataset, val_idx),
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=4,
            pin_memory=True
        )

        model = model_ctor().to(DEVICE)
        opt   = optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-4)
        sch   = CosineAnnealingLR(opt, T_max=EPOCHS)

        fold_losses = []
        for _ in trange(EPOCHS, desc=f"{model_name}-F{fold}", leave=False):
            loss = train_one(model, train_loader, opt)
            sch.step()
            fold_losses.append(loss)
        loss_hist.append(fold_losses)

        y_fold, p_fold = eval_fold(model, val_loader, num_cls)
        preds = p_fold.argmax(axis=1)

        acc  = accuracy_score(y_fold, preds)
        f1   = f1_score(y_fold, preds, average="weighted")
        prec = precision_score(y_fold, preds, average="weighted")
        rec  = recall_score(y_fold, preds, average="weighted")

        if num_cls == 2:
            auc_val = roc_auc_score(y_fold, p_fold[:,1])
        else:
            y_bin = label_binarize(y_fold, classes=list(range(num_cls)))
            auc_val = roc_auc_score(y_bin, p_fold, average="micro", multi_class="ovo")

        print(f" Fold {fold}: Acc={acc:.4f} F1={f1:.4f} Prec={prec:.4f} Rec={rec:.4f} AUC={auc_val:.4f}")
        for k,v in zip(("acc","f1","prec","rec","auc"), (acc,f1,prec,rec,auc_val)):
            metrics[k].append(v)

        all_y.append(y_fold)
        all_probs.append(p_fold)

        # free model this fold
        del model
        torch.cuda.empty_cache()

    all_y = np.concatenate(all_y)
    all_probs = np.concatenate(all_probs, axis=0)

    # Global ROC for the model
    if num_cls == 2:
        fpr, tpr, _ = roc_curve(all_y, all_probs[:,1], pos_label=1)
        auc_global = roc_auc_score(all_y, all_probs[:,1])
    else:
        y_bin = label_binarize(all_y, classes=list(range(num_cls)))
        fpr, tpr, _ = roc_curve(y_bin.ravel(), all_probs.ravel())
        auc_global = roc_auc_score(y_bin, all_probs, average="micro", multi_class="ovo")

    return metrics, loss_hist, all_y, all_probs, (fpr, tpr, auc_global)

# -----------------------------------------------------------------------------
# 10) MAIN: DATA CHOICE + MODELS + PLOTS
# -----------------------------------------------------------------------------

def main():
    global WINDOW_SIZE

    # -----------------------
    # Dataset choice
    # -----------------------
    if DATASET == "WISDM":
        download_wisdm()
        df = load_wisdm_dataframe(WISDM_PATH)
        X_segments, y_labels = segment_wisdm(df)

        labels_cat = pd.Series(y_labels).astype("category")
        class_names = list(labels_cat.cat.categories)
        y_int = labels_cat.cat.codes.to_numpy().astype(np.int64)

        WINDOW_SIZE = X_segments.shape[1]

    elif DATASET == "UCIHAR":
        download_uci_har()
        X_segments, y_int = load_uci_har_magnitude()

        class_names = [
            "WALKING",
            "WALK_UP",
            "WALK_DOWN",
            "SITTING",
            "STANDING",
            "LAYING",
        ]
        WINDOW_SIZE = X_segments.shape[1]

    else:
        raise ValueError(f"Unknown DATASET: {DATASET}")

    print("[INFO] Using dataset:", DATASET)
    print("[INFO] Classes:", class_names)

    # Balance dataset
    X_bal, y_bal = balance_by_min_class(X_segments, y_int, seed=42)
    dataset = WISDMDataset(X_bal, y_bal)
    N = len(dataset)
    print(f"[INFO] Balanced dataset size: {N}")

    num_cls = len(class_names)
    seq_len = X_bal.shape[1]
    dims = (1, seq_len)

    # ------------------------
    # Model factories
    # ------------------------
    model_factories = {
        "QTModel":           lambda: QTModel(num_cls=num_cls, in_ch=1),
        "ResNet1D":          lambda: ResNet1D(num_inputs=1, num_classes=num_cls),
        "EfficientNetB0_1D": lambda: EfficientNetB0_1D(n_classes=num_cls),
        "MnasNet_1D":        lambda: MnasNet_1D(n_classes=num_cls),
        "MobileNet_1D":      lambda: MobileNet_1D(n_classes=num_cls),
        "MobileNetV2_1D":    lambda: MobileNetV2_1D(n_classes=num_cls),
        "TSLANet":           lambda: TSLANet(C_in=1, T_in=seq_len, n_classes=num_cls),
    }

    # ------------------------
    # Complexity (params/FLOPs) for all models
    # ------------------------
    complexity = {}
    for name, ctor in model_factories.items():
        model = ctor().to(DEVICE)
        try:
            macs, params = get_model_complexity_info(
                model, dims,
                as_strings=True,
                print_per_layer_stat=False
            )
        except Exception as e:
            print(f"[WARN] ptflops failed on {name}: {e}")
            macs, params = "N/A", "N/A"
        complexity[name] = (params, macs)
        del model
        torch.cuda.empty_cache()
        print(f"[INFO] {name}: params={params}, FLOPs={macs}")

    # ------------------------
    # Run CV for each model
    # ------------------------
    all_metrics  = {}
    all_losses   = {}
    all_yprobs   = {}
    all_rocs     = {}

    for name, ctor in model_factories.items():
        metrics, loss_hist, y_all, p_all, roc_info = run_cv_for_model(
            name, ctor, num_cls, dataset
        )
        all_metrics[name] = metrics
        all_losses[name]  = loss_hist
        all_yprobs[name]  = (y_all, p_all)
        all_rocs[name]    = roc_info

    # Summary table
    gpu_mem = None
    if DEVICE.type == "cuda":
        gpu_mem = (torch.cuda.memory_allocated()/1e6,
                   torch.cuda.memory_reserved()/1e6)

    summary_rows = []
    for name in model_factories.keys():
        m = all_metrics[name]
        params, macs = complexity.get(name, ("N/A", "N/A"))

        row = {
            "model":        name,
            "dataset":      DATASET,
            "acc_mean±sd":  f"{np.mean(m['acc']):.4f}±{np.std(m['acc']):.4f}",
            "f1_mean±sd":   f"{np.mean(m['f1']):.4f}±{np.std(m['f1']):.4f}",
            "prec_mean±sd": f"{np.mean(m['prec']):.4f}±{np.std(m['prec']):.4f}",
            "rec_mean±sd":  f"{np.mean(m['rec']):.4f}±{np.std(m['rec']):.4f}",
            "auc_mean±sd":  f"{np.mean(m['auc']):.4f}±{np.std(m['auc']):.4f}",
            "GPU_mem_MB":   gpu_mem,
            "params":       params,
            "FLOPs":        macs,
        }
        summary_rows.append(row)

    df_sum = pd.DataFrame(summary_rows)
    print("\n[SUMMARY COMPARISON]")
    print(df_sum.to_markdown(index=False))

    # -------------------------------------------------
    # PLOTS
    # -------------------------------------------------

    # 1) Training Loss Curves (mean over folds) per model
    plt.figure()
    for name, loss_hist in all_losses.items():
        loss_arr = np.array(loss_hist)  # [n_folds, epochs]
        mean_loss = loss_arr.mean(axis=0)
        plt.plot(range(1, EPOCHS+1), mean_loss, label=name)
    plt.xlabel("Epoch")
    plt.ylabel("Training Loss")
    plt.title(f"Training Loss Curves (mean over folds) - {DATASET}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # 2) ROC curves per model (Validation)
    plt.figure()
    for name, (fpr, tpr, auc_val) in all_rocs.items():
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})")
    plt.plot([0,1],[0,1],'k--',alpha=0.5)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curves (Validation, all folds aggregated) - {DATASET}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # 3) Classification HeatMaps (Confusion Matrices in %)
    num_cls = len(class_names)
    for name, (y_all, p_all) in all_yprobs.items():
        preds = p_all.argmax(axis=1)
        cm = confusion_matrix(y_all, preds, labels=list(range(num_cls)))
        cm_sum = cm.sum(axis=1, keepdims=True) + 1e-8
        cm_percent = cm.astype(float) / cm_sum * 100.0

        plt.figure(figsize=(6,5))
        im = plt.imshow(cm_percent, interpolation='nearest', cmap="Blues", vmin=0, vmax=100)
        plt.title(f"Confusion Matrix (%) - {name} ({DATASET})")
        plt.colorbar(im, fraction=0.046, pad=0.04)
        tick_marks = np.arange(num_cls)
        plt.xticks(tick_marks, class_names, rotation=45, ha="right")
        plt.yticks(tick_marks, class_names)

        thresh = cm_percent.max() / 2.
        for i in range(num_cls):
            for j in range(num_cls):
                plt.text(j, i, f"{cm_percent[i, j]:.1f}",
                         ha="center", va="center",
                         color="white" if cm_percent[i, j] > thresh else "black",
                         fontsize=8)
        plt.ylabel("True label")
        plt.xlabel("Predicted label")
        plt.tight_layout()
        plt.show()

if __name__ == "__main__":
    main()


### On UCI HAR

In [ ]:
#!/usr/bin/env python3
"""
HAR with QTModel vs Baseline 1D CNN models on WISDM / UCI HAR.

QTModel:
 - Strong 1D CNN backbone (64-d features)
 - Stacked quantum blocks:
     * Linear -> angles for Q/K/V
     * TorchQuantum QFM for Q, K, V
     * Self-attention after each QFM block (element-wise Q·K gating of V)
     * Transformer-style FFN + LayerNorm with residuals
 - Final quantum gate (extra QFM) on top of last features
 - Fused [features + gate] -> MLP classifier

Baselines:
 - ResNet1D (DeepILS)
 - EfficientNetB0_1D
 - MnasNet_1D
 - MobileNet_1D
 - MobileNetV2_1D
 - TSLANet

Outputs:
 - 6-fold CV metrics for each model
 - Training Loss Curves (mean over folds)
 - ROC curves (Validation, AUC)
 - Classification HeatMaps (confusion matrices in %, per model)
"""

import os
import urllib.request
from pathlib import Path
import zipfile

import psutil
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR

from ptflops import get_model_complexity_info

from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, confusion_matrix
)
from sklearn.preprocessing import label_binarize

import matplotlib.pyplot as plt
from tqdm import trange

import torchquantum as tq

# -----------------------------------------------------------------------------
# GLOBAL CONFIG
# -----------------------------------------------------------------------------

DATASET = "UCIHAR"  # "WISDM" or "UCIHAR"

QSize = 4                # number of quantum wires
Q_LAYERS = 4             # depth of quantum feature map
Q_BLOCKS = 4             # number of quantum blocks
EPOCHS = 40              # training epochs per fold
BATCH_SIZE = 256         # batch size
N_SPLITS = 6             # K-fold CV

WINDOW_SIZE = 200        # default for WISDM, overwritten for UCIHAR if needed
STEP_SIZE = 100          # WISDM step size

DATA_DIR = Path(os.environ.get("DATA_DIR", "../data"))
DATA_DIR.mkdir(exist_ok=True)

# WISDM
WISDM_URL = (
    "https://raw.githubusercontent.com/soham97/WISD_HAR_files/master/"
    "WISDM_ar_v1.1_raw.txt"
)
WISDM_PATH = DATA_DIR / "WISDM_ar_v1.1_raw.txt"

# UCI HAR
UCIHAR_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "00240/UCI%20HAR%20Dataset.zip"
)
UCIHAR_ZIP_PATH = DATA_DIR / "UCI_HAR_Dataset.zip"
UCIHAR_EXTRACT_DIR = DATA_DIR / "UCI_HAR_Dataset"

# -----------------------------------------------------------------------------
# 1) DEVICE SELECTION (fixed cuda:0 if available)
# -----------------------------------------------------------------------------

if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
    print("Using device: cuda:0")
else:
    DEVICE = torch.device("cpu")
    print("CUDA not available, using CPU")

ram = psutil.virtual_memory()
print(f"RAM Usage: {(ram.total-ram.available)/1e9:.1f}/{ram.total/1e9:.1f} GiB ({ram.percent}%)\n")

# -----------------------------------------------------------------------------
# 2) WISDM & UCI HAR LOADING / PREPROCESSING
# -----------------------------------------------------------------------------

def download_wisdm():
    if WISDM_PATH.exists():
        print(f"[INFO] Found WISDM file at {WISDM_PATH}")
        return
    print(f"[INFO] Downloading WISDM from {WISDM_URL} ...")
    urllib.request.urlretrieve(WISDM_URL, WISDM_PATH)
    print("[INFO] Download complete.")

def load_wisdm_dataframe(path: Path) -> pd.DataFrame:
    """Load and clean WISDM dataset."""
    print("[INFO] Loading WISDM data into DataFrame...")
    columns = ["user", "activity", "timestamp", "x", "y", "z"]
    df = pd.read_csv(
        path,
        header=None,
        names=columns,
        engine="python",
        on_bad_lines="skip",
    )

    df = df.dropna()
    df["z"] = df["z"].astype(str).str.replace(";", "", regex=False)
    df["z"] = df["z"].astype(float)
    df = df[df["timestamp"] != 0]
    df = df.sort_values(by=["user", "timestamp"], ignore_index=True)

    print("[INFO] Cleaned WISDM shape:", df.shape)
    return df

def segment_wisdm(df: pd.DataFrame,
                  window_size: int = WINDOW_SIZE,
                  step_size: int = STEP_SIZE):
    """Segment accelerometer data into windows and compute magnitude."""
    print("[INFO] Segmenting WISDM time series into windows...")
    X_segments = []
    y_labels = []

    users = df["user"].unique()
    for user in users:
        df_user = df[df["user"] == user]
        activities = df_user["activity"].unique()

        for act in activities:
            df_ua = df_user[df_user["activity"] == act]
            signal = df_ua[["x", "y", "z"]].values  # [T, 3]

            for start in range(0, len(signal) - window_size, step_size):
                end = start + window_size
                window = signal[start:end]  # [window_size, 3]
                mag = np.sqrt((window ** 2).sum(axis=1))  # [window_size]
                X_segments.append(mag)
                y_labels.append(act)

    X_segments = np.array(X_segments, dtype=np.float32)
    y_labels = np.array(y_labels)
    print("[INFO] WISDM Segmented X shape:", X_segments.shape)
    print("[INFO] WISDM Label distribution (raw):\n", pd.Series(y_labels).value_counts())
    return X_segments, y_labels

def download_uci_har():
    if UCIHAR_EXTRACT_DIR.exists():
        print(f"[INFO] Found UCI HAR at {UCIHAR_EXTRACT_DIR}")
        return
    if not UCIHAR_ZIP_PATH.exists():
        print(f"[INFO] Downloading UCI HAR from {UCIHAR_URL} ...")
        urllib.request.urlretrieve(UCIHAR_URL, UCIHAR_ZIP_PATH)
        print("[INFO] Download complete.")
    print(f"[INFO] Extracting UCI HAR zip to {UCIHAR_EXTRACT_DIR} ...")
    with zipfile.ZipFile(UCIHAR_ZIP_PATH, "r") as zf:
        zf.extractall(UCIHAR_EXTRACT_DIR)
    print("[INFO] Extraction complete.")

def load_uci_har_magnitude():
    """
    Use body_acc_x/y/z from UCI HAR to build magnitude windows:
      - Each window length = 128
      - Combine train + test
      - Labels: 6 activities (0..5)
    """
    base = UCIHAR_EXTRACT_DIR / "UCI HAR Dataset"

    def load_split(split: str):
        split_dir = base / split
        sig_dir = split_dir / "Inertial Signals"

        # Each: [N_windows, 128]
        x = np.loadtxt(sig_dir / f"body_acc_x_{split}.txt")
        y = np.loadtxt(sig_dir / f"body_acc_y_{split}.txt")
        z = np.loadtxt(sig_dir / f"body_acc_z_{split}.txt")

        mag = np.sqrt(x**2 + y**2 + z**2).astype(np.float32)  # [N, 128]

        # Labels are 1..6 -> 0..5
        y_lab = np.loadtxt(split_dir / f"y_{split}.txt").astype(int) - 1
        return mag, y_lab

    X_train, y_train = load_split("train")
    X_test,  y_test  = load_split("test")

    X_all = np.concatenate([X_train, X_test], axis=0)
    y_all = np.concatenate([y_train, y_test], axis=0)

    print("[INFO] UCI HAR magnitude shape:", X_all.shape)
    print("[INFO] UCI HAR label distribution:",
          dict(zip(*np.unique(y_all, return_counts=True))))
    return X_all, y_all

def balance_by_min_class(X: np.ndarray, y_int: np.ndarray, seed: int = 42):
    """
    Downsample all classes to the smallest class size.
    """
    rng = np.random.default_rng(seed)
    unique_classes = np.unique(y_int)
    indices_per_class = {c: np.where(y_int == c)[0] for c in unique_classes}
    class_sizes = {c: len(idx) for c, idx in indices_per_class.items()}
    min_count = min(class_sizes.values())

    print("[INFO] Class sizes before balancing:", class_sizes)
    print("[INFO] Using min_count =", min_count, "samples per class")

    balanced_indices_list = []
    for c in unique_classes:
        idxs = indices_per_class[c]
        if len(idxs) > min_count:
            chosen = rng.choice(idxs, size=min_count, replace=False)
        else:
            chosen = idxs
        balanced_indices_list.append(chosen)

    balanced_indices = np.concatenate(balanced_indices_list)
    rng.shuffle(balanced_indices)

    balanced_counts = np.bincount(y_int[balanced_indices])
    print("[INFO] Class sizes after balancing:",
          dict(zip(unique_classes, balanced_counts)))
    return X[balanced_indices], y_int[balanced_indices]

# -----------------------------------------------------------------------------
# 3) DATASET CLASS
# -----------------------------------------------------------------------------

class WISDMDataset(Dataset):
    """
    Generic window-level dataset for time-series magnitude signals.
    Each sample: x -> [1, L], y -> integer label.
    """

    def __init__(self, X_windows: np.ndarray, y_int: np.ndarray):
        X_tensor = torch.tensor(X_windows, dtype=torch.float32)
        y_tensor = torch.tensor(y_int, dtype=torch.long)

        # Per-window normalization
        means = X_tensor.mean(dim=1, keepdim=True)
        stds = X_tensor.std(dim=1, keepdim=True) + 1e-8
        X_tensor = (X_tensor - means) / stds

        self.X = X_tensor
        self.y = y_tensor

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx].unsqueeze(0)  # [1, L]
        y = self.y[idx]
        return x, y

# -----------------------------------------------------------------------------
# 4) STRONGER CNN BACKBONE (for QTModel)
# -----------------------------------------------------------------------------

class EnhancedCNN1D(nn.Module):
    """
    Strong 1D CNN backbone for QTModel.
    Input:  [B, 1, L]
    Output: [B, 64]
    """
    def __init__(self, in_ch: int = 1):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv1d(in_ch, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2)   # L -> L/2
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2)   # L/2 -> L/4
        )
        self.block3 = nn.Sequential(
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)  # [B,64,1]
        )
        self.dropout = nn.Dropout(p=0.2)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.dropout(x)
        return x.flatten(1)  # [B, 64]

# -----------------------------------------------------------------------------
# 5) TORCHQUANTUM FEATURE MAP
# -----------------------------------------------------------------------------

class QuantumFeatureMapTQ(tq.QuantumModule):
    """
    TorchQuantum-based feature map:
     - Angle embedding via RY
     - Ring entangling CNOT chain
     - Measure all Z
    """
    def __init__(self, n_wires=4, n_layers=4):
        super().__init__()
        self.n_wires  = n_wires
        self.n_layers = n_layers
        self.measure  = tq.MeasureAll(tq.PauliZ)

    def forward(self, qdev, angles):
        # angles: (batch, n_wires)
        for w in range(self.n_wires):
            qdev.ry(wires=w, params=angles[:, w], static=self.static_mode)
        for _ in range(self.n_layers):
            for i in range(self.n_wires - 1):
                qdev.cnot(wires=[i, i+1])
            qdev.cnot(wires=[self.n_wires-1, 0])
        return self.measure(qdev)  # (batch, n_wires)

# -----------------------------------------------------------------------------
# 6) QUANTUM BLOCK WITH SELF-ATTENTION + FFN
# -----------------------------------------------------------------------------

class QBlockQT(tq.QuantumModule):
    """
    Single quantum block for QTModel:

      Input:  x ∈ R^{B×D}
      Steps:
        - Linear -> angles for Q, K, V (each in R^{B×n_wires})
        - QFM for Q, K, V (TorchQuantum)
        - Self-attention:
            q = Wq(q_out), k = Wk(k_out), v = Wv(v_out) ∈ R^{B×D}
            scores = softmax( (q * k) / sqrt(D) )
            attn_out = scores * v
        - Residual: z = x + attn_out
        - LayerNorm + Transformer-style FFN + residual + LayerNorm
    """
    def __init__(self, dim=64, n_wires=4, n_layers=4, ff_mult=2, dropout=0.1):
        super().__init__()
        self.dim     = dim
        self.n_wires = n_wires

        # QFM for Q, K, V
        self.q_fm = QuantumFeatureMapTQ(n_wires, n_layers)
        self.k_fm = QuantumFeatureMapTQ(n_wires, n_layers)
        self.v_fm = QuantumFeatureMapTQ(n_wires, n_layers)

        # Map input features -> Q/K/V angles
        self.reduce = nn.Linear(dim, 3 * n_wires)

        # Project Q/K/V outputs up to feature dim
        self.q_proj = nn.Linear(n_wires, dim)
        self.k_proj = nn.Linear(n_wires, dim)
        self.v_proj = nn.Linear(n_wires, dim)

        # Transformer-style FFN
        self.ff = nn.Sequential(
            nn.Linear(dim, ff_mult * dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_mult * dim, dim),
        )

        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        # x: [B, D]
        B, D = x.shape

        # Classical to quantum angles
        r = self.reduce(x)                      # [B, 3 * n_wires]
        q_ang, k_ang, v_ang = r.chunk(3, dim=1) # each [B, n_wires]

        # Quantum device
        qdev = tq.QuantumDevice(
            n_wires=self.n_wires,
            bsz=B,
            device=x.device
        )

        # QFM for Q, K, V
        q_out = self.q_fm(qdev, q_ang)  # [B, n_wires]
        k_out = self.k_fm(qdev, k_ang)
        v_out = self.v_fm(qdev, v_ang)

        # Project to feature dimension
        q = self.q_proj(q_out)  # [B, D]
        k = self.k_proj(k_out)  # [B, D]
        v = self.v_proj(v_out)  # [B, D]

        # Self-attention (element-wise gating)
        scale = float(D) ** 0.5
        scores = torch.softmax((q * k) / scale, dim=-1)  # [B, D]
        attn_out = scores * v                            # [B, D]

        # Residual + FFN + norms
        z = x + attn_out          # first residual
        z_norm = self.norm1(z)
        ff_out = self.ff(z_norm)
        y = self.norm2(z_norm + ff_out)  # second residual
        return y

# -----------------------------------------------------------------------------
# 7) QTModel: CNN -> stacked QBlocks -> quantum gate -> classifier
# -----------------------------------------------------------------------------

class QTModel(nn.Module):
    """
    QTModel:
      - EnhancedCNN1D -> 64-d feature
      - Multiple QBlockQT (each with QFM+SelfAttention+FFN)
      - Final quantum gate:
          angles = W_gate(e)
          q_gate = QFM(angles)
          fused = [e, q_gate]
          MLP classifier
    """
    def __init__(self, num_cls, in_ch=1,
                 n_blocks=Q_BLOCKS, n_wires=QSize, n_layers=Q_LAYERS,
                 dim=64, ff_mult=2, dropout=0.1):
        super().__init__()

        self.dim      = dim
        self.n_wires  = n_wires
        self.cnn      = EnhancedCNN1D(in_ch)

        # Stacked quantum blocks
        self.blocks = nn.ModuleList([
            QBlockQT(dim=dim, n_wires=n_wires, n_layers=n_layers,
                     ff_mult=ff_mult, dropout=dropout)
            for _ in range(n_blocks)
        ])

        # Final quantum gate
        self.gate_linear = nn.Linear(dim, n_wires)
        self.gate_qfm    = QuantumFeatureMapTQ(n_wires, n_layers)

        # Classifier on fused [features + quantum gate]
        self.cls_head = nn.Sequential(
            nn.LayerNorm(dim + n_wires),
            nn.Linear(dim + n_wires, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, num_cls),
        )

    def forward(self, x):
        # CNN backbone
        e = self.cnn(x)  # [B, 64]

        # Quantum blocks
        for block in self.blocks:
            e = block(e)  # [B, 64]

        # Final quantum gate
        angles = self.gate_linear(e)  # [B, n_wires]
        qdev = tq.QuantumDevice(
            n_wires=self.n_wires,
            bsz=angles.size(0),
            device=angles.device
        )
        q_gate = self.gate_qfm(qdev, angles)  # [B, n_wires]

        fused = torch.cat([e, q_gate], dim=1)  # [B, 64 + n_wires]
        logits = self.cls_head(fused)          # raw logits
        return logits

# -----------------------------------------------------------------------------
# 8) BASELINE MODELS
# -----------------------------------------------------------------------------

def conv_dw(in_planes, out_planes, kernel_size, stride=1, dilation=1):
    return nn.Sequential(
        nn.Conv1d(in_planes, in_planes, kernel_size=kernel_size,
                  stride=stride, padding=(kernel_size//2)*dilation, dilation=dilation,
                  groups=in_planes, bias=False),
        nn.Conv1d(in_planes, out_planes, kernel_size=1, bias=False)
    )

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool1d(1)
        self.max = nn.AdaptiveMaxPool1d(1)
        red = max(1, in_planes // ratio)
        self.fc = nn.Sequential(
            nn.Conv1d(in_planes, red, 1, bias=False), nn.ReLU(),
            nn.Conv1d(red, in_planes, 1, bias=False)
        )
        self.sig = nn.Sigmoid()
    def forward(self, x):
        return self.sig(self.fc(self.avg(x)) + self.fc(self.max(x)))

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv1 = nn.Conv1d(2, 1, k, padding=k//2, bias=False)
        self.sig = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sig(self.conv1(torch.cat([avg_out, max_out], dim=1)))

class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, out_planes, k, stride=1, dilation=1, downsample=None):
        super().__init__()
        self.conv1 = conv_dw(in_planes, out_planes, k, stride, dilation)
        self.bn1 = nn.BatchNorm1d(out_planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv_dw(out_planes, out_planes, k)
        self.bn2 = nn.BatchNorm1d(out_planes)
        self.ca = ChannelAttention(out_planes)
        self.sa = SpatialAttention()
        self.downsample = downsample
    def forward(self, x):
        res = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.ca(out)*out
        out = self.sa(out)*out
        if self.downsample is not None:
            res = self.downsample(x)
        return self.relu(out + res)

class DeepILS(nn.Module):
    def __init__(self, num_inputs=1, num_classes=6, block=BasicBlock1D, group_sizes=(2,2,2,2), base=64, k=3):
        super().__init__()
        self.inp = nn.Sequential(
            nn.Conv1d(num_inputs, base, 5, 2, 3, bias=False),
            nn.BatchNorm1d(base),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(3,2,1)
        )
        self.inplanes = base
        self.groups = nn.ModuleList()
        planes = [base*(2**i) for i in range(len(group_sizes))]
        strides = [1] + [2]*(len(group_sizes)-1)
        for i, blocks in enumerate(group_sizes):
            pl = planes[i]
            st = strides[i]
            down = None
            if st != 1 or self.inplanes != pl*block.expansion:
                down = nn.Sequential(
                    nn.Conv1d(self.inplanes, pl*block.expansion, 1, st, bias=False),
                    nn.BatchNorm1d(pl*block.expansion)
                )
            layers = [block(self.inplanes, pl, k, stride=st, downsample=down)]
            self.inplanes = pl*block.expansion
            for _ in range(1, blocks):
                layers.append(block(self.inplanes, pl, k))
            self.groups.append(nn.Sequential(*layers))
        self.groups = nn.Sequential(*self.groups)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(planes[-1]*block.expansion, num_classes)
        )
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
    def forward(self, x):
        x = self.inp(x)
        x = self.groups(x)
        return self.head(x)

class ResNet1D(DeepILS):
    pass

class Swish(nn.Module):
    def forward(self, x): return x*torch.sigmoid(x)

def Conv_3x3(inp, oup, s):
    return nn.Sequential(
        nn.Conv1d(inp,oup,3,s,1,bias=False),
        nn.BatchNorm1d(oup),
        Swish()
    )

def Conv_1x1(inp, oup):
    return nn.Sequential(
        nn.Conv1d(inp,oup,1,1,0,bias=False),
        nn.BatchNorm1d(oup),
        Swish()
    )

class InvertedResidual_Eff(nn.Module):
    def __init__(self, inp, oup, s, t, k):
        super().__init__()
        self.use=(s==1 and inp==oup)
        hid=inp*t
        self.conv=nn.Sequential(
            nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), Swish(),
            nn.Conv1d(hid,hid,k,s,k//2,groups=hid,bias=False), nn.BatchNorm1d(hid), Swish(),
            nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
        )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

class EfficientNetB0_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        setting=[[1,16,1,1,3],[6,24,2,2,3],[6,40,2,2,5],
                 [6,80,3,2,3],[6,112,3,1,5],[6,192,4,2,5],[6,320,1,1,3]]
        in_ch=int(32*wm)
        last=1280 if wm<=1 else int(1280*wm)
        feats=[Conv_3x3(1,in_ch,2)]
        ch=in_ch
        for t,c,n,s,k in setting:
            oc=int(c*wm)
            for i in range(n):
                feats.append(InvertedResidual_Eff(ch, oc, s if i==0 else 1, t, k))
                ch=oc
        feats += [Conv_1x1(ch,last), nn.AdaptiveAvgPool1d(1)]
        self.features=nn.Sequential(*feats)
        self.fc=nn.Linear(last,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        return self.fc(self.features(x).squeeze(-1))

class InvertedResidual_Mnas(nn.Module):
    def __init__(self, inp, oup, s, t, k):
        super().__init__()
        self.use=(s==1 and inp==oup)
        hid=inp*t
        self.conv=nn.Sequential(
            nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
            nn.Conv1d(hid,hid,k,s,k//2,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
            nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
        )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

def SepConv_3x3(inp,oup):
    return nn.Sequential(
        nn.Conv1d(inp,inp,3,1,1,groups=inp,bias=False), nn.BatchNorm1d(inp), nn.ReLU6(inplace=True),
        nn.Conv1d(inp,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
    )

def Conv3_3(inp,oup,s):
    return nn.Sequential(
        nn.Conv1d(inp,oup,3,s,1,bias=False), nn.BatchNorm1d(oup), nn.ReLU6(inplace=True)
    )

def Conv1_1(inp,oup):
    return nn.Sequential(
        nn.Conv1d(inp,oup,1,1,0,bias=False), nn.BatchNorm1d(oup), nn.ReLU6(inplace=True)
    )

class MnasNet_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        setting=[[3,24,3,2,3],[3,40,3,2,5],[6,80,3,2,5],
                 [6,96,2,1,3],[6,192,4,2,5],[6,320,1,1,3]]
        in_ch=int(32*wm)
        last=1280 if wm<=1 else int(1280*wm)
        feats=[Conv3_3(1,in_ch,2), SepConv_3x3(in_ch,16)]
        ch=16
        for t,c,n,s,k in setting:
            oc=int(c*wm)
            for i in range(n):
                feats.append(InvertedResidual_Mnas(ch, oc, s if i==0 else 1, t, k))
                ch=oc
        feats += [Conv1_1(ch,last), nn.AdaptiveAvgPool1d(1)]
        self.features=nn.Sequential(*feats)
        self.fc=nn.Linear(last,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        return self.fc(self.features(x).squeeze(-1))

class DSConv_Mobile(nn.Module):
    def __init__(self, f3, f1, s=1, p=1):
        super().__init__()
        self.seq=nn.Sequential(
            nn.Conv1d(f3,f3,3,s,p,groups=f3,bias=False), nn.BatchNorm1d(f3), nn.ReLU(inplace=True),
            nn.Conv1d(f3,f1,1,1,0,bias=False), nn.BatchNorm1d(f1), nn.ReLU(inplace=True)
        )
    def forward(self,x): return self.seq(x)

class MobileNet_1D(nn.Module):
    def __init__(self, channels=[32,64,128,256,512,1024], wm=1.0, n_classes=6):
        super().__init__()
        ch=[int(c*wm) for c in channels]
        self.conv=nn.Sequential(
            nn.Conv1d(1,ch[0],3,2,1,bias=False), nn.BatchNorm1d(ch[0]), nn.ReLU(inplace=True)
        )
        self.features=nn.Sequential(
            DSConv_Mobile(ch[0],ch[1],1,1),
            DSConv_Mobile(ch[1],ch[2],2,1),
            DSConv_Mobile(ch[2],ch[2],1,1),
            DSConv_Mobile(ch[2],ch[3],2,1),
            DSConv_Mobile(ch[3],ch[3],1,1),
            DSConv_Mobile(ch[3],ch[4],2,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[5],2,1),
            DSConv_Mobile(ch[5],ch[5],1,1),
        )
        self.avg=nn.AdaptiveAvgPool1d(1)
        self.fc=nn.Linear(ch[5],n_classes)
    def forward(self,x):
        x=self.conv(x)
        x=self.features(x)
        x=self.avg(x).squeeze(-1)
        return self.fc(x)

class InvertedResidual_V2(nn.Module):
    def __init__(self, inp, oup, s, t):
        super().__init__()
        hid=int(inp*t)
        self.use=(s==1 and inp==oup)
        if t==1:
            self.conv=nn.Sequential(
                nn.Conv1d(hid,hid,3,s,1,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
            )
        else:
            self.conv=nn.Sequential(
                nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,hid,3,s,1,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
            )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

def make_divisible(x, d=8):
    return int((x + d - 1) // d * d)

class MobileNetV2_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        input_channel=32
        last=1280
        setting=[[1,16,1,1],[6,24,2,2],[6,32,3,2],[6,64,4,2],
                 [6,96,3,1],[6,160,3,2],[6,320,1,1]]
        self.last_ch=make_divisible(last*wm) if wm>1 else last
        feats=[nn.Conv1d(1,input_channel,3,2,1,bias=False), nn.BatchNorm1d(input_channel), nn.ReLU6(inplace=True)]
        ch=input_channel
        for t,c,n,s in setting:
            oc=make_divisible(c*wm) if t>1 else c
            for i in range(n):
                feats.append(InvertedResidual_V2(ch, oc, s if i==0 else 1, t))
                ch=oc
        feats += [nn.Conv1d(ch,self.last_ch,1,1,0,bias=False), nn.BatchNorm1d(self.last_ch), nn.ReLU6(inplace=True)]
        self.features=nn.Sequential(*feats)
        self.pool=nn.AdaptiveAvgPool1d(1)
        self.fc=nn.Linear(self.last_ch,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        x=self.features(x)
        x=self.pool(x).squeeze(-1)
        return self.fc(x)

# --- TSLANet (compact) ---
class TSLANet(nn.Module):
    def __init__(self, C_in, T_in, n_classes=6, emb=128, depth=3, patch=8, drop=0.10):
        super().__init__()
        stride=max(1, patch//2)
        self.proj=nn.Conv1d(C_in,emb,patch,stride)
        N=int((T_in-patch)/stride+1)
        self.pos=nn.Parameter(torch.zeros(1,N,emb))
        self.blocks=nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(emb),
                nn.Identity(),  # spectral placeholder
                nn.LayerNorm(emb),
                nn.Sequential(
                    nn.Linear(emb,int(emb*3)), nn.GELU(),
                    nn.Linear(int(emb*3),emb)
                )
            ) for _ in range(depth)
        ])
        self.drop=drop
        self.head=nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(emb,emb//2),
            nn.GELU(),
            nn.Linear(emb//2,n_classes)
        )
    def forward(self,x):
        x=self.proj(x).flatten(2).transpose(1,2)
        x=x+self.pos
        x=F.dropout(x,p=self.drop,training=self.training)
        for b in self.blocks:
            x=x + b[3](b[1](b[0](x)))
        return self.head(x.mean(1))

# -----------------------------------------------------------------------------
# 9) TRAIN / EVAL HELPERS
# -----------------------------------------------------------------------------

def process_target(y: torch.Tensor) -> torch.Tensor:
    if y.dim() > 1:
        if y.size(1) > 1:
            y = y.argmax(dim=1)
        else:
            y = y.squeeze(1)
    return y.long()

def train_one(model, loader, opt):
    model.train()
    losses = []
    for x, y in loader:
        x = x.to(DEVICE)
        y = process_target(y.to(DEVICE))

        out = model(x)               # logits
        loss = F.cross_entropy(out, y)

        opt.zero_grad()
        loss.backward()
        opt.step()

        losses.append(loss.item())
    return float(np.mean(losses)) if losses else 0.0

def eval_fold(model, loader, num_cls):
    """Evaluate on one fold: return y_true and probabilities."""
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = process_target(y.to(DEVICE))
            out  = model(x)  # logits
            prob = F.softmax(out, dim=1).cpu().numpy()
            ys.append(y.cpu().numpy())
            ps.append(prob)
    ys = np.concatenate(ys)
    ps = np.concatenate(ps)
    return ys, ps

def run_cv_for_model(model_name, model_ctor, num_cls, dataset):
    N = len(dataset)
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=0)

    metrics = {"acc": [], "f1": [], "prec": [], "rec": [], "auc": []}
    loss_hist = []
    all_y = []
    all_probs = []

    print(f"\n====== Running 6-fold CV for {model_name} ======")

    for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(N)), start=1):
        print(f"\n--- {model_name} Fold {fold}/{N_SPLITS} ---")

        train_loader = DataLoader(
            Subset(dataset, train_idx),
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=4,
            pin_memory=True
        )
        val_loader = DataLoader(
            Subset(dataset, val_idx),
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=4,
            pin_memory=True
        )

        model = model_ctor().to(DEVICE)
        opt   = optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-4)
        sch   = CosineAnnealingLR(opt, T_max=EPOCHS)

        fold_losses = []
        for _ in trange(EPOCHS, desc=f"{model_name}-F{fold}", leave=False):
            loss = train_one(model, train_loader, opt)
            sch.step()
            fold_losses.append(loss)
        loss_hist.append(fold_losses)

        y_fold, p_fold = eval_fold(model, val_loader, num_cls)
        preds = p_fold.argmax(axis=1)

        acc  = accuracy_score(y_fold, preds)
        f1   = f1_score(y_fold, preds, average="weighted")
        prec = precision_score(y_fold, preds, average="weighted")
        rec  = recall_score(y_fold, preds, average="weighted")

        if num_cls == 2:
            auc_val = roc_auc_score(y_fold, p_fold[:,1])
        else:
            y_bin = label_binarize(y_fold, classes=list(range(num_cls)))
            auc_val = roc_auc_score(y_bin, p_fold, average="micro", multi_class="ovo")

        print(f" Fold {fold}: Acc={acc:.4f} F1={f1:.4f} Prec={prec:.4f} Rec={rec:.4f} AUC={auc_val:.4f}")
        for k,v in zip(("acc","f1","prec","rec","auc"), (acc,f1,prec,rec,auc_val)):
            metrics[k].append(v)

        all_y.append(y_fold)
        all_probs.append(p_fold)

        # free model this fold
        del model
        torch.cuda.empty_cache()

    all_y = np.concatenate(all_y)
    all_probs = np.concatenate(all_probs, axis=0)

    # Global ROC for the model
    if num_cls == 2:
        fpr, tpr, _ = roc_curve(all_y, all_probs[:,1], pos_label=1)
        auc_global = roc_auc_score(all_y, all_probs[:,1])
    else:
        y_bin = label_binarize(all_y, classes=list(range(num_cls)))
        fpr, tpr, _ = roc_curve(y_bin.ravel(), all_probs.ravel())
        auc_global = roc_auc_score(y_bin, all_probs, average="micro", multi_class="ovo")

    return metrics, loss_hist, all_y, all_probs, (fpr, tpr, auc_global)

# -----------------------------------------------------------------------------
# 10) MAIN: DATA CHOICE + MODELS + PLOTS
# -----------------------------------------------------------------------------

def main():
    global WINDOW_SIZE

    # -----------------------
    # Dataset choice
    # -----------------------
    if DATASET == "WISDM":
        download_wisdm()
        df = load_wisdm_dataframe(WISDM_PATH)
        X_segments, y_labels = segment_wisdm(df)

        labels_cat = pd.Series(y_labels).astype("category")
        class_names = list(labels_cat.cat.categories)
        y_int = labels_cat.cat.codes.to_numpy().astype(np.int64)

        WINDOW_SIZE = X_segments.shape[1]

    elif DATASET == "UCIHAR":
        download_uci_har()
        X_segments, y_int = load_uci_har_magnitude()

        class_names = [
            "WALKING",
            "WALK_UP",
            "WALK_DOWN",
            "SITTING",
            "STANDING",
            "LAYING",
        ]
        WINDOW_SIZE = X_segments.shape[1]

    else:
        raise ValueError(f"Unknown DATASET: {DATASET}")

    print("[INFO] Using dataset:", DATASET)
    print("[INFO] Classes:", class_names)

    # Balance dataset
    X_bal, y_bal = balance_by_min_class(X_segments, y_int, seed=42)
    dataset = WISDMDataset(X_bal, y_bal)
    N = len(dataset)
    print(f"[INFO] Balanced dataset size: {N}")

    num_cls = len(class_names)
    seq_len = X_bal.shape[1]
    dims = (1, seq_len)

    # ------------------------
    # Model factories
    # ------------------------
    model_factories = {
        "QTModel":           lambda: QTModel(num_cls=num_cls, in_ch=1),
        "ResNet1D":          lambda: ResNet1D(num_inputs=1, num_classes=num_cls),
        "EfficientNetB0_1D": lambda: EfficientNetB0_1D(n_classes=num_cls),
        "MnasNet_1D":        lambda: MnasNet_1D(n_classes=num_cls),
        "MobileNet_1D":      lambda: MobileNet_1D(n_classes=num_cls),
        "MobileNetV2_1D":    lambda: MobileNetV2_1D(n_classes=num_cls),
        "TSLANet":           lambda: TSLANet(C_in=1, T_in=seq_len, n_classes=num_cls),
    }

    # ------------------------
    # Complexity (params/FLOPs) for all models
    # ------------------------
    complexity = {}
    for name, ctor in model_factories.items():
        model = ctor().to(DEVICE)
        try:
            macs, params = get_model_complexity_info(
                model, dims,
                as_strings=True,
                print_per_layer_stat=False
            )
        except Exception as e:
            print(f"[WARN] ptflops failed on {name}: {e}")
            macs, params = "N/A", "N/A"
        complexity[name] = (params, macs)
        del model
        torch.cuda.empty_cache()
        print(f"[INFO] {name}: params={params}, FLOPs={macs}")

    # ------------------------
    # Run CV for each model
    # ------------------------
    all_metrics  = {}
    all_losses   = {}
    all_yprobs   = {}
    all_rocs     = {}

    for name, ctor in model_factories.items():
        metrics, loss_hist, y_all, p_all, roc_info = run_cv_for_model(
            name, ctor, num_cls, dataset
        )
        all_metrics[name] = metrics
        all_losses[name]  = loss_hist
        all_yprobs[name]  = (y_all, p_all)
        all_rocs[name]    = roc_info

    # Summary table
    gpu_mem = None
    if DEVICE.type == "cuda":
        gpu_mem = (torch.cuda.memory_allocated()/1e6,
                   torch.cuda.memory_reserved()/1e6)

    summary_rows = []
    for name in model_factories.keys():
        m = all_metrics[name]
        params, macs = complexity.get(name, ("N/A", "N/A"))

        row = {
            "model":        name,
            "dataset":      DATASET,
            "acc_mean±sd":  f"{np.mean(m['acc']):.4f}±{np.std(m['acc']):.4f}",
            "f1_mean±sd":   f"{np.mean(m['f1']):.4f}±{np.std(m['f1']):.4f}",
            "prec_mean±sd": f"{np.mean(m['prec']):.4f}±{np.std(m['prec']):.4f}",
            "rec_mean±sd":  f"{np.mean(m['rec']):.4f}±{np.std(m['rec']):.4f}",
            "auc_mean±sd":  f"{np.mean(m['auc']):.4f}±{np.std(m['auc']):.4f}",
            "GPU_mem_MB":   gpu_mem,
            "params":       params,
            "FLOPs":        macs,
        }
        summary_rows.append(row)

    df_sum = pd.DataFrame(summary_rows)
    print("\n[SUMMARY COMPARISON]")
    print(df_sum.to_markdown(index=False))

    # -------------------------------------------------
    # PLOTS
    # -------------------------------------------------

    # 1) Training Loss Curves (mean over folds) per model
    plt.figure()
    for name, loss_hist in all_losses.items():
        loss_arr = np.array(loss_hist)  # [n_folds, epochs]
        mean_loss = loss_arr.mean(axis=0)
        plt.plot(range(1, EPOCHS+1), mean_loss, label=name)
    plt.xlabel("Epoch")
    plt.ylabel("Training Loss")
    plt.title(f"Training Loss Curves (mean over folds) - {DATASET}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # 2) ROC curves per model (Validation)
    plt.figure()
    for name, (fpr, tpr, auc_val) in all_rocs.items():
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})")
    plt.plot([0,1],[0,1],'k--',alpha=0.5)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curves (Validation, all folds aggregated) - {DATASET}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # 3) Classification HeatMaps (Confusion Matrices in %)
    num_cls = len(class_names)
    for name, (y_all, p_all) in all_yprobs.items():
        preds = p_all.argmax(axis=1)
        cm = confusion_matrix(y_all, preds, labels=list(range(num_cls)))
        cm_sum = cm.sum(axis=1, keepdims=True) + 1e-8
        cm_percent = cm.astype(float) / cm_sum * 100.0

        plt.figure(figsize=(6,5))
        im = plt.imshow(cm_percent, interpolation='nearest', cmap="Blues", vmin=0, vmax=100)
        plt.title(f"Confusion Matrix (%) - {name} ({DATASET})")
        plt.colorbar(im, fraction=0.046, pad=0.04)
        tick_marks = np.arange(num_cls)
        plt.xticks(tick_marks, class_names, rotation=45, ha="right")
        plt.yticks(tick_marks, class_names)

        thresh = cm_percent.max() / 2.
        for i in range(num_cls):
            for j in range(num_cls):
                plt.text(j, i, f"{cm_percent[i, j]:.1f}",
                         ha="center", va="center",
                         color="white" if cm_percent[i, j] > thresh else "black",
                         fontsize=8)
        plt.ylabel("True label")
        plt.xlabel("Predicted label")
        plt.tight_layout()
        plt.show()

if __name__ == "__main__":
    main()


### On MotionSense

In [ ]:
#!/usr/bin/env python3
"""
HAR with QTModel vs Baseline 1D CNN models on WISDM / UCI HAR / MotionSense.

QTModel:
 - Strong 1D CNN backbone (64-d features)
 - Stacked quantum blocks:
     * Linear -> angles for Q/K/V
     * TorchQuantum QFM for Q, K, V
     * Self-attention after each QFM block (element-wise Q·K gating of V)
     * Transformer-style FFN + LayerNorm with residuals
 - Final quantum gate (extra QFM) on top of last features
 - Fused [features + gate] -> MLP classifier

Baselines:
 - ResNet1D (DeepILS)
 - EfficientNetB0_1D
 - MnasNet_1D
 - MobileNet_1D
 - MobileNetV2_1D
 - TSLANet

Outputs:
 - 6-fold CV metrics for each model
 - Training Loss Curves (mean over folds)
 - ROC curves (Validation, AUC)
 - Classification HeatMaps (confusion matrices in %, per model)
"""

import os
import glob
import urllib.request
from pathlib import Path
import zipfile

import psutil
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR

from ptflops import get_model_complexity_info

from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, confusion_matrix
)
from sklearn.preprocessing import label_binarize

import matplotlib.pyplot as plt
from tqdm import trange

import torchquantum as tq

# -----------------------------------------------------------------------------
# GLOBAL CONFIG
# -----------------------------------------------------------------------------

DATASET = "MOTIONSENSE"  # "WISDM", "UCIHAR", or "MOTIONSENSE"

QSize = 4                # number of quantum wires
Q_LAYERS = 4             # depth of quantum feature map
Q_BLOCKS = 4             # number of quantum blocks
EPOCHS = 40              # training epochs per fold
BATCH_SIZE = 256         # batch size
N_SPLITS = 6             # K-fold CV

WINDOW_SIZE = 200        # default (samples per window)
STEP_SIZE = 100          # hop size

DATA_DIR = Path(os.environ.get("DATA_DIR", "../data"))
DATA_DIR.mkdir(exist_ok=True)

# WISDM
WISDM_URL = (
    "https://raw.githubusercontent.com/soham97/WISD_HAR_files/master/"
    "WISDM_ar_v1.1_raw.txt"
)
WISDM_PATH = DATA_DIR / "WISDM_ar_v1.1_raw.txt"

# UCI HAR
UCIHAR_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "00240/UCI%20HAR%20Dataset.zip"
)
UCIHAR_ZIP_PATH = DATA_DIR / "UCI_HAR_Dataset.zip"
UCIHAR_EXTRACT_DIR = DATA_DIR / "UCI_HAR_Dataset"

# MotionSense (set to your local extracted Kaggle path if different)
MOTIONSENSE_DIR = DATA_DIR / "motionsense-dataset"  # put the unzipped Kaggle folder here

# -----------------------------------------------------------------------------
# 1) DEVICE SELECTION (fixed cuda:0 if available)
# -----------------------------------------------------------------------------

if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
    print("Using device: cuda:0")
else:
    DEVICE = torch.device("cpu")
    print("CUDA not available, using CPU")

ram = psutil.virtual_memory()
print(f"RAM Usage: {(ram.total-ram.available)/1e9:.1f}/{ram.total/1e9:.1f} GiB ({ram.percent}%)\n")

# -----------------------------------------------------------------------------
# 2) WISDM & UCI HAR LOADING / PREPROCESSING
# -----------------------------------------------------------------------------

def download_wisdm():
    if WISDM_PATH.exists():
        print(f"[INFO] Found WISDM file at {WISDM_PATH}")
        return
    print(f"[INFO] Downloading WISDM from {WISDM_URL} ...")
    urllib.request.urlretrieve(WISDM_URL, WISDM_PATH)
    print("[INFO] Download complete.")

def load_wisdm_dataframe(path: Path) -> pd.DataFrame:
    """Load and clean WISDM dataset."""
    print("[INFO] Loading WISDM data into DataFrame...")
    columns = ["user", "activity", "timestamp", "x", "y", "z"]
    df = pd.read_csv(
        path,
        header=None,
        names=columns,
        engine="python",
        on_bad_lines="skip",
    )
    df = df.dropna()
    df["z"] = df["z"].astype(str).str.replace(";", "", regex=False)
    df["z"] = df["z"].astype(float)
    df = df[df["timestamp"] != 0]
    df = df.sort_values(by=["user", "timestamp"], ignore_index=True)
    print("[INFO] Cleaned WISDM shape:", df.shape)
    return df

def segment_wisdm(df: pd.DataFrame,
                  window_size: int = WINDOW_SIZE,
                  step_size: int = STEP_SIZE):
    """Segment accelerometer data into windows and compute magnitude."""
    print("[INFO] Segmenting WISDM time series into windows...")
    X_segments = []
    y_labels = []

    users = df["user"].unique()
    for user in users:
        df_user = df[df["user"] == user]
        activities = df_user["activity"].unique()
        for act in activities:
            df_ua = df_user[df_user["activity"] == act]
            signal = df_ua[["x", "y", "z"]].values  # [T, 3]
            for start in range(0, len(signal) - window_size, step_size):
                end = start + window_size
                window = signal[start:end]  # [window_size, 3]
                mag = np.sqrt((window ** 2).sum(axis=1))  # [window_size]
                X_segments.append(mag)
                y_labels.append(act)

    X_segments = np.array(X_segments, dtype=np.float32)
    y_labels = np.array(y_labels)
    print("[INFO] WISDM Segmented X shape:", X_segments.shape)
    print("[INFO] WISDM Label distribution (raw):\n", pd.Series(y_labels).value_counts())
    return X_segments, y_labels

def download_uci_har():
    if UCIHAR_EXTRACT_DIR.exists():
        print(f"[INFO] Found UCI HAR at {UCIHAR_EXTRACT_DIR}")
        return
    if not UCIHAR_ZIP_PATH.exists():
        print(f"[INFO] Downloading UCI HAR from {UCIHAR_URL} ...")
        urllib.request.urlretrieve(UCIHAR_URL, UCIHAR_ZIP_PATH)
        print("[INFO] Download complete.")
    print(f"[INFO] Extracting UCI HAR zip to {UCIHAR_EXTRACT_DIR} ...")
    with zipfile.ZipFile(UCIHAR_ZIP_PATH, "r") as zf:
        zf.extractall(UCIHAR_EXTRACT_DIR)
    print("[INFO] Extraction complete.")

def load_uci_har_magnitude():
    """
    Use body_acc_x/y/z from UCI HAR to build magnitude windows:
      - Each window length = 128
      - Combine train + test
      - Labels: 6 activities (0..5)
    """
    base = UCIHAR_EXTRACT_DIR / "UCI HAR Dataset"

    def load_split(split: str):
        split_dir = base / split
        sig_dir = split_dir / "Inertial Signals"
        x = np.loadtxt(sig_dir / f"body_acc_x_{split}.txt")
        y = np.loadtxt(sig_dir / f"body_acc_y_{split}.txt")
        z = np.loadtxt(sig_dir / f"body_acc_z_{split}.txt")
        mag = np.sqrt(x**2 + y**2 + z**2).astype(np.float32)  # [N, 128]
        y_lab = np.loadtxt(split_dir / f"y_{split}.txt").astype(int) - 1
        return mag, y_lab

    X_train, y_train = load_split("train")
    X_test,  y_test  = load_split("test")

    X_all = np.concatenate([X_train, X_test], axis=0)
    y_all = np.concatenate([y_train, y_test], axis=0)

    print("[INFO] UCI HAR magnitude shape:", X_all.shape)
    print("[INFO] UCI HAR label distribution:",
          dict(zip(*np.unique(y_all, return_counts=True))))
    return X_all, y_all

# -----------------------------------------------------------------------------
# 2.5) MOTIONSENSE LOADING / PREPROCESSING (supports A_DeviceMotion or B_Accelerometer)
# -----------------------------------------------------------------------------

def ensure_motionsense() -> Path:
    """
    Return a local path to MotionSense. Set MOTIONSENSE_DIR above to your unzipped Kaggle folder.
    Accepted structures include:
      - MOTIONSENSE_DIR/A_DeviceMotion_data
      - MOTIONSENSE_DIR/B_Accelerometer_data
      - (Nested) MOTIONSENSE_DIR/.../A_DeviceMotion_data, etc.
    """
    if MOTIONSENSE_DIR.exists():
        print(f"[INFO] Using MotionSense at {MOTIONSENSE_DIR}")
        return MOTIONSENSE_DIR
    raise FileNotFoundError(
        f"MotionSense dataset not found at {MOTIONSENSE_DIR}. "
        "Please unzip the Kaggle dataset there or set MOTIONSENSE_DIR to the correct path."
    )

def load_motionsense_magnitude(window_size: int = WINDOW_SIZE, step_size: int = STEP_SIZE):
    """
    Reads MotionSense and builds 1D magnitude windows.
    Supports BOTH:
      - B_Accelerometer_data (raw accel)
      - A_DeviceMotion_data  (CoreMotion; uses userAcceleration.{x,y,z} by default)

    Activity folders: dws, jog, sit, std, ups, wlk
    """
    base = ensure_motionsense()

    # Find the activity root (handle nested paths)
    target_dirnames = {"B_Accelerometer_data", "A_DeviceMotion_data"}
    activity_root = None
    for dirpath, dirnames, _ in os.walk(str(base)):
        for dn in dirnames:
            if dn in target_dirnames:
                candidate = Path(dirpath) / dn
                if candidate.is_dir():
                    activity_root = candidate
                    break
        if activity_root is not None:
            break

    if activity_root is None:
        raise FileNotFoundError(
            "Could not find 'B_Accelerometer_data' or 'A_DeviceMotion_data' "
            "anywhere under the MotionSense directory."
        )

    uses_device_motion = activity_root.name == "A_DeviceMotion_data"
    print(f"[INFO] MotionSense activity root: {activity_root.name} at {activity_root}")

    # Activity map (folder key -> pretty class name)
    activity_map = {
        "dws": "Downstairs",
        "jog": "Jogging",
        "sit": "Sitting",
        "std": "Standing",
        "ups": "Upstairs",
        "wlk": "Walking",
    }
    inv_classes = list(activity_map.keys())
    class_names = [activity_map[k] for k in inv_classes]
    label_from_key = {k: i for i, k in enumerate(inv_classes)}

    # Pick (x,y,z) columns for a given CSV
    def pick_xyz_columns(df: pd.DataFrame, prefer_dev_motion: bool):
        if prefer_dev_motion:
            candidates = [
                ["userAcceleration.x", "userAcceleration.y", "userAcceleration.z"],
                ["acceleration.x", "acceleration.y", "acceleration.z"],
                # fallback (not ideal for activity classification, but last resort)
                ["gravity.x", "gravity.y", "gravity.z"],
                ["x", "y", "z"],
            ]
        else:
            candidates = [
                ["x", "y", "z"],
                ["accelerometer.x", "accelerometer.y", "accelerometer.z"],
                ["userAcceleration.x", "userAcceleration.y", "userAcceleration.z"],
                ["acceleration.x", "acceleration.y", "acceleration.z"],
            ]
        for cols in candidates:
            if all(c in df.columns for c in cols):
                return cols
        # heuristic auto-detect
        xyz_like = [c for c in df.columns
                    if ("x" in c.lower() or "y" in c.lower() or "z" in c.lower())
                    and ("mag" not in c.lower())]
        if len(xyz_like) >= 3:
            return xyz_like[:3]
        return None

    X_segments, y_labels = [], []
    total_files = 0
    used_cols_counter = {}

    subdirs = [d for d in activity_root.iterdir() if d.is_dir()]
    # Only activity subdirs we care about
    subdirs = [d for d in subdirs if any(d.name == k or d.name.startswith(k) for k in inv_classes)]
    if not subdirs:
        raise RuntimeError(
            f"No activity subfolders (dws/jog/sit/std/ups/wlk) found under {activity_root}."
        )

    for act_key in inv_classes:
        act_dirs = [d for d in subdirs if d.name == act_key or d.name.startswith(act_key)]
        if not act_dirs:
            print(f"[WARN] Missing folder for activity '{act_key}' under {activity_root} (skipping).")
            continue

        for act_dir in act_dirs:
            csvs = sorted(glob.glob(str(act_dir / "*.csv")))
            if not csvs:
                csvs = sorted(glob.glob(str(act_dir / "**" / "*.csv"), recursive=True))
            if not csvs:
                print(f"[WARN] No CSV files in {act_dir}")
                continue

            for fpath in csvs:
                total_files += 1
                df = pd.read_csv(fpath)
                cols = pick_xyz_columns(df, prefer_dev_motion=uses_device_motion)
                if cols is None:
                    print(f"[WARN] Could not find (x,y,z)-like columns in {fpath}; first cols: {list(df.columns)[:6]}")
                    continue

                used_cols_counter[tuple(cols)] = used_cols_counter.get(tuple(cols), 0) + 1

                sig = df[cols].to_numpy(dtype=np.float32)  # [T,3]
                mag = np.sqrt((sig ** 2).sum(axis=1))      # [T]

                # Sliding windows
                T = len(mag)
                for start in range(0, T - window_size, step_size):
                    end = start + window_size
                    X_segments.append(mag[start:end])
                    y_labels.append(label_from_key[act_key])

    if len(X_segments) == 0:
        raise RuntimeError(
            "No segments extracted from MotionSense (check paths and CSV schema/columns)."
        )

    X = np.stack(X_segments).astype(np.float32)
    y = np.array(y_labels, dtype=np.int64)

    print(f"[INFO] MotionSense: parsed {total_files} CSV files")
    print(f"[INFO] Column triplets used (count): { {k: v for k, v in used_cols_counter.items()} }")
    print("[INFO] MotionSense segmented X shape:", X.shape)
    print("[INFO] MotionSense label distribution:", dict(zip(*np.unique(y, return_counts=True))))
    return X, y, class_names

def balance_by_min_class(X: np.ndarray, y_int: np.ndarray, seed: int = 42):
    """
    Downsample all classes to the smallest class size.
    """
    rng = np.random.default_rng(seed)
    unique_classes = np.unique(y_int)
    indices_per_class = {c: np.where(y_int == c)[0] for c in unique_classes}
    class_sizes = {c: len(idx) for c, idx in indices_per_class.items()}
    min_count = min(class_sizes.values())

    print("[INFO] Class sizes before balancing:", class_sizes)
    print("[INFO] Using min_count =", min_count, "samples per class")

    balanced_indices_list = []
    for c in unique_classes:
        idxs = indices_per_class[c]
        if len(idxs) > min_count:
            chosen = rng.choice(idxs, size=min_count, replace=False)
        else:
            chosen = idxs
        balanced_indices_list.append(chosen)

    balanced_indices = np.concatenate(balanced_indices_list)
    rng.shuffle(balanced_indices)

    balanced_counts = np.bincount(y_int[balanced_indices])
    print("[INFO] Class sizes after balancing:",
          dict(zip(unique_classes, balanced_counts)))
    return X[balanced_indices], y_int[balanced_indices]

# -----------------------------------------------------------------------------
# 3) DATASET CLASS
# -----------------------------------------------------------------------------

class WISDMDataset(Dataset):
    """
    Generic window-level dataset for time-series magnitude signals.
    Each sample: x -> [1, L], y -> integer label.
    """

    def __init__(self, X_windows: np.ndarray, y_int: np.ndarray):
        X_tensor = torch.tensor(X_windows, dtype=torch.float32)
        y_tensor = torch.tensor(y_int, dtype=torch.long)

        # Per-window normalization
        means = X_tensor.mean(dim=1, keepdim=True)
        stds = X_tensor.std(dim=1, keepdim=True) + 1e-8
        X_tensor = (X_tensor - means) / stds

        self.X = X_tensor
        self.y = y_tensor

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx].unsqueeze(0)  # [1, L]
        y = self.y[idx]
        return x, y

# -----------------------------------------------------------------------------
# 4) STRONGER CNN BACKBONE (for QTModel)
# -----------------------------------------------------------------------------

class EnhancedCNN1D(nn.Module):
    """
    Strong 1D CNN backbone for QTModel.
    Input:  [B, 1, L]
    Output: [B, 64]
    """
    def __init__(self, in_ch: int = 1):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv1d(in_ch, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2)   # L -> L/2
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2)   # L/2 -> L/4
        )
        self.block3 = nn.Sequential(
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)  # [B,64,1]
        )
        self.dropout = nn.Dropout(p=0.2)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.dropout(x)
        return x.flatten(1)  # [B, 64]

# -----------------------------------------------------------------------------
# 5) TORCHQUANTUM FEATURE MAP
# -----------------------------------------------------------------------------

class QuantumFeatureMapTQ(tq.QuantumModule):
    """
    TorchQuantum-based feature map:
     - Angle embedding via RY
     - Ring entangling CNOT chain
     - Measure all Z
    """
    def __init__(self, n_wires=4, n_layers=4):
        super().__init__()
        self.n_wires  = n_wires
        self.n_layers = n_layers
        self.measure  = tq.MeasureAll(tq.PauliZ)

    def forward(self, qdev, angles):
        # angles: (batch, n_wires)
        for w in range(self.n_wires):
            qdev.ry(wires=w, params=angles[:, w], static=self.static_mode)
        for _ in range(self.n_layers):
            for i in range(self.n_wires - 1):
                qdev.cnot(wires=[i, i+1])
            qdev.cnot(wires=[self.n_wires-1, 0])
        return self.measure(qdev)  # (batch, n_wires)

# -----------------------------------------------------------------------------
# 6) QUANTUM BLOCK WITH SELF-ATTENTION + FFN
# -----------------------------------------------------------------------------

class QBlockQT(tq.QuantumModule):
    """
    Single quantum block for QTModel.
    """
    def __init__(self, dim=64, n_wires=4, n_layers=4, ff_mult=2, dropout=0.1):
        super().__init__()
        self.dim     = dim
        self.n_wires = n_wires

        self.q_fm = QuantumFeatureMapTQ(n_wires, n_layers)
        self.k_fm = QuantumFeatureMapTQ(n_wires, n_layers)
        self.v_fm = QuantumFeatureMapTQ(n_wires, n_layers)

        self.reduce = nn.Linear(dim, 3 * n_wires)

        self.q_proj = nn.Linear(n_wires, dim)
        self.k_proj = nn.Linear(n_wires, dim)
        self.v_proj = nn.Linear(n_wires, dim)

        self.ff = nn.Sequential(
            nn.Linear(dim, ff_mult * dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_mult * dim, dim),
        )

        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        # x: [B, D]
        B, D = x.shape
        r = self.reduce(x)                      # [B, 3*n_wires]
        q_ang, k_ang, v_ang = r.chunk(3, dim=1)

        qdev = tq.QuantumDevice(n_wires=self.n_wires, bsz=B, device=x.device)
        q_out = self.q_fm(qdev, q_ang)  # [B, n_wires]
        k_out = self.k_fm(qdev, k_ang)
        v_out = self.v_fm(qdev, v_ang)

        q = self.q_proj(q_out)  # [B, D]
        k = self.k_proj(k_out)
        v = self.v_proj(v_out)

        scale = float(D) ** 0.5
        scores = torch.softmax((q * k) / scale, dim=-1)  # [B, D]
        attn_out = scores * v                            # [B, D]

        z = x + attn_out
        z_norm = self.norm1(z)
        ff_out = self.ff(z_norm)
        y = self.norm2(z_norm + ff_out)
        return y

# -----------------------------------------------------------------------------
# 7) QTModel: CNN -> stacked QBlocks -> quantum gate -> classifier
# -----------------------------------------------------------------------------

class QTModel(nn.Module):
    def __init__(self, num_cls, in_ch=1,
                 n_blocks=Q_BLOCKS, n_wires=QSize, n_layers=Q_LAYERS,
                 dim=64, ff_mult=2, dropout=0.1):
        super().__init__()

        self.dim      = dim
        self.n_wires  = n_wires
        self.cnn      = EnhancedCNN1D(in_ch)

        self.blocks = nn.ModuleList([
            QBlockQT(dim=dim, n_wires=n_wires, n_layers=n_layers,
                     ff_mult=ff_mult, dropout=dropout)
            for _ in range(n_blocks)
        ])

        self.gate_linear = nn.Linear(dim, n_wires)
        self.gate_qfm    = QuantumFeatureMapTQ(n_wires, n_layers)

        self.cls_head = nn.Sequential(
            nn.LayerNorm(dim + n_wires),
            nn.Linear(dim + n_wires, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, num_cls),
        )

    def forward(self, x):
        e = self.cnn(x)  # [B, 64]
        for block in self.blocks:
            e = block(e)  # [B, 64]
        angles = self.gate_linear(e)
        qdev = tq.QuantumDevice(n_wires=self.n_wires, bsz=angles.size(0), device=angles.device)
        q_gate = self.gate_qfm(qdev, angles)  # [B, n_wires]
        fused = torch.cat([e, q_gate], dim=1)  # [B, 64 + n_wires]
        logits = self.cls_head(fused)
        return logits

# -----------------------------------------------------------------------------
# 8) BASELINE MODELS
# -----------------------------------------------------------------------------

def conv_dw(in_planes, out_planes, kernel_size, stride=1, dilation=1):
    return nn.Sequential(
        nn.Conv1d(in_planes, in_planes, kernel_size=kernel_size,
                  stride=stride, padding=(kernel_size//2)*dilation, dilation=dilation,
                  groups=in_planes, bias=False),
        nn.Conv1d(in_planes, out_planes, kernel_size=1, bias=False)
    )

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool1d(1)
        self.max = nn.AdaptiveMaxPool1d(1)
        red = max(1, in_planes // ratio)
        self.fc = nn.Sequential(
            nn.Conv1d(in_planes, red, 1, bias=False), nn.ReLU(),
            nn.Conv1d(red, in_planes, 1, bias=False)
        )
        self.sig = nn.Sigmoid()
    def forward(self, x):
        return self.sig(self.fc(self.avg(x)) + self.fc(self.max(x)))

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv1 = nn.Conv1d(2, 1, k, padding=k//2, bias=False)
        self.sig = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sig(self.conv1(torch.cat([avg_out, max_out], dim=1)))

class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, out_planes, k, stride=1, dilation=1, downsample=None):
        super().__init__()
        self.conv1 = conv_dw(in_planes, out_planes, k, stride, dilation)
        self.bn1 = nn.BatchNorm1d(out_planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv_dw(out_planes, out_planes, k)
        self.bn2 = nn.BatchNorm1d(out_planes)
        self.ca = ChannelAttention(out_planes)
        self.sa = SpatialAttention()
        self.downsample = downsample
    def forward(self, x):
        res = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.ca(out)*out
        out = self.sa(out)*out
        if self.downsample is not None:
            res = self.downsample(x)
        return self.relu(out + res)

class DeepILS(nn.Module):
    def __init__(self, num_inputs=1, num_classes=6, block=BasicBlock1D, group_sizes=(2,2,2,2), base=64, k=3):
        super().__init__()
        self.inp = nn.Sequential(
            nn.Conv1d(num_inputs, base, 5, 2, 3, bias=False),
            nn.BatchNorm1d(base),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(3,2,1)
        )
        self.inplanes = base
        self.groups = nn.ModuleList()
        planes = [base*(2**i) for i in range(len(group_sizes))]
        strides = [1] + [2]*(len(group_sizes)-1)
        for i, blocks in enumerate(group_sizes):
            pl = planes[i]
            st = strides[i]
            down = None
            if st != 1 or self.inplanes != pl*block.expansion:
                down = nn.Sequential(
                    nn.Conv1d(self.inplanes, pl*block.expansion, 1, st, bias=False),
                    nn.BatchNorm1d(pl*block.expansion)
                )
            layers = [block(self.inplanes, pl, k, stride=st, downsample=down)]
            self.inplanes = pl*block.expansion
            for _ in range(1, blocks):
                layers.append(block(self.inplanes, pl, k))
            self.groups.append(nn.Sequential(*layers))
        self.groups = nn.Sequential(*self.groups)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(planes[-1]*block.expansion, num_classes)
        )
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
    def forward(self, x):
        x = self.inp(x)
        x = self.groups(x)
        return self.head(x)

class ResNet1D(DeepILS):
    pass

class Swish(nn.Module):
    def forward(self, x): return x*torch.sigmoid(x)

def Conv_3x3(inp, oup, s):
    return nn.Sequential(
        nn.Conv1d(inp,oup,3,s,1,bias=False),
        nn.BatchNorm1d(oup),
        Swish()
    )

def Conv_1x1(inp, oup):
    return nn.Sequential(
        nn.Conv1d(inp,oup,1,1,0,bias=False),
        nn.BatchNorm1d(oup),
        Swish()
    )

class InvertedResidual_Eff(nn.Module):
    def __init__(self, inp, oup, s, t, k):
        super().__init__()
        self.use=(s==1 and inp==oup)
        hid=inp*t
        self.conv=nn.Sequential(
            nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), Swish(),
            nn.Conv1d(hid,hid,k,s,k//2,groups=hid,bias=False), nn.BatchNorm1d(hid), Swish(),
            nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
        )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

class EfficientNetB0_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        setting=[[1,16,1,1,3],[6,24,2,2,3],[6,40,2,2,5],
                 [6,80,3,2,3],[6,112,3,1,5],[6,192,4,2,5],[6,320,1,1,3]]
        in_ch=int(32*wm)
        last=1280 if wm<=1 else int(1280*wm)
        feats=[Conv_3x3(1,in_ch,2)]
        ch=in_ch
        for t,c,n,s,k in setting:
            oc=int(c*wm)
            for i in range(n):
                feats.append(InvertedResidual_Eff(ch, oc, s if i==0 else 1, t, k))
                ch=oc
        feats += [Conv_1x1(ch,last), nn.AdaptiveAvgPool1d(1)]
        self.features=nn.Sequential(*feats)
        self.fc=nn.Linear(last,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        return self.fc(self.features(x).squeeze(-1))

class InvertedResidual_Mnas(nn.Module):
    def __init__(self, inp, oup, s, t, k):
        super().__init__()
        self.use=(s==1 and inp==oup)
        hid=inp*t
        self.conv=nn.Sequential(
            nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
            nn.Conv1d(hid,hid,k,s,k//2,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
            nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
        )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

def SepConv_3x3(inp,oup):
    return nn.Sequential(
        nn.Conv1d(inp,inp,3,1,1,groups=inp,bias=False), nn.BatchNorm1d(inp), nn.ReLU6(inplace=True),
        nn.Conv1d(inp,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
    )

def Conv3_3(inp,oup,s):
    return nn.Sequential(
        nn.Conv1d(inp,oup,3,s,1,bias=False), nn.BatchNorm1d(oup), nn.ReLU6(inplace=True)
    )

def Conv1_1(inp,oup):
    return nn.Sequential(
        nn.Conv1d(inp,oup,1,1,0,bias=False), nn.BatchNorm1d(oup), nn.ReLU6(inplace=True)
    )

class MnasNet_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        setting=[[3,24,3,2,3],[3,40,3,2,5],[6,80,3,2,5],
                 [6,96,2,1,3],[6,192,4,2,5],[6,320,1,1,3]]
        in_ch=int(32*wm)
        last=1280 if wm<=1 else int(1280*wm)
        feats=[Conv3_3(1,in_ch,2), SepConv_3x3(in_ch,16)]
        ch=16
        for t,c,n,s,k in setting:
            oc=int(c*wm)
            for i in range(n):
                feats.append(InvertedResidual_Mnas(ch, oc, s if i==0 else 1, t, k))
                ch=oc
        feats += [Conv1_1(ch,last), nn.AdaptiveAvgPool1d(1)]
        self.features=nn.Sequential(*feats)
        self.fc=nn.Linear(last,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        return self.fc(self.features(x).squeeze(-1))

class DSConv_Mobile(nn.Module):
    def __init__(self, f3, f1, s=1, p=1):
        super().__init__()
        self.seq=nn.Sequential(
            nn.Conv1d(f3,f3,3,s,p,groups=f3,bias=False), nn.BatchNorm1d(f3), nn.ReLU(inplace=True),
            nn.Conv1d(f3,f1,1,1,0,bias=False), nn.BatchNorm1d(f1), nn.ReLU(inplace=True)
        )
    def forward(self,x): return self.seq(x)

class MobileNet_1D(nn.Module):
    def __init__(self, channels=[32,64,128,256,512,1024], wm=1.0, n_classes=6):
        super().__init__()
        ch=[int(c*wm) for c in channels]
        self.conv=nn.Sequential(
            nn.Conv1d(1,ch[0],3,2,1,bias=False), nn.BatchNorm1d(ch[0]), nn.ReLU(inplace=True)
        )
        self.features=nn.Sequential(
            DSConv_Mobile(ch[0],ch[1],1,1),
            DSConv_Mobile(ch[1],ch[2],2,1),
            DSConv_Mobile(ch[2],ch[2],1,1),
            DSConv_Mobile(ch[2],ch[3],2,1),
            DSConv_Mobile(ch[3],ch[3],1,1),
            DSConv_Mobile(ch[3],ch[4],2,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[5],2,1),
            DSConv_Mobile(ch[5],ch[5],1,1),
        )
        self.avg=nn.AdaptiveAvgPool1d(1)
        self.fc=nn.Linear(ch[5],n_classes)
    def forward(self,x):
        x=self.conv(x)
        x=self.features(x)
        x=self.avg(x).squeeze(-1)
        return self.fc(x)

class InvertedResidual_V2(nn.Module):
    def __init__(self, inp, oup, s, t):
        super().__init__()
        hid=int(inp*t)
        self.use=(s==1 and inp==oup)
        if t==1:
            self.conv=nn.Sequential(
                nn.Conv1d(hid,hid,3,s,1,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
            )
        else:
            self.conv=nn.Sequential(
                nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,hid,3,s,1,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
            )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

def make_divisible(x, d=8):
    return int((x + d - 1) // d * d)

class MobileNetV2_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        input_channel=32
        last=1280
        setting=[[1,16,1,1],[6,24,2,2],[6,32,3,2],[6,64,4,2],
                 [6,96,3,1],[6,160,3,2],[6,320,1,1]]
        self.last_ch=make_divisible(last*wm) if wm>1 else last
        feats=[nn.Conv1d(1,input_channel,3,2,1,bias=False), nn.BatchNorm1d(input_channel), nn.ReLU6(inplace=True)]
        ch=input_channel
        for t,c,n,s in setting:
            oc=make_divisible(c*wm) if t>1 else c
            for i in range(n):
                feats.append(InvertedResidual_V2(ch, oc, s if i==0 else 1, t))
                ch=oc
        feats += [nn.Conv1d(ch,self.last_ch,1,1,0,bias=False), nn.BatchNorm1d(self.last_ch), nn.ReLU6(inplace=True)]
        self.features=nn.Sequential(*feats)
        self.pool=nn.AdaptiveAvgPool1d(1)
        self.fc=nn.Linear(self.last_ch,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        x=self.features(x)
        x=self.pool(x).squeeze(-1)
        return self.fc(x)

# --- TSLANet (compact) ---
class TSLANet(nn.Module):
    def __init__(self, C_in, T_in, n_classes=6, emb=128, depth=3, patch=8, drop=0.10):
        super().__init__()
        stride=max(1, patch//2)
        self.proj=nn.Conv1d(C_in,emb,patch,stride)
        N=int((T_in-patch)/stride+1)
        self.pos=nn.Parameter(torch.zeros(1,N,emb))
        self.blocks=nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(emb),
                nn.Identity(),  # spectral placeholder
                nn.LayerNorm(emb),
                nn.Sequential(
                    nn.Linear(emb,int(emb*3)), nn.GELU(),
                    nn.Linear(int(emb*3),emb)
                )
            ) for _ in range(depth)
        ])
        self.drop=drop
        self.head=nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(emb,emb//2),
            nn.GELU(),
            nn.Linear(emb//2,n_classes)
        )
    def forward(self,x):
        x=self.proj(x).flatten(2).transpose(1,2)
        x=x+self.pos
        x=F.dropout(x,p=self.drop,training=self.training)
        for b in self.blocks:
            x=x + b[3](b[1](b[0](x)))
        return self.head(x.mean(1))

# -----------------------------------------------------------------------------
# 9) TRAIN / EVAL HELPERS
# -----------------------------------------------------------------------------

def process_target(y: torch.Tensor) -> torch.Tensor:
    if y.dim() > 1:
        if y.size(1) > 1:
            y = y.argmax(dim=1)
        else:
            y = y.squeeze(1)
    return y.long()

def train_one(model, loader, opt):
    model.train()
    losses = []
    for x, y in loader:
        x = x.to(DEVICE)
        y = process_target(y.to(DEVICE))

        out = model(x)               # logits
        loss = F.cross_entropy(out, y)

        opt.zero_grad()
        loss.backward()
        opt.step()

        losses.append(loss.item())
    return float(np.mean(losses)) if losses else 0.0

def eval_fold(model, loader, num_cls):
    """Evaluate on one fold: return y_true and probabilities."""
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = process_target(y.to(DEVICE))
            out  = model(x)  # logits
            prob = F.softmax(out, dim=1).cpu().numpy()
            ys.append(y.cpu().numpy())
            ps.append(prob)
    ys = np.concatenate(ys)
    ps = np.concatenate(ps)
    return ys, ps

def run_cv_for_model(model_name, model_ctor, num_cls, dataset):
    N = len(dataset)
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=0)

    metrics = {"acc": [], "f1": [], "prec": [], "rec": [], "auc": []}
    loss_hist = []
    all_y = []
    all_probs = []

    print(f"\n====== Running {N_SPLITS}-fold CV for {model_name} ======")

    for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(N)), start=1):
        print(f"\n--- {model_name} Fold {fold}/{N_SPLITS} ---")

        train_loader = DataLoader(
            Subset(dataset, train_idx),
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=4,
            pin_memory=True
        )
        val_loader = DataLoader(
            Subset(dataset, val_idx),
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=4,
            pin_memory=True
        )

        model = model_ctor().to(DEVICE)
        opt   = optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-4)
        sch   = CosineAnnealingLR(opt, T_max=EPOCHS)

        fold_losses = []
        for _ in trange(EPOCHS, desc=f"{model_name}-F{fold}", leave=False):
            loss = train_one(model, train_loader, opt)
            sch.step()
            fold_losses.append(loss)
        loss_hist.append(fold_losses)

        y_fold, p_fold = eval_fold(model, val_loader, num_cls)
        preds = p_fold.argmax(axis=1)

        acc  = accuracy_score(y_fold, preds)
        f1   = f1_score(y_fold, preds, average="weighted")
        prec = precision_score(y_fold, preds, average="weighted")
        rec  = recall_score(y_fold, preds, average="weighted")

        if num_cls == 2:
            auc_val = roc_auc_score(y_fold, p_fold[:,1])
        else:
            y_bin = label_binarize(y_fold, classes=list(range(num_cls)))
            auc_val = roc_auc_score(y_bin, p_fold, average="micro", multi_class="ovo")

        print(f" Fold {fold}: Acc={acc:.4f} F1={f1:.4f} Prec={prec:.4f} Rec={rec:.4f} AUC={auc_val:.4f}")
        for k,v in zip(("acc","f1","prec","rec","auc"), (acc,f1,prec,rec,auc_val)):
            metrics[k].append(v)

        all_y.append(y_fold)
        all_probs.append(p_fold)

        # free model this fold
        del model
        torch.cuda.empty_cache()

    all_y = np.concatenate(all_y)
    all_probs = np.concatenate(all_probs, axis=0)

    # Global ROC for the model
    if num_cls == 2:
        fpr, tpr, _ = roc_curve(all_y, all_probs[:,1], pos_label=1)
        auc_global = roc_auc_score(all_y, all_probs[:,1])
    else:
        y_bin = label_binarize(all_y, classes=list(range(num_cls)))
        fpr, tpr, _ = roc_curve(y_bin.ravel(), all_probs.ravel())
        auc_global = roc_auc_score(y_bin, all_probs, average="micro", multi_class="ovo")

    return metrics, loss_hist, all_y, all_probs, (fpr, tpr, auc_global)

# -----------------------------------------------------------------------------
# 10) MAIN: DATA CHOICE + MODELS + PLOTS
# -----------------------------------------------------------------------------

def main():
    global WINDOW_SIZE

    # -----------------------
    # Dataset choice
    # -----------------------
    if DATASET == "WISDM":
        download_wisdm()
        df = load_wisdm_dataframe(WISDM_PATH)
        X_segments, y_labels = segment_wisdm(df)

        labels_cat = pd.Series(y_labels).astype("category")
        class_names = list(labels_cat.cat.categories)
        y_int = labels_cat.cat.codes.to_numpy().astype(np.int64)

        WINDOW_SIZE = X_segments.shape[1]

    elif DATASET == "UCIHAR":
        download_uci_har()
        X_segments, y_int = load_uci_har_magnitude()

        class_names = [
            "WALKING",
            "WALK_UP",
            "WALK_DOWN",
            "SITTING",
            "STANDING",
            "LAYING",
        ]
        WINDOW_SIZE = X_segments.shape[1]

    elif DATASET == "MOTIONSENSE":
        # MotionSense: supports both A_DeviceMotion_data and B_Accelerometer_data
        X_segments, y_int, class_names = load_motionsense_magnitude(
            window_size=WINDOW_SIZE, step_size=STEP_SIZE
        )
        WINDOW_SIZE = X_segments.shape[1]

    else:
        raise ValueError(f"Unknown DATASET: {DATASET}")

    print("[INFO] Using dataset:", DATASET)
    print("[INFO] Classes:", class_names)

    # Balance dataset
    X_bal, y_bal = balance_by_min_class(X_segments, y_int, seed=42)
    dataset = WISDMDataset(X_bal, y_bal)
    N = len(dataset)
    print(f"[INFO] Balanced dataset size: {N}")

    num_cls = len(class_names)
    seq_len = X_bal.shape[1]
    dims = (1, seq_len)

    # ------------------------
    # Model factories
    # ------------------------
    model_factories = {
        "QTModel":           lambda: QTModel(num_cls=num_cls, in_ch=1),
        "ResNet1D":          lambda: ResNet1D(num_inputs=1, num_classes=num_cls),
        "EfficientNetB0_1D": lambda: EfficientNetB0_1D(n_classes=num_cls),
        "MnasNet_1D":        lambda: MnasNet_1D(n_classes=num_cls),
        "MobileNet_1D":      lambda: MobileNet_1D(n_classes=num_cls),
        "MobileNetV2_1D":    lambda: MobileNetV2_1D(n_classes=num_cls),
        "TSLANet":           lambda: TSLANet(C_in=1, T_in=seq_len, n_classes=num_cls),
    }

    # ------------------------
    # Complexity (params/FLOPs) for all models
    # ------------------------
    complexity = {}
    for name, ctor in model_factories.items():
        model = ctor().to(DEVICE)
        try:
            macs, params = get_model_complexity_info(
                model, dims,
                as_strings=True,
                print_per_layer_stat=False
            )
        except Exception as e:
            print(f"[WARN] ptflops failed on {name}: {e}")
            macs, params = "N/A", "N/A"
        complexity[name] = (params, macs)
        del model
        torch.cuda.empty_cache()
        print(f"[INFO] {name}: params={params}, FLOPs={macs}")

    # ------------------------
    # Run CV for each model
    # ------------------------
    all_metrics  = {}
    all_losses   = {}
    all_yprobs   = {}
    all_rocs     = {}

    for name, ctor in model_factories.items():
        metrics, loss_hist, y_all, p_all, roc_info = run_cv_for_model(
            name, ctor, num_cls, dataset
        )
        all_metrics[name] = metrics
        all_losses[name]  = loss_hist
        all_yprobs[name]  = (y_all, p_all)
        all_rocs[name]    = roc_info

    # Summary table
    gpu_mem = None
    if DEVICE.type == "cuda":
        gpu_mem = (torch.cuda.memory_allocated()/1e6,
                   torch.cuda.memory_reserved()/1e6)

    summary_rows = []
    for name in model_factories.keys():
        m = all_metrics[name]
        params, macs = complexity.get(name, ("N/A", "N/A"))

        row = {
            "model":        name,
            "dataset":      DATASET,
            "acc_mean±sd":  f"{np.mean(m['acc']):.4f}±{np.std(m['acc']):.4f}",
            "f1_mean±sd":   f"{np.mean(m['f1']):.4f}±{np.std(m['f1']):.4f}",
            "prec_mean±sd": f"{np.mean(m['prec']):.4f}±{np.std(m['prec']):.4f}",
            "rec_mean±sd":  f"{np.mean(m['rec']):.4f}±{np.std(m['rec']):.4f}",
            "auc_mean±sd":  f"{np.mean(m['auc']):.4f}±{np.std(m['auc']):.4f}",
            "GPU_mem_MB":   gpu_mem,
            "params":       params,
            "FLOPs":        macs,
        }
        summary_rows.append(row)

    df_sum = pd.DataFrame(summary_rows)
    print("\n[SUMMARY COMPARISON]")
    print(df_sum.to_markdown(index=False))

    # -------------------------------------------------
    # PLOTS
    # -------------------------------------------------

    # 1) Training Loss Curves (mean over folds) per model
    plt.figure()
    for name, loss_hist in all_losses.items():
        loss_arr = np.array(loss_hist)  # [n_folds, epochs]
        mean_loss = loss_arr.mean(axis=0)
        plt.plot(range(1, EPOCHS+1), mean_loss, label=name)
    plt.xlabel("Epoch")
    plt.ylabel("Training Loss")
    plt.title(f"Training Loss Curves (mean over folds) - {DATASET}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # 2) ROC curves per model (Validation)
    plt.figure()
    for name, (fpr, tpr, auc_val) in all_rocs.items():
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})")
    plt.plot([0,1],[0,1],'k--',alpha=0.5)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curves (Validation, all folds aggregated) - {DATASET}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # 3) Classification HeatMaps (Confusion Matrices in %)
    num_cls = len(class_names)
    for name, (y_all, p_all) in all_yprobs.items():
        preds = p_all.argmax(axis=1)
        cm = confusion_matrix(y_all, preds, labels=list(range(num_cls)))
        cm_sum = cm.sum(axis=1, keepdims=True) + 1e-8
        cm_percent = cm.astype(float) / cm_sum * 100.0

        plt.figure(figsize=(6,5))
        im = plt.imshow(cm_percent, interpolation='nearest', cmap="Blues", vmin=0, vmax=100)
        plt.title(f"Confusion Matrix (%) - {name} ({DATASET})")
        plt.colorbar(im, fraction=0.046, pad=0.04)
        tick_marks = np.arange(num_cls)
        plt.xticks(tick_marks, class_names, rotation=45, ha="right")
        plt.yticks(tick_marks, class_names)

        thresh = cm_percent.max() / 2.
        for i in range(num_cls):
            for j in range(num_cls):
                plt.text(j, i, f"{cm_percent[i, j]:.1f}",
                         ha="center", va="center",
                         color="white" if cm_percent[i, j] > thresh else "black",
                         fontsize=8)
        plt.ylabel("True label")
        plt.xlabel("Predicted label")
        plt.tight_layout()
        plt.show()

if __name__ == "__main__":
    main()


### On MHEALTH

In [ ]:
#!/usr/bin/env python3
"""
HAR with QTModel vs Baseline 1D CNN models on WISDM / UCI HAR / MHEALTH.

QTModel:
 - Strong 1D CNN backbone (64-d features)
 - Stacked quantum blocks:
     * Linear -> angles for Q/K/V
     * TorchQuantum QFM for Q, K, V
     * Self-attention after each QFM block (element-wise Q·K gating of V)
     * Transformer-style FFN + LayerNorm with residuals
 - Final quantum gate (extra QFM) on top of last features
 - Fused [features + gate] -> MLP classifier

Baselines:
 - ResNet1D (DeepILS)
 - EfficientNetB0_1D
 - MnasNet_1D
 - MobileNet_1D
 - MobileNetV2_1D
 - TSLANet

Outputs:
 - 6-fold CV metrics for each model
 - Training Loss Curves (mean over folds)
 - ROC curves (Validation, AUC)
 - Classification HeatMaps (confusion matrices in %, per model)
"""

import os
import urllib.request
from pathlib import Path
import zipfile
import io
import shutil
import glob

import psutil
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR

from ptflops import get_model_complexity_info

from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, confusion_matrix
)
from sklearn.preprocessing import label_binarize

import matplotlib.pyplot as plt
from tqdm import trange

import torchquantum as tq

# -----------------------------------------------------------------------------
# GLOBAL CONFIG
# -----------------------------------------------------------------------------

DATASET = "MHEALTH"  # "MHEALTH", "WISDM", or "UCIHAR"

QSize = 4                # number of quantum wires
Q_LAYERS = 4             # depth of quantum feature map
Q_BLOCKS = 4             # number of quantum blocks
EPOCHS = 40              # training epochs per fold
BATCH_SIZE = 256         # batch size
N_SPLITS = 6             # K-fold CV

WINDOW_SIZE = 200        # default (overwritten per dataset if needed)
STEP_SIZE = 100          # step between windows

DATA_DIR = Path(os.environ.get("DATA_DIR", "../data"))
DATA_DIR.mkdir(exist_ok=True)

# WISDM
WISDM_URL = (
    "https://raw.githubusercontent.com/soham97/WISD_HAR_files/master/"
    "WISDM_ar_v1.1_raw.txt"
)
WISDM_PATH = DATA_DIR / "WISDM_ar_v1.1_raw.txt"

# UCI HAR
UCIHAR_URLS = [
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip",
    "https://archive.ics.uci.edu/static/public/240/uci%20har%20dataset.zip",
]
UCIHAR_ZIP_PATH = DATA_DIR / "UCI_HAR_Dataset.zip"
UCIHAR_EXTRACT_DIR = DATA_DIR / "UCI_HAR_Dataset"

# MHEALTH (multiple mirrors; first working one will be used)
MHEALTH_URLS = [
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00319/MHEALTHDATASET.zip",
    "https://archive.ics.uci.edu/static/public/319/mhealth%20dataset.zip",
]
MHEALTH_ZIP_PATH = DATA_DIR / "MHEALTH.zip"
MHEALTH_EXTRACT_DIR = DATA_DIR / "MHEALTHDATASET"

# -----------------------------------------------------------------------------
# 1) DEVICE SELECTION (fixed cuda:0 if available)
# -----------------------------------------------------------------------------

if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
    print("Using device: cuda:0")
else:
    DEVICE = torch.device("cpu")
    print("CUDA not available, using CPU")

ram = psutil.virtual_memory()
print(f"RAM Usage: {(ram.total-ram.available)/1e9:.1f}/{ram.total/1e9:.1f} GiB ({ram.percent}%)\n")

# -----------------------------------------------------------------------------
# 2) HELPERS: robust download
# -----------------------------------------------------------------------------

def _download_first_ok(urls, dst: Path) -> bool:
    dst.parent.mkdir(parents=True, exist_ok=True)
    for u in urls:
        try:
            print(f"[INFO] Trying download: {u}")
            urllib.request.urlretrieve(u, dst)
            print("[INFO] Downloaded successfully.")
            return True
        except Exception as e:
            print(f"[WARN] Download failed for {u}: {e}")
    return False

# -----------------------------------------------------------------------------
# 3) WISDM & UCI HAR LOADING / PREPROCESSING
# -----------------------------------------------------------------------------

def download_wisdm():
    if WISDM_PATH.exists():
        print(f"[INFO] Found WISDM file at {WISDM_PATH}")
        return
    print(f"[INFO] Downloading WISDM from {WISDM_URL} ...")
    urllib.request.urlretrieve(WISDM_URL, WISDM_PATH)
    print("[INFO] Download complete.")

def load_wisdm_dataframe(path: Path) -> pd.DataFrame:
    """Load and clean WISDM dataset."""
    print("[INFO] Loading WISDM data into DataFrame...")
    columns = ["user", "activity", "timestamp", "x", "y", "z"]
    df = pd.read_csv(
        path,
        header=None,
        names=columns,
        engine="python",
        on_bad_lines="skip",
    )

    df = df.dropna()
    df["z"] = df["z"].astype(str).str.replace(";", "", regex=False)
    df["z"] = df["z"].astype(float)
    df = df[df["timestamp"] != 0]
    df = df.sort_values(by=["user", "timestamp"], ignore_index=True)

    print("[INFO] Cleaned WISDM shape:", df.shape)
    return df

def segment_wisdm(df: pd.DataFrame,
                  window_size: int = WINDOW_SIZE,
                  step_size: int = STEP_SIZE):
    """Segment accelerometer data into windows and compute magnitude."""
    print("[INFO] Segmenting WISDM time series into windows...")
    X_segments = []
    y_labels = []

    users = df["user"].unique()
    for user in users:
        df_user = df[df["user"] == user]
        activities = df_user["activity"].unique()

        for act in activities:
            df_ua = df_user[df_user["activity"] == act]
            signal = df_ua[["x", "y", "z"]].values  # [T, 3]

            for start in range(0, len(signal) - window_size, step_size):
                end = start + window_size
                window = signal[start:end]  # [window_size, 3]
                mag = np.sqrt((window ** 2).sum(axis=1))  # [window_size]
                X_segments.append(mag)
                y_labels.append(act)

    X_segments = np.array(X_segments, dtype=np.float32)
    y_labels = np.array(y_labels)
    print("[INFO] WISDM Segmented X shape:", X_segments.shape)
    print("[INFO] WISDM Label distribution (raw):\n", pd.Series(y_labels).value_counts())
    return X_segments, y_labels

def download_uci_har():
    if UCIHAR_EXTRACT_DIR.exists():
        print(f"[INFO] Found UCI HAR at {UCIHAR_EXTRACT_DIR}")
        return
    if not UCIHAR_ZIP_PATH.exists():
        print(f"[INFO] Downloading UCI HAR ...")
        ok = _download_first_ok(UCIHAR_URLS, UCIHAR_ZIP_PATH)
        if not ok:
            raise FileNotFoundError("Could not download UCI HAR dataset from known mirrors.")
    print(f"[INFO] Extracting UCI HAR zip to {UCIHAR_EXTRACT_DIR} ...")
    with zipfile.ZipFile(UCIHAR_ZIP_PATH, "r") as zf:
        zf.extractall(UCIHAR_EXTRACT_DIR)
    print("[INFO] Extraction complete.")

def load_uci_har_magnitude():
    """
    Use body_acc_x/y/z from UCI HAR to build magnitude windows:
      - Each window length = 128
      - Combine train + test
      - Labels: 6 activities (0..5)
    """
    base = UCIHAR_EXTRACT_DIR / "UCI HAR Dataset"

    def load_split(split: str):
        split_dir = base / split
        sig_dir = split_dir / "Inertial Signals"

        # Each: [N_windows, 128]
        x = np.loadtxt(sig_dir / f"body_acc_x_{split}.txt")
        y = np.loadtxt(sig_dir / f"body_acc_y_{split}.txt")
        z = np.loadtxt(sig_dir / f"body_acc_z_{split}.txt")

        mag = np.sqrt(x**2 + y**2 + z**2).astype(np.float32)  # [N, 128]

        # Labels are 1..6 -> 0..5
        y_lab = np.loadtxt(split_dir / f"y_{split}.txt").astype(int) - 1
        return mag, y_lab

    X_train, y_train = load_split("train")
    X_test,  y_test  = load_split("test")

    X_all = np.concatenate([X_train, X_test], axis=0)
    y_all = np.concatenate([y_train, y_test], axis=0)

    print("[INFO] UCI HAR magnitude shape:", X_all.shape)
    print("[INFO] UCI HAR label distribution:",
          dict(zip(*np.unique(y_all, return_counts=True))))
    return X_all, y_all

# -----------------------------------------------------------------------------
# 4) MHEALTH LOADING / PREPROCESSING
# -----------------------------------------------------------------------------

def download_mhealth():
    if MHEALTH_EXTRACT_DIR.exists() and any(MHEALTH_EXTRACT_DIR.glob("mHealth_subject*.log")):
        print(f"[INFO] Found MHEALTH at {MHEALTH_EXTRACT_DIR}")
        return
    if not MHEALTH_ZIP_PATH.exists():
        print("[INFO] Downloading MHEALTH ...")
        ok = _download_first_ok(MHEALTH_URLS, MHEALTH_ZIP_PATH)
        if not ok:
            raise FileNotFoundError("Could not download MHEALTH dataset from known mirrors.")
    print(f"[INFO] Extracting MHEALTH zip to {DATA_DIR} ...")
    with zipfile.ZipFile(MHEALTH_ZIP_PATH, "r") as zf:
        zf.extractall(DATA_DIR)
    # After extract, ensure we have MHEALTH_EXTRACT_DIR path pointing to folder containing mHealth_subject*.log
    # Some mirrors nest the folder; try to find it and normalize.
    candidates = list(DATA_DIR.glob("**/mHealth_subject*.log"))
    if candidates:
        target_dir = candidates[0].parent
        if target_dir != MHEALTH_EXTRACT_DIR:
            MHEALTH_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
            # Copy all subject logs into the normalized directory
            for f in target_dir.glob("mHealth_subject*.log"):
                shutil.copy2(f, MHEALTH_EXTRACT_DIR / f.name)
    print("[INFO] MHEALTH ready.")

def load_mhealth_magnitude(window_size: int = 256, step_size: int = 128):
    """
    Build magnitude windows from MHEALTH subject logs.

    - Reads all mHealth_subject*.log files.
    - Treats the **last column** as activity label (int); 0 = 'null'.
    - Treats all preceding columns as numeric signals; computes per-sample magnitude
      sqrt(sum(channels^2)) to keep 1D input compatible with existing models.
    - Builds sliding windows; assigns window label by majority vote over non-zero labels.
      Windows with no non-zero majority are dropped.
    """
    files = sorted(MHEALTH_EXTRACT_DIR.glob("mHealth_subject*.log"))
    if not files:
        raise FileNotFoundError(
            f"No subject logs found under {MHEALTH_EXTRACT_DIR}. "
            "Check extraction or update the path."
        )

    mags = []
    labs = []

    for fp in files:
        arr = np.loadtxt(fp)  # whitespace-separated
        if arr.ndim == 1:
            arr = arr.reshape(1, -1)
        if arr.shape[1] < 2:
            continue
        signals = arr[:, :-1].astype(np.float32)   # all but last
        labels  = arr[:, -1].astype(int)           # last col

        # normalize each channel for stability (optional but helpful)
        sig = signals
        sig = (sig - sig.mean(axis=0, keepdims=True)) / (sig.std(axis=0, keepdims=True) + 1e-8)

        # magnitude over *all* channels to keep input as 1D
        mag = np.sqrt((sig ** 2).sum(axis=1))  # [T]
        mags.append(mag)
        labs.append(labels)

    x_all = np.concatenate(mags, axis=0)
    y_all = np.concatenate(labs, axis=0)
    print("[INFO] MHEALTH raw timeline:", x_all.shape, y_all.shape)

    # Sliding windows
    X_segments = []
    y_labels   = []
    T = len(x_all)
    for start in range(0, T - window_size, step_size):
        end = start + window_size
        win_mag = x_all[start:end]             # [L]
        win_lab = y_all[start:end]             # [L]

        # majority vote over non-zero labels
        nonzero = win_lab[win_lab != 0]
        if nonzero.size == 0:
            continue
        vals, cnts = np.unique(nonzero, return_counts=True)
        maj = vals[np.argmax(cnts)]            # in {1..12}
        X_segments.append(win_mag.astype(np.float32))
        y_labels.append(int(maj))

    X_segments = np.stack(X_segments, axis=0) if X_segments else np.empty((0, window_size), dtype=np.float32)
    y_labels   = np.array(y_labels, dtype=int)
    print("[INFO] MHEALTH segmented X shape:", X_segments.shape)
    print("[INFO] MHEALTH label distribution (raw, 1..12):",
          dict(zip(*np.unique(y_labels, return_counts=True))))
    return X_segments, y_labels

# -----------------------------------------------------------------------------
# 5) BALANCING
# -----------------------------------------------------------------------------

def balance_by_min_class(X: np.ndarray, y_int: np.ndarray, seed: int = 42):
    """
    Downsample all classes to the smallest class size.
    """
    rng = np.random.default_rng(seed)
    unique_classes = np.unique(y_int)
    indices_per_class = {c: np.where(y_int == c)[0] for c in unique_classes}
    class_sizes = {c: len(idx) for c, idx in indices_per_class.items()}
    min_count = min(class_sizes.values())

    print("[INFO] Class sizes before balancing:", class_sizes)
    print("[INFO] Using min_count =", min_count, "samples per class")

    balanced_indices_list = []
    for c in unique_classes:
        idxs = indices_per_class[c]
        if len(idxs) > min_count:
            chosen = rng.choice(idxs, size=min_count, replace=False)
        else:
            chosen = idxs
        balanced_indices_list.append(chosen)

    balanced_indices = np.concatenate(balanced_indices_list)
    rng.shuffle(balanced_indices)

    balanced_counts = dict(zip(unique_classes, np.bincount(y_int[balanced_indices])[unique_classes]))
    print("[INFO] Class sizes after balancing:", balanced_counts)
    return X[balanced_indices], y_int[balanced_indices]

# -----------------------------------------------------------------------------
# 6) DATASET CLASS
# -----------------------------------------------------------------------------

class WISDMDataset(Dataset):
    """
    Generic window-level dataset for 1D magnitude signals.
    Each sample: x -> [1, L], y -> integer label.
    """

    def __init__(self, X_windows: np.ndarray, y_int: np.ndarray):
        X_tensor = torch.tensor(X_windows, dtype=torch.float32)
        y_tensor = torch.tensor(y_int, dtype=torch.long)

        # Per-window normalization (z-score)
        means = X_tensor.mean(dim=1, keepdim=True)
        stds = X_tensor.std(dim=1, keepdim=True) + 1e-8
        X_tensor = (X_tensor - means) / stds

        self.X = X_tensor
        self.y = y_tensor

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx].unsqueeze(0)  # [1, L]
        y = self.y[idx]
        return x, y

# -----------------------------------------------------------------------------
# 7) STRONGER CNN BACKBONE (for QTModel)
# -----------------------------------------------------------------------------

class EnhancedCNN1D(nn.Module):
    """
    Strong 1D CNN backbone for QTModel.
    Input:  [B, 1, L]
    Output: [B, 64]
    """
    def __init__(self, in_ch: int = 1):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv1d(in_ch, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2)   # L -> L/2
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2)   # L/2 -> L/4
        )
        self.block3 = nn.Sequential(
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)  # [B,64,1]
        )
        self.dropout = nn.Dropout(p=0.2)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.dropout(x)
        return x.flatten(1)  # [B, 64]

# -----------------------------------------------------------------------------
# 8) TORCHQUANTUM FEATURE MAP
# -----------------------------------------------------------------------------

class QuantumFeatureMapTQ(tq.QuantumModule):
    """
    TorchQuantum-based feature map:
     - Angle embedding via RY
     - Ring entangling CNOT chain
     - Measure all Z
    """
    def __init__(self, n_wires=4, n_layers=4):
        super().__init__()
        self.n_wires  = n_wires
        self.n_layers = n_layers
        self.measure  = tq.MeasureAll(tq.PauliZ)

    def forward(self, qdev, angles):
        # angles: (batch, n_wires)
        for w in range(self.n_wires):
            qdev.ry(wires=w, params=angles[:, w], static=self.static_mode)
        for _ in range(self.n_layers):
            for i in range(self.n_wires - 1):
                qdev.cnot(wires=[i, i+1])
            qdev.cnot(wires=[self.n_wires-1, 0])
        return self.measure(qdev)  # (batch, n_wires)

# -----------------------------------------------------------------------------
# 9) QUANTUM BLOCK WITH SELF-ATTENTION + FFN
# -----------------------------------------------------------------------------

class QBlockQT(tq.QuantumModule):
    """
    Single quantum block for QTModel (see docstring in original file).
    """
    def __init__(self, dim=64, n_wires=4, n_layers=4, ff_mult=2, dropout=0.1):
        super().__init__()
        self.dim     = dim
        self.n_wires = n_wires

        # QFM for Q, K, V
        self.q_fm = QuantumFeatureMapTQ(n_wires, n_layers)
        self.k_fm = QuantumFeatureMapTQ(n_wires, n_layers)
        self.v_fm = QuantumFeatureMapTQ(n_wires, n_layers)

        # Map input features -> Q/K/V angles
        self.reduce = nn.Linear(dim, 3 * n_wires)

        # Project Q/K/V outputs up to feature dim
        self.q_proj = nn.Linear(n_wires, dim)
        self.k_proj = nn.Linear(n_wires, dim)
        self.v_proj = nn.Linear(n_wires, dim)

        # Transformer-style FFN
        self.ff = nn.Sequential(
            nn.Linear(dim, ff_mult * dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_mult * dim, dim),
        )

        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        # x: [B, D]
        B, D = x.shape

        # Classical to quantum angles
        r = self.reduce(x)                      # [B, 3 * n_wires]
        q_ang, k_ang, v_ang = r.chunk(3, dim=1) # each [B, n_wires]

        # Quantum device
        qdev = tq.QuantumDevice(
            n_wires=self.n_wires,
            bsz=B,
            device=x.device
        )

        # QFM for Q, K, V
        q_out = self.q_fm(qdev, q_ang)  # [B, n_wires]
        k_out = self.k_fm(qdev, k_ang)
        v_out = self.v_fm(qdev, v_ang)

        # Project to feature dimension
        q = self.q_proj(q_out)  # [B, D]
        k = self.k_proj(k_out)  # [B, D]
        v = self.v_proj(v_out)  # [B, D]

        # Self-attention (element-wise gating)
        scale = float(D) ** 0.5
        scores = torch.softmax((q * k) / scale, dim=-1)  # [B, D]
        attn_out = scores * v                            # [B, D]

        # Residual + FFN + norms
        z = x + attn_out          # first residual
        z_norm = self.norm1(z)
        ff_out = self.ff(z_norm)
        y = self.norm2(z_norm + ff_out)  # second residual
        return y

# -----------------------------------------------------------------------------
# 10) QTModel: CNN -> stacked QBlocks -> quantum gate -> classifier
# -----------------------------------------------------------------------------

class QTModel(nn.Module):
    """
    QTModel full stack.
    """
    def __init__(self, num_cls, in_ch=1,
                 n_blocks=Q_BLOCKS, n_wires=QSize, n_layers=Q_LAYERS,
                 dim=64, ff_mult=2, dropout=0.1):
        super().__init__()

        self.dim      = dim
        self.n_wires  = n_wires
        self.cnn      = EnhancedCNN1D(in_ch)

        # Stacked quantum blocks
        self.blocks = nn.ModuleList([
            QBlockQT(dim=dim, n_wires=n_wires, n_layers=n_layers,
                     ff_mult=ff_mult, dropout=dropout)
            for _ in range(n_blocks)
        ])

        # Final quantum gate
        self.gate_linear = nn.Linear(dim, n_wires)
        self.gate_qfm    = QuantumFeatureMapTQ(n_wires, n_layers)

        # Classifier on fused [features + quantum gate]
        self.cls_head = nn.Sequential(
            nn.LayerNorm(dim + n_wires),
            nn.Linear(dim + n_wires, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, num_cls),
        )

    def forward(self, x):
        # CNN backbone
        e = self.cnn(x)  # [B, 64]

        # Quantum blocks
        for block in self.blocks:
            e = block(e)  # [B, 64]

        # Final quantum gate
        angles = self.gate_linear(e)  # [B, n_wires]
        qdev = tq.QuantumDevice(
            n_wires=self.n_wires,
            bsz=angles.size(0),
            device=angles.device
        )
        q_gate = self.gate_qfm(qdev, angles)  # [B, n_wires]

        fused = torch.cat([e, q_gate], dim=1)  # [B, 64 + n_wires]
        logits = self.cls_head(fused)          # raw logits
        return logits

# -----------------------------------------------------------------------------
# 11) BASELINE MODELS (unchanged)
# -----------------------------------------------------------------------------

def conv_dw(in_planes, out_planes, kernel_size, stride=1, dilation=1):
    return nn.Sequential(
        nn.Conv1d(in_planes, in_planes, kernel_size=kernel_size,
                  stride=stride, padding=(kernel_size//2)*dilation, dilation=dilation,
                  groups=in_planes, bias=False),
        nn.Conv1d(in_planes, out_planes, kernel_size=1, bias=False)
    )

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool1d(1)
        self.max = nn.AdaptiveMaxPool1d(1)
        red = max(1, in_planes // ratio)
        self.fc = nn.Sequential(
            nn.Conv1d(in_planes, red, 1, bias=False), nn.ReLU(),
            nn.Conv1d(red, in_planes, 1, bias=False)
        )
        self.sig = nn.Sigmoid()
    def forward(self, x):
        return self.sig(self.fc(self.avg(x)) + self.fc(self.max(x)))

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv1 = nn.Conv1d(2, 1, k, padding=k//2, bias=False)
        self.sig = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sig(self.conv1(torch.cat([avg_out, max_out], dim=1)))

class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, out_planes, k, stride=1, dilation=1, downsample=None):
        super().__init__()
        self.conv1 = conv_dw(in_planes, out_planes, k, stride, dilation)
        self.bn1 = nn.BatchNorm1d(out_planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv_dw(out_planes, out_planes, k)
        self.bn2 = nn.BatchNorm1d(out_planes)
        self.ca = ChannelAttention(out_planes)
        self.sa = SpatialAttention()
        self.downsample = downsample
    def forward(self, x):
        res = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.ca(out)*out
        out = self.sa(out)*out
        if self.downsample is not None:
            res = self.downsample(x)
        return self.relu(out + res)

class DeepILS(nn.Module):
    def __init__(self, num_inputs=1, num_classes=6, block=BasicBlock1D, group_sizes=(2,2,2,2), base=64, k=3):
        super().__init__()
        self.inp = nn.Sequential(
            nn.Conv1d(num_inputs, base, 5, 2, 3, bias=False),
            nn.BatchNorm1d(base),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(3,2,1)
        )
        self.inplanes = base
        self.groups = nn.ModuleList()
        planes = [base*(2**i) for i in range(len(group_sizes))]
        strides = [1] + [2]*(len(group_sizes)-1)
        for i, blocks in enumerate(group_sizes):
            pl = planes[i]
            st = strides[i]
            down = None
            if st != 1 or self.inplanes != pl*block.expansion:
                down = nn.Sequential(
                    nn.Conv1d(self.inplanes, pl*block.expansion, 1, st, bias=False),
                    nn.BatchNorm1d(pl*block.expansion)
                )
            layers = [block(self.inplanes, pl, k, stride=st, downsample=down)]
            self.inplanes = pl*block.expansion
            for _ in range(1, blocks):
                layers.append(block(self.inplanes, pl, k))
            self.groups.append(nn.Sequential(*layers))
        self.groups = nn.Sequential(*self.groups)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(planes[-1]*block.expansion, num_classes)
        )
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
    def forward(self, x):
        x = self.inp(x)
        x = self.groups(x)
        return self.head(x)

class ResNet1D(DeepILS):
    pass

class Swish(nn.Module):
    def forward(self, x): return x*torch.sigmoid(x)

def Conv_3x3(inp, oup, s):
    return nn.Sequential(
        nn.Conv1d(inp,oup,3,s,1,bias=False),
        nn.BatchNorm1d(oup),
        Swish()
    )

def Conv_1x1(inp, oup):
    return nn.Sequential(
        nn.Conv1d(inp,oup,1,1,0,bias=False),
        nn.BatchNorm1d(oup),
        Swish()
    )

class InvertedResidual_Eff(nn.Module):
    def __init__(self, inp, oup, s, t, k):
        super().__init__()
        self.use=(s==1 and inp==oup)
        hid=inp*t
        self.conv=nn.Sequential(
            nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), Swish(),
            nn.Conv1d(hid,hid,k,s,k//2,groups=hid,bias=False), nn.BatchNorm1d(hid), Swish(),
            nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
        )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

class EfficientNetB0_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        setting=[[1,16,1,1,3],[6,24,2,2,3],[6,40,2,2,5],
                 [6,80,3,2,3],[6,112,3,1,5],[6,192,4,2,5],[6,320,1,1,3]]
        in_ch=int(32*wm)
        last=1280 if wm<=1 else int(1280*wm)
        feats=[Conv_3x3(1,in_ch,2)]
        ch=in_ch
        for t,c,n,s,k in setting:
            oc=int(c*wm)
            for i in range(n):
                feats.append(InvertedResidual_Eff(ch, oc, s if i==0 else 1, t, k))
                ch=oc
        feats += [Conv_1x1(ch,last), nn.AdaptiveAvgPool1d(1)]
        self.features=nn.Sequential(*feats)
        self.fc=nn.Linear(last,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        return self.fc(self.features(x).squeeze(-1))

class InvertedResidual_Mnas(nn.Module):
    def __init__(self, inp, oup, s, t, k):
        super().__init__()
        self.use=(s==1 and inp==oup)
        hid=inp*t
        self.conv=nn.Sequential(
            nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
            nn.Conv1d(hid,hid,k,s,k//2,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
            nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
        )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

def SepConv_3x3(inp,oup):
    return nn.Sequential(
        nn.Conv1d(inp,inp,3,1,1,groups=inp,bias=False), nn.BatchNorm1d(inp), nn.ReLU6(inplace=True),
        nn.Conv1d(inp,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
    )

def Conv3_3(inp,oup,s):
    return nn.Sequential(
        nn.Conv1d(inp,oup,3,s,1,bias=False), nn.BatchNorm1d(oup), nn.ReLU6(inplace=True)
    )

def Conv1_1(inp,oup):
    return nn.Sequential(
        nn.Conv1d(inp,oup,1,1,0,bias=False), nn.BatchNorm1d(oup), nn.ReLU6(inplace=True)
    )

class MnasNet_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        setting=[[3,24,3,2,3],[3,40,3,2,5],[6,80,3,2,5],
                 [6,96,2,1,3],[6,192,4,2,5],[6,320,1,1,3]]
        in_ch=int(32*wm)
        last=1280 if wm<=1 else int(1280*wm)
        feats=[Conv3_3(1,in_ch,2), SepConv_3x3(in_ch,16)]
        ch=16
        for t,c,n,s,k in setting:
            oc=int(c*wm)
            for i in range(n):
                feats.append(InvertedResidual_Mnas(ch, oc, s if i==0 else 1, t, k))
                ch=oc
        feats += [Conv1_1(ch,last), nn.AdaptiveAvgPool1d(1)]
        self.features=nn.Sequential(*feats)
        self.fc=nn.Linear(last,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        return self.fc(self.features(x).squeeze(-1))

class DSConv_Mobile(nn.Module):
    def __init__(self, f3, f1, s=1, p=1):
        super().__init__()
        self.seq=nn.Sequential(
            nn.Conv1d(f3,f3,3,s,p,groups=f3,bias=False), nn.BatchNorm1d(f3), nn.ReLU(inplace=True),
            nn.Conv1d(f3,f1,1,1,0,bias=False), nn.BatchNorm1d(f1), nn.ReLU(inplace=True)
        )
    def forward(self,x): return self.seq(x)

class MobileNet_1D(nn.Module):
    def __init__(self, channels=[32,64,128,256,512,1024], wm=1.0, n_classes=6):
        super().__init__()
        ch=[int(c*wm) for c in channels]
        self.conv=nn.Sequential(
            nn.Conv1d(1,ch[0],3,2,1,bias=False), nn.BatchNorm1d(ch[0]), nn.ReLU(inplace=True)
        )
        self.features=nn.Sequential(
            DSConv_Mobile(ch[0],ch[1],1,1),
            DSConv_Mobile(ch[1],ch[2],2,1),
            DSConv_Mobile(ch[2],ch[2],1,1),
            DSConv_Mobile(ch[2],ch[3],2,1),
            DSConv_Mobile(ch[3],ch[3],1,1),
            DSConv_Mobile(ch[3],ch[4],2,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[5],2,1),
            DSConv_Mobile(ch[5],ch[5],1,1),
        )
        self.avg=nn.AdaptiveAvgPool1d(1)
        self.fc=nn.Linear(ch[5],n_classes)
    def forward(self,x):
        x=self.conv(x)
        x=self.features(x)
        x=self.avg(x).squeeze(-1)
        return self.fc(x)

class InvertedResidual_V2(nn.Module):
    def __init__(self, inp, oup, s, t):
        super().__init__()
        hid=int(inp*t)
        self.use=(s==1 and inp==oup)
        if t==1:
            self.conv=nn.Sequential(
                nn.Conv1d(hid,hid,3,s,1,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
            )
        else:
            self.conv=nn.Sequential(
                nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,hid,3,s,1,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
            )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

def make_divisible(x, d=8):
    return int((x + d - 1) // d * d)

class MobileNetV2_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        input_channel=32
        last=1280
        setting=[[1,16,1,1],[6,24,2,2],[6,32,3,2],[6,64,4,2],
                 [6,96,3,1],[6,160,3,2],[6,320,1,1]]
        self.last_ch=make_divisible(last*wm) if wm>1 else last
        feats=[nn.Conv1d(1,input_channel,3,2,1,bias=False), nn.BatchNorm1d(input_channel), nn.ReLU6(inplace=True)]
        ch=input_channel
        for t,c,n,s in setting:
            oc=make_divisible(c*wm) if t>1 else c
            for i in range(n):
                feats.append(InvertedResidual_V2(ch, oc, s if i==0 else 1, t))
                ch=oc
        feats += [nn.Conv1d(ch,self.last_ch,1,1,0,bias=False), nn.BatchNorm1d(self.last_ch), nn.ReLU6(inplace=True)]
        self.features=nn.Sequential(*feats)
        self.pool=nn.AdaptiveAvgPool1d(1)
        self.fc=nn.Linear(self.last_ch,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        x=self.features(x)
        x=self.pool(x).squeeze(-1)
        return self.fc(x)

# --- TSLANet (compact) ---
class TSLANet(nn.Module):
    def __init__(self, C_in, T_in, n_classes=6, emb=128, depth=3, patch=8, drop=0.10):
        super().__init__()
        stride=max(1, patch//2)
        self.proj=nn.Conv1d(C_in,emb,patch,stride)
        N=int((T_in-patch)/stride+1)
        self.pos=nn.Parameter(torch.zeros(1,N,emb))
        self.blocks=nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(emb),
                nn.Identity(),  # spectral placeholder
                nn.LayerNorm(emb),
                nn.Sequential(
                    nn.Linear(emb,int(emb*3)), nn.GELU(),
                    nn.Linear(int(emb*3),emb)
                )
            ) for _ in range(depth)
        ])
        self.drop=drop
        self.head=nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(emb,emb//2),
            nn.GELU(),
            nn.Linear(emb//2,n_classes)
        )
    def forward(self,x):
        x=self.proj(x).flatten(2).transpose(1,2)
        x=x+self.pos
        x=F.dropout(x,p=self.drop,training=self.training)
        for b in self.blocks:
            x=x + b[3](b[1](b[0](x)))
        return self.head(x.mean(1))

# -----------------------------------------------------------------------------
# 12) TRAIN / EVAL HELPERS
# -----------------------------------------------------------------------------

def process_target(y: torch.Tensor) -> torch.Tensor:
    if y.dim() > 1:
        if y.size(1) > 1:
            y = y.argmax(dim=1)
        else:
            y = y.squeeze(1)
    return y.long()

def train_one(model, loader, opt):
    model.train()
    losses = []
    for x, y in loader:
        x = x.to(DEVICE)
        y = process_target(y.to(DEVICE))

        out = model(x)               # logits
        loss = F.cross_entropy(out, y)

        opt.zero_grad()
        loss.backward()
        opt.step()

        losses.append(loss.item())
    return float(np.mean(losses)) if losses else 0.0

def eval_fold(model, loader, num_cls):
    """Evaluate on one fold: return y_true and probabilities."""
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = process_target(y.to(DEVICE))
            out  = model(x)  # logits
            prob = F.softmax(out, dim=1).cpu().numpy()
            ys.append(y.cpu().numpy())
            ps.append(prob)
    ys = np.concatenate(ys)
    ps = np.concatenate(ps)
    return ys, ps

def run_cv_for_model(model_name, model_ctor, num_cls, dataset):
    N = len(dataset)
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=0)

    metrics = {"acc": [], "f1": [], "prec": [], "rec": [], "auc": []}
    loss_hist = []
    all_y = []
    all_probs = []

    print(f"\n====== Running 6-fold CV for {model_name} ======")

    for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(N)), start=1):
        print(f"\n--- {model_name} Fold {fold}/{N_SPLITS} ---")

        train_loader = DataLoader(
            Subset(dataset, train_idx),
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=4,
            pin_memory=True
        )
        val_loader = DataLoader(
            Subset(dataset, val_idx),
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=4,
            pin_memory=True
        )

        model = model_ctor().to(DEVICE)
        opt   = optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-4)
        sch   = CosineAnnealingLR(opt, T_max=EPOCHS)

        fold_losses = []
        for _ in trange(EPOCHS, desc=f"{model_name}-F{fold}", leave=False):
            loss = train_one(model, train_loader, opt)
            sch.step()
            fold_losses.append(loss)
        loss_hist.append(fold_losses)

        y_fold, p_fold = eval_fold(model, val_loader, num_cls)
        preds = p_fold.argmax(axis=1)

        acc  = accuracy_score(y_fold, preds)
        f1   = f1_score(y_fold, preds, average="weighted")
        prec = precision_score(y_fold, preds, average="weighted")
        rec  = recall_score(y_fold, preds, average="weighted")

        if num_cls == 2:
            auc_val = roc_auc_score(y_fold, p_fold[:,1])
        else:
            y_bin = label_binarize(y_fold, classes=list(range(num_cls)))
            auc_val = roc_auc_score(y_bin, p_fold, average="micro", multi_class="ovo")

        print(f" Fold {fold}: Acc={acc:.4f} F1={f1:.4f} Prec={prec:.4f} Rec={rec:.4f} AUC={auc_val:.4f}")
        for k,v in zip(("acc","f1","prec","rec","auc"), (acc,f1,prec,rec,auc_val)):
            metrics[k].append(v)

        all_y.append(y_fold)
        all_probs.append(p_fold)

        # free model this fold
        del model
        torch.cuda.empty_cache()

    all_y = np.concatenate(all_y)
    all_probs = np.concatenate(all_probs, axis=0)

    # Global ROC for the model
    if num_cls == 2:
        fpr, tpr, _ = roc_curve(all_y, all_probs[:,1], pos_label=1)
        auc_global = roc_auc_score(all_y, all_probs[:,1])
    else:
        y_bin = label_binarize(all_y, classes=list(range(num_cls)))
        fpr, tpr, _ = roc_curve(y_bin.ravel(), all_probs.ravel())
        auc_global = roc_auc_score(y_bin, all_probs, average="micro", multi_class="ovo")

    return metrics, loss_hist, all_y, all_probs, (fpr, tpr, auc_global)

# -----------------------------------------------------------------------------
# 13) MAIN: DATA CHOICE + MODELS + PLOTS
# -----------------------------------------------------------------------------

def main():
    global WINDOW_SIZE, STEP_SIZE

    # -----------------------
    # Dataset choice
    # -----------------------
    if DATASET == "WISDM":
        download_wisdm()
        df = load_wisdm_dataframe(WISDM_PATH)
        X_segments, y_labels = segment_wisdm(df, window_size=200, step_size=100)

        labels_cat = pd.Series(y_labels).astype("category")
        class_names = list(labels_cat.cat.categories)
        y_int = labels_cat.cat.codes.to_numpy().astype(np.int64)

        WINDOW_SIZE = X_segments.shape[1]

    elif DATASET == "UCIHAR":
        download_uci_har()
        X_segments, y_int = load_uci_har_magnitude()

        class_names = [
            "WALKING",
            "WALK_UP",
            "WALK_DOWN",
            "SITTING",
            "STANDING",
            "LAYING",
        ]
        WINDOW_SIZE = X_segments.shape[1]

    elif DATASET == "MHEALTH":
        download_mhealth()
        # MHEALTH has higher native sampling; use a slightly larger window/step
        WINDOW_SIZE = 256
        STEP_SIZE   = 128
        X_segments, y_labels = load_mhealth_magnitude(window_size=WINDOW_SIZE, step_size=STEP_SIZE)

        # labels are 1..12; map to 0..11 (contiguous)
        uniq = np.sort(np.unique(y_labels))
        # build a compact map {label_value: 0..C-1}
        lab_to_idx = {lab:i for i, lab in enumerate(uniq)}
        y_int = np.array([lab_to_idx[int(v)] for v in y_labels], dtype=np.int64)

        # class names generic (A1..Ak). Replace with human-readable list if desired.
        class_names = [f"A{i}" for i in range(1, len(uniq)+1)]

    else:
        raise ValueError(f"Unknown DATASET: {DATASET}")

    print("[INFO] Using dataset:", DATASET)
    print("[INFO] Classes:", class_names)

    # Balance dataset
    X_bal, y_bal = balance_by_min_class(X_segments, y_int, seed=42)
    dataset = WISDMDataset(X_bal, y_bal)
    N = len(dataset)
    print(f"[INFO] Balanced dataset size: {N}")

    num_cls = len(class_names)
    seq_len = X_bal.shape[1]
    dims = (1, seq_len)

    # ------------------------
    # Model factories
    # ------------------------
    model_factories = {
        "QTModel":           lambda: QTModel(num_cls=num_cls, in_ch=1),
        "ResNet1D":          lambda: ResNet1D(num_inputs=1, num_classes=num_cls),
        "EfficientNetB0_1D": lambda: EfficientNetB0_1D(n_classes=num_cls),
        "MnasNet_1D":        lambda: MnasNet_1D(n_classes=num_cls),
        "MobileNet_1D":      lambda: MobileNet_1D(n_classes=num_cls),
        "MobileNetV2_1D":    lambda: MobileNetV2_1D(n_classes=num_cls),
        "TSLANet":           lambda: TSLANet(C_in=1, T_in=seq_len, n_classes=num_cls),
    }

    # ------------------------
    # Complexity (params/FLOPs) for all models
    # ------------------------
    complexity = {}
    for name, ctor in model_factories.items():
        model = ctor().to(DEVICE)
        try:
            macs, params = get_model_complexity_info(
                model, dims,
                as_strings=True,
                print_per_layer_stat=False
            )
        except Exception as e:
            print(f"[WARN] ptflops failed on {name}: {e}")
            macs, params = "N/A", "N/A"
        complexity[name] = (params, macs)
        del model
        torch.cuda.empty_cache()
        print(f"[INFO] {name}: params={params}, FLOPs={macs}")

    # ------------------------
    # Run CV for each model
    # ------------------------
    all_metrics  = {}
    all_losses   = {}
    all_yprobs   = {}
    all_rocs     = {}

    for name, ctor in model_factories.items():
        metrics, loss_hist, y_all, p_all, roc_info = run_cv_for_model(
            name, ctor, num_cls, dataset
        )
        all_metrics[name] = metrics
        all_losses[name]  = loss_hist
        all_yprobs[name]  = (y_all, p_all)
        all_rocs[name]    = roc_info

    # Summary table
    gpu_mem = None
    if DEVICE.type == "cuda":
        gpu_mem = (torch.cuda.memory_allocated()/1e6,
                   torch.cuda.memory_reserved()/1e6)

    summary_rows = []
    for name in model_factories.keys():
        m = all_metrics[name]
        params, macs = complexity.get(name, ("N/A", "N/A"))

        row = {
            "model":        name,
            "dataset":      DATASET,
            "acc_mean±sd":  f"{np.mean(m['acc']):.4f}±{np.std(m['acc']):.4f}",
            "f1_mean±sd":   f"{np.mean(m['f1']):.4f}±{np.std(m['f1']):.4f}",
            "prec_mean±sd": f"{np.mean(m['prec']):.4f}±{np.std(m['prec']):.4f}",
            "rec_mean±sd":  f"{np.mean(m['rec']):.4f}±{np.std(m['rec']):.4f}",
            "auc_mean±sd":  f"{np.mean(m['auc']):.4f}±{np.std(m['auc']):.4f}",
            "GPU_mem_MB":   gpu_mem,
            "params":       params,
            "FLOPs":        macs,
        }
        summary_rows.append(row)

    df_sum = pd.DataFrame(summary_rows)
    print("\n[SUMMARY COMPARISON]")
    print(df_sum.to_markdown(index=False))

    # -------------------------------------------------
    # PLOTS
    # -------------------------------------------------

    # 1) Training Loss Curves (mean over folds) per model
    plt.figure()
    for name, loss_hist in all_losses.items():
        loss_arr = np.array(loss_hist)  # [n_folds, epochs]
        mean_loss = loss_arr.mean(axis=0)
        plt.plot(range(1, EPOCHS+1), mean_loss, label=name)
    plt.xlabel("Epoch")
    plt.ylabel("Training Loss")
    plt.title(f"Training Loss Curves (mean over folds) - {DATASET}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # 2) ROC curves per model (Validation)
    plt.figure()
    for name, (fpr, tpr, auc_val) in all_rocs.items():
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})")
    plt.plot([0,1],[0,1],'k--',alpha=0.5)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curves (Validation, all folds aggregated) - {DATASET}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # 3) Classification HeatMaps (Confusion Matrices in %)
    num_cls_local = len(class_names)
    for name, (y_all, p_all) in all_yprobs.items():
        preds = p_all.argmax(axis=1)
        cm = confusion_matrix(y_all, preds, labels=list(range(num_cls_local)))
        cm_sum = cm.sum(axis=1, keepdims=True) + 1e-8
        cm_percent = cm.astype(float) / cm_sum * 100.0

        plt.figure(figsize=(6,5))
        im = plt.imshow(cm_percent, interpolation='nearest', cmap="Blues", vmin=0, vmax=100)
        plt.title(f"Confusion Matrix (%) - {name} ({DATASET})")
        plt.colorbar(im, fraction=0.046, pad=0.04)
        tick_marks = np.arange(num_cls_local)
        plt.xticks(tick_marks, class_names, rotation=45, ha="right")
        plt.yticks(tick_marks, class_names)

        thresh = cm_percent.max() / 2.
        for i in range(num_cls_local):
            for j in range(num_cls_local):
                plt.text(j, i, f"{cm_percent[i, j]:.1f}",
                         ha="center", va="center",
                         color="white" if cm_percent[i, j] > thresh else "black",
                         fontsize=8)
        plt.ylabel("True label")
        plt.xlabel("Predicted label")
        plt.tight_layout()
        plt.show()

if __name__ == "__main__":
    main()


### On PAMAP2

In [ ]:
#!/usr/bin/env python3
"""
HAR with QTModel vs Baseline 1D CNN models on WISDM / UCI HAR / PAMAP2.

QTModel:
 - Strong 1D CNN backbone (64-d features)
 - Stacked quantum blocks:
     * Linear -> angles for Q/K/V
     * TorchQuantum QFM for Q, K, V
     * Self-attention after each QFM block (element-wise Q·K gating of V)
     * Transformer-style FFN + LayerNorm with residuals
 - Final quantum gate (extra QFM) on top of last features
 - Fused [features + gate] -> MLP classifier

Baselines:
 - ResNet1D (DeepILS)
 - EfficientNetB0_1D
 - MnasNet_1D
 - MobileNet_1D
 - MobileNetV2_1D
 - TSLANet

Outputs:
 - 6-fold CV metrics for each model
 - Training Loss Curves (mean over folds)
 - ROC curves (Validation, AUC)
 - Classification HeatMaps (confusion matrices in %, per model)
"""

import os
import urllib.request
from pathlib import Path
import zipfile
import glob

import psutil
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR

from ptflops import get_model_complexity_info

from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, confusion_matrix
)
from sklearn.preprocessing import label_binarize

import matplotlib.pyplot as plt
from tqdm import trange

import torchquantum as tq

# -----------------------------------------------------------------------------
# GLOBAL CONFIG
# -----------------------------------------------------------------------------

DATASET = "PAMAP2"  # "WISDM", "UCIHAR", or "PAMAP2"

QSize = 4                # number of quantum wires
Q_LAYERS = 4             # depth of quantum feature map
Q_BLOCKS = 4             # number of quantum blocks
EPOCHS = 40              # training epochs per fold
BATCH_SIZE = 256         # batch size
N_SPLITS = 6             # K-fold CV

WINDOW_SIZE = 200        # default; dataset-specific overwrite below
STEP_SIZE = 100

DATA_DIR = Path(os.environ.get("DATA_DIR", "../data"))
DATA_DIR.mkdir(exist_ok=True)

# WISDM
WISDM_URL = (
    "https://raw.githubusercontent.com/soham97/WISD_HAR_files/master/"
    "WISDM_ar_v1.1_raw.txt"
)
WISDM_PATH = DATA_DIR / "WISDM_ar_v1.1_raw.txt"

# UCI HAR
UCIHAR_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "00240/UCI%20HAR%20Dataset.zip"
)
UCIHAR_ZIP_PATH = DATA_DIR / "UCI_HAR_Dataset.zip"
UCIHAR_EXTRACT_DIR = DATA_DIR / "UCI_HAR_Dataset"

# PAMAP2
PAMAP2_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00231/PAMAP2_Dataset.zip"
PAMAP2_ZIP_PATH = DATA_DIR / "PAMAP2_Dataset.zip"
PAMAP2_EXTRACT_DIR = DATA_DIR / "PAMAP2_Dataset"

# Reasonable default windowing for PAMAP2 @100Hz
PAMAP2_WINDOW = 256
PAMAP2_STEP = 128

# -----------------------------------------------------------------------------
# 1) DEVICE SELECTION (fixed cuda:0 if available)
# -----------------------------------------------------------------------------

if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
    print("Using device: cuda:0")
else:
    DEVICE = torch.device("cpu")
    print("CUDA not available, using CPU")

ram = psutil.virtual_memory()
print(f"RAM Usage: {(ram.total-ram.available)/1e9:.1f}/{ram.total/1e9:.1f} GiB ({ram.percent}%)\n")

# -----------------------------------------------------------------------------
# 2) WISDM & UCI HAR LOADING / PREPROCESSING
#    + NEW: PAMAP2 LOADING / PREPROCESSING
# -----------------------------------------------------------------------------

def download_wisdm():
    if WISDM_PATH.exists():
        print(f"[INFO] Found WISDM file at {WISDM_PATH}")
        return
    print(f"[INFO] Downloading WISDM from {WISDM_URL} ...")
    urllib.request.urlretrieve(WISDM_URL, WISDM_PATH)
    print("[INFO] Download complete.")

def load_wisdm_dataframe(path: Path) -> pd.DataFrame:
    """Load and clean WISDM dataset."""
    print("[INFO] Loading WISDM data into DataFrame...")
    columns = ["user", "activity", "timestamp", "x", "y", "z"]
    df = pd.read_csv(
        path,
        header=None,
        names=columns,
        engine="python",
        on_bad_lines="skip",
    )

    df = df.dropna()
    df["z"] = df["z"].astype(str).str.replace(";", "", regex=False)
    df["z"] = df["z"].astype(float)
    df = df[df["timestamp"] != 0]
    df = df.sort_values(by=["user", "timestamp"], ignore_index=True)

    print("[INFO] Cleaned WISDM shape:", df.shape)
    return df

def segment_wisdm(df: pd.DataFrame,
                  window_size: int = WINDOW_SIZE,
                  step_size: int = STEP_SIZE):
    """Segment accelerometer data into windows and compute magnitude."""
    print("[INFO] Segmenting WISDM time series into windows...")
    X_segments = []
    y_labels = []

    users = df["user"].unique()
    for user in users:
        df_user = df[df["user"] == user]
        activities = df_user["activity"].unique()

        for act in activities:
            df_ua = df_user[df_user["activity"] == act]
            signal = df_ua[["x", "y", "z"]].values  # [T, 3]

            for start in range(0, len(signal) - window_size, step_size):
                end = start + window_size
                window = signal[start:end]  # [window_size, 3]
                mag = np.sqrt((window ** 2).sum(axis=1))  # [window_size]
                X_segments.append(mag)
                y_labels.append(act)

    X_segments = np.array(X_segments, dtype=np.float32)
    y_labels = np.array(y_labels)
    print("[INFO] WISDM Segmented X shape:", X_segments.shape)
    print("[INFO] WISDM Label distribution (raw):\n", pd.Series(y_labels).value_counts())
    return X_segments, y_labels

def download_uci_har():
    if UCIHAR_EXTRACT_DIR.exists():
        print(f"[INFO] Found UCI HAR at {UCIHAR_EXTRACT_DIR}")
        return
    if not UCIHAR_ZIP_PATH.exists():
        print(f"[INFO] Downloading UCI HAR from {UCIHAR_URL} ...")
        urllib.request.urlretrieve(UCIHAR_URL, UCIHAR_ZIP_PATH)
        print("[INFO] Download complete.")
    print(f"[INFO] Extracting UCI HAR zip to {UCIHAR_EXTRACT_DIR} ...")
    with zipfile.ZipFile(UCIHAR_ZIP_PATH, "r") as zf:
        zf.extractall(UCIHAR_EXTRACT_DIR)
    print("[INFO] Extraction complete.")

def load_uci_har_magnitude():
    """
    Use body_acc_x/y/z from UCI HAR to build magnitude windows:
      - Each window length = 128
      - Combine train + test
      - Labels: 6 activities (0..5)
    """
    base = UCIHAR_EXTRACT_DIR / "UCI HAR Dataset"

    def load_split(split: str):
        split_dir = base / split
        sig_dir = split_dir / "Inertial Signals"

        # Each: [N_windows, 128]
        x = np.loadtxt(sig_dir / f"body_acc_x_{split}.txt")
        y = np.loadtxt(sig_dir / f"body_acc_y_{split}.txt")
        z = np.loadtxt(sig_dir / f"body_acc_z_{split}.txt")

        mag = np.sqrt(x**2 + y**2 + z**2).astype(np.float32)  # [N, 128]

        # Labels are 1..6 -> 0..5
        y_lab = np.loadtxt(split_dir / f"y_{split}.txt").astype(int) - 1
        return mag, y_lab

    X_train, y_train = load_split("train")
    X_test,  y_test  = load_split("test")

    X_all = np.concatenate([X_train, X_test], axis=0)
    y_all = np.concatenate([y_train, y_test], axis=0)

    print("[INFO] UCI HAR magnitude shape:", X_all.shape)
    print("[INFO] UCI HAR label distribution:",
          dict(zip(*np.unique(y_all, return_counts=True))))
    return X_all, y_all

# -------------------------
# PAMAP2 helpers
# -------------------------

def download_pamap2():
    if PAMAP2_EXTRACT_DIR.exists():
        print(f"[INFO] Found PAMAP2 at {PAMAP2_EXTRACT_DIR}")
        return
    if not PAMAP2_ZIP_PATH.exists():
        print(f"[INFO] Downloading PAMAP2 from {PAMAP2_URL} ...")
        urllib.request.urlretrieve(PAMAP2_URL, PAMAP2_ZIP_PATH)
        print("[INFO] Download complete.")
    print(f"[INFO] Extracting PAMAP2 zip to {PAMAP2_EXTRACT_DIR} ...")
    with zipfile.ZipFile(PAMAP2_ZIP_PATH, "r") as zf:
        zf.extractall(DATA_DIR)
    # After extraction, it creates folder 'PAMAP2_Dataset'
    if not PAMAP2_EXTRACT_DIR.exists():
        # If archive extracted differently, try to find it
        cand = next((p for p in DATA_DIR.iterdir() if p.is_dir() and "PAMAP2" in p.name), None)
        if cand:
            cand.rename(PAMAP2_EXTRACT_DIR)
    print("[INFO] Extraction complete.")

# Activity ID mapping from UCI (excluding 0 'other')
PAMAP2_ACTIVITY_MAP = {
    1: "lying",
    2: "sitting",
    3: "standing",
    4: "walking",
    5: "running",
    6: "cycling",
    7: "nordic_walking",
    9: "watching_tv",
    10: "computer_work",
    11: "car_driving",
    12: "ascending_stairs",
    13: "descending_stairs",
    16: "vacuum_cleaning",
    17: "ironing",
    18: "folding_laundry",
    19: "house_cleaning",
    20: "playing_soccer",
    24: "rope_jumping",
    # 0: "other"  # excluded
}

def _pamap2_block_indices_0based(block_start_1based: int):
    """
    Given the 1-based start column of a device block (hand=4, chest=21, ankle=38),
    return zero-based indices for the ±16g accelerometer x,y,z within that block.
    Per UCI spec:
      - 1: timestamp
      - 2: activityID
      - 3: heart rate
      - 4-20: IMU hand (16 cols)
      - 21-37: IMU chest (16 cols)
      - 38-54: IMU ankle (16 cols)
    Within each IMU block (length=16):
      - 1: temperature
      - 2-4: 3D-acc (±16g)
      - 5-7: 3D-acc (±6g)
      - 8-10: 3D-gyro
      - 11-13: 3D-magnetometer
      - 14-17: orientation (invalid)
    We will use the ±16g accelerometer triplet (positions 2-4 in-block).
    """
    start0 = block_start_1based - 1
    acc16_x = start0 + 1  # 2nd in-block -> +1 from start0
    acc16_y = start0 + 2
    acc16_z = start0 + 3
    return [acc16_x, acc16_y, acc16_z]

# Choose CHEST IMU (commonly used)
PAMAP2_CHEST_ACC16_IDX = _pamap2_block_indices_0based(21)  # [21,22,23] zero-based

def _read_pamap2_file(file_path: Path) -> np.ndarray:
    """
    Reads a single PAMAP2 .dat file (space-separated with NaNs), returns numpy array.
    Columns: 54 total.
    """
    try:
        # faster than pandas for big plain numeric arrays with NaNs
        arr = np.genfromtxt(str(file_path), delimiter=' ', dtype=float)
        # Ensure 2D (some readers return 1D if file has 1 row by mistake)
        if arr.ndim == 1:
            arr = arr[None, :]
        # Filter columns length issues
        if arr.shape[1] < 54:
            # pad missing trailing cols with NaN
            pad = 54 - arr.shape[1]
            arr = np.hstack([arr, np.full((arr.shape[0], pad), np.nan, dtype=float)])
        return arr
    except Exception as e:
        print(f"[WARN] Failed reading {file_path}: {e}")
        return np.zeros((0, 54), dtype=float)

def load_pamap2_magnitude(window_size: int = PAMAP2_WINDOW,
                          step_size: int = PAMAP2_STEP,
                          use_subject_splits: bool = False):
    """
    Build magnitude windows from PAMAP2 chest IMU ±16g accelerometer.
    - Reads all *.dat files from Protocol/ and Optional/ (if present)
    - Drops rows with missing activity ID or missing sensor triplet
    - Excludes activity ID 0 ('other'), keeps the rest
    - Segments per (subject, activity) into sliding windows
    Returns:
      X_segments: [N, window_size] float32
      y_int:      [N] int64 (0..C-1)
      class_names: list[str] in the code order
    """
    base = PAMAP2_EXTRACT_DIR
    protocol_dir = base / "Protocol"
    optional_dir = base / "Optional"

    files = []
    if protocol_dir.exists():
        files.extend(sorted(protocol_dir.glob("subject*.dat")))
    if optional_dir.exists():
        files.extend(sorted(optional_dir.glob("subject*.dat")))

    if not files:
        raise FileNotFoundError(
            f"No PAMAP2 .dat files found under {base}. "
            "Please check extraction."
        )

    print(f"[INFO] PAMAP2: found {len(files)} files")

    # Concatenate all rows with a 'subject' id derived from filename
    all_rows = []
    all_subjects = []
    for fp in files:
        arr = _read_pamap2_file(fp)
        if arr.shape[0] == 0:
            continue
        subj = fp.stem  # e.g., 'subject101'
        subj_id = ''.join([c for c in subj if c.isdigit()]) or "0"
        # Keep only necessary columns to save RAM: activity (1), chest acc16 x/y/z
        # Zero-based indices:
        #  1 -> activityID
        #  PAMAP2_CHEST_ACC16_IDX -> x,y,z
        cols = [1] + PAMAP2_CHEST_ACC16_IDX
        sub = arr[:, cols]
        # Drop rows where activity is NaN
        sub = sub[~np.isnan(sub[:, 0])]
        if sub.size == 0:
            continue
        # Drop rows with NaNs in any of the three acc channels
        mask_valid = ~np.isnan(sub[:, 1:]).any(axis=1)
        sub = sub[mask_valid]
        if sub.size == 0:
            continue
        all_rows.append(sub)
        all_subjects.append(np.full((sub.shape[0],), int(subj_id), dtype=np.int32))

    if not all_rows:
        raise RuntimeError("PAMAP2 parsing produced no valid rows.")

    data = np.vstack(all_rows)                   # [T, 4] -> (activity, ax, ay, az)
    subj_arr = np.concatenate(all_subjects)      # [T]

    # Exclude 'other' (activityID == 0)
    act_id = data[:, 0].astype(int)
    keep = act_id != 0
    data = data[keep]
    subj_arr = subj_arr[keep]
    act_id = act_id[keep]

    # Map ids -> names -> compact integer labels (0..C-1)
    names = np.array([PAMAP2_ACTIVITY_MAP.get(a, None) for a in act_id])
    valid_mask = names != None
    data = data[valid_mask]
    subj_arr = subj_arr[valid_mask]
    act_id = act_id[valid_mask]
    names = names[valid_mask]

    # Build consistent class ordering by sorted activity id present
    present_ids = np.sort(np.unique(act_id))
    id_to_idx = {aid: i for i, aid in enumerate(present_ids)}
    class_names = [PAMAP2_ACTIVITY_MAP[aid] for aid in present_ids]
    y_int = np.array([id_to_idx[a] for a in act_id], dtype=np.int64)

    # Compute magnitude of chest acc16
    acc = data[:, 1:4].astype(np.float32)
    mag = np.sqrt((acc ** 2).sum(axis=1))  # [T]

    # Segment per (subject, activity) to avoid mixing boundaries
    X_segments = []
    y_segments = []

    # Create indices grouped by subject and label to keep windows homogeneous
    df_idx = pd.DataFrame({
        "subj": subj_arr,
        "y": y_int
    })
    # For performance, iterate contiguous blocks by scanning df_idx
    # but simpler: loop per (subj, y)
    for sj in np.unique(subj_arr):
        mask_s = subj_arr == sj
        y_s = y_int[mask_s]
        mag_s = mag[mask_s]
        # iterate over runs of constant activity within this subject
        start = 0
        while start < len(y_s):
            curr_y = y_s[start]
            end = start
            while end < len(y_s) and y_s[end] == curr_y:
                end += 1
            segment = mag_s[start:end]
            # slide
            if len(segment) >= window_size:
                for st in range(0, len(segment) - window_size, step_size):
                    ed = st + window_size
                    X_segments.append(segment[st:ed])
                    y_segments.append(curr_y)
            start = end

    X_segments = np.array(X_segments, dtype=np.float32)
    y_segments = np.array(y_segments, dtype=np.int64)

    print("[INFO] PAMAP2 magnitude windows:", X_segments.shape)
    print("[INFO] PAMAP2 labels distribution:",
          dict(zip(*np.unique(y_segments, return_counts=True))))
    return X_segments, y_segments, class_names

def balance_by_min_class(X: np.ndarray, y_int: np.ndarray, seed: int = 42):
    """
    Downsample all classes to the smallest class size.
    """
    rng = np.random.default_rng(seed)
    unique_classes = np.unique(y_int)
    indices_per_class = {c: np.where(y_int == c)[0] for c in unique_classes}
    class_sizes = {c: len(idx) for c, idx in indices_per_class.items()}
    min_count = min(class_sizes.values())

    print("[INFO] Class sizes before balancing:", class_sizes)
    print("[INFO] Using min_count =", min_count, "samples per class")

    balanced_indices_list = []
    for c in unique_classes:
        idxs = indices_per_class[c]
        if len(idxs) > min_count:
            chosen = rng.choice(idxs, size=min_count, replace=False)
        else:
            chosen = idxs
        balanced_indices_list.append(chosen)

    balanced_indices = np.concatenate(balanced_indices_list)
    rng.shuffle(balanced_indices)

    balanced_counts = np.bincount(y_int[balanced_indices])
    print("[INFO] Class sizes after balancing:",
          dict(zip(unique_classes, balanced_counts)))
    return X[balanced_indices], y_int[balanced_indices]

# -----------------------------------------------------------------------------
# 3) DATASET CLASS
# -----------------------------------------------------------------------------

class WISDMDataset(Dataset):
    """
    Generic window-level dataset for time-series magnitude signals.
    Each sample: x -> [1, L], y -> integer label.
    """

    def __init__(self, X_windows: np.ndarray, y_int: np.ndarray):
        X_tensor = torch.tensor(X_windows, dtype=torch.float32)
        y_tensor = torch.tensor(y_int, dtype=torch.long)

        # Per-window normalization
        means = X_tensor.mean(dim=1, keepdim=True)
        stds = X_tensor.std(dim=1, keepdim=True) + 1e-8
        X_tensor = (X_tensor - means) / stds

        self.X = X_tensor
        self.y = y_tensor

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx].unsqueeze(0)  # [1, L]
        y = self.y[idx]
        return x, y

# -----------------------------------------------------------------------------
# 4) STRONGER CNN BACKBONE (for QTModel)
# -----------------------------------------------------------------------------

class EnhancedCNN1D(nn.Module):
    """
    Strong 1D CNN backbone for QTModel.
    Input:  [B, 1, L]
    Output: [B, 64]
    """
    def __init__(self, in_ch: int = 1):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv1d(in_ch, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2)   # L -> L/2
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2)   # L/2 -> L/4
        )
        self.block3 = nn.Sequential(
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)  # [B,64,1]
        )
        self.dropout = nn.Dropout(p=0.2)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.dropout(x)
        return x.flatten(1)  # [B, 64]

# -----------------------------------------------------------------------------
# 5) TORCHQUANTUM FEATURE MAP
# -----------------------------------------------------------------------------

class QuantumFeatureMapTQ(tq.QuantumModule):
    """
    TorchQuantum-based feature map:
     - Angle embedding via RY
     - Ring entangling CNOT chain
     - Measure all Z
    """
    def __init__(self, n_wires=4, n_layers=4):
        super().__init__()
        self.n_wires  = n_wires
        self.n_layers = n_layers
        self.measure  = tq.MeasureAll(tq.PauliZ)

    def forward(self, qdev, angles):
        # angles: (batch, n_wires)
        for w in range(self.n_wires):
            qdev.ry(wires=w, params=angles[:, w], static=self.static_mode)
        for _ in range(self.n_layers):
            for i in range(self.n_wires - 1):
                qdev.cnot(wires=[i, i+1])
            qdev.cnot(wires=[self.n_wires-1, 0])
        return self.measure(qdev)  # (batch, n_wires)

# -----------------------------------------------------------------------------
# 6) QUANTUM BLOCK WITH SELF-ATTENTION + FFN
# -----------------------------------------------------------------------------

class QBlockQT(tq.QuantumModule):
    """
    Single quantum block for QTModel:

      Input:  x ∈ R^{B×D}
      Steps:
        - Linear -> angles for Q, K, V (each in R^{B×n_wires})
        - QFM for Q, K, V (TorchQuantum)
        - Self-attention:
            q = Wq(q_out), k = Wk(k_out), v = Wv(v_out) ∈ R^{B×D}
            scores = softmax( (q * k) / sqrt(D) )
            attn_out = scores * v
        - Residual: z = x + attn_out
        - LayerNorm + Transformer-style FFN + residual + LayerNorm
    """
    def __init__(self, dim=64, n_wires=4, n_layers=4, ff_mult=2, dropout=0.1):
        super().__init__()
        self.dim     = dim
        self.n_wires = n_wires

        # QFM for Q, K, V
        self.q_fm = QuantumFeatureMapTQ(n_wires, n_layers)
        self.k_fm = QuantumFeatureMapTQ(n_wires, n_layers)
        self.v_fm = QuantumFeatureMapTQ(n_wires, n_layers)

        # Map input features -> Q/K/V angles
        self.reduce = nn.Linear(dim, 3 * n_wires)

        # Project Q/K/V outputs up to feature dim
        self.q_proj = nn.Linear(n_wires, dim)
        self.k_proj = nn.Linear(n_wires, dim)
        self.v_proj = nn.Linear(n_wires, dim)

        # Transformer-style FFN
        self.ff = nn.Sequential(
            nn.Linear(dim, ff_mult * dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_mult * dim, dim),
        )

        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        # x: [B, D]
        B, D = x.shape

        # Classical to quantum angles
        r = self.reduce(x)                      # [B, 3 * n_wires]
        q_ang, k_ang, v_ang = r.chunk(3, dim=1) # each [B, n_wires]

        # Quantum device
        qdev = tq.QuantumDevice(
            n_wires=self.n_wires,
            bsz=B,
            device=x.device
        )

        # QFM for Q, K, V
        q_out = self.q_fm(qdev, q_ang)  # [B, n_wires]
        k_out = self.k_fm(qdev, k_ang)
        v_out = self.v_fm(qdev, v_ang)

        # Project to feature dimension
        q = self.q_proj(q_out)  # [B, D]
        k = self.k_proj(k_out)  # [B, D]
        v = self.v_proj(v_out)  # [B, D]

        # Self-attention (element-wise gating)
        scale = float(D) ** 0.5
        scores = torch.softmax((q * k) / scale, dim=-1)  # [B, D]
        attn_out = scores * v                            # [B, D]

        # Residual + FFN + norms
        z = x + attn_out          # first residual
        z_norm = self.norm1(z)
        ff_out = self.ff(z_norm)
        y = self.norm2(z_norm + ff_out)  # second residual
        return y

# -----------------------------------------------------------------------------
# 7) QTModel: CNN -> stacked QBlocks -> quantum gate -> classifier
# -----------------------------------------------------------------------------

class QTModel(nn.Module):
    """
    QTModel:
      - EnhancedCNN1D -> 64-d feature
      - Multiple QBlockQT (each with QFM+SelfAttention+FFN)
      - Final quantum gate:
          angles = W_gate(e)
          q_gate = QFM(angles)
          fused = [e, q_gate]
          MLP classifier
    """
    def __init__(self, num_cls, in_ch=1,
                 n_blocks=Q_BLOCKS, n_wires=QSize, n_layers=Q_LAYERS,
                 dim=64, ff_mult=2, dropout=0.1):
        super().__init__()

        self.dim      = dim
        self.n_wires  = n_wires
        self.cnn      = EnhancedCNN1D(in_ch)

        # Stacked quantum blocks
        self.blocks = nn.ModuleList([
            QBlockQT(dim=dim, n_wires=n_wires, n_layers=n_layers,
                     ff_mult=ff_mult, dropout=dropout)
            for _ in range(n_blocks)
        ])

        # Final quantum gate
        self.gate_linear = nn.Linear(dim, n_wires)
        self.gate_qfm    = QuantumFeatureMapTQ(n_wires, n_layers)

        # Classifier on fused [features + quantum gate]
        self.cls_head = nn.Sequential(
            nn.LayerNorm(dim + n_wires),
            nn.Linear(dim + n_wires, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, num_cls),
        )

    def forward(self, x):
        # CNN backbone
        e = self.cnn(x)  # [B, 64]

        # Quantum blocks
        for block in self.blocks:
            e = block(e)  # [B, 64]

        # Final quantum gate
        angles = self.gate_linear(e)  # [B, n_wires]
        qdev = tq.QuantumDevice(
            n_wires=self.n_wires,
            bsz=angles.size(0),
            device=angles.device
        )
        q_gate = self.gate_qfm(qdev, angles)  # [B, n_wires]

        fused = torch.cat([e, q_gate], dim=1)  # [B, 64 + n_wires]
        logits = self.cls_head(fused)          # raw logits
        return logits

# -----------------------------------------------------------------------------
# 8) BASELINE MODELS (unchanged)
# -----------------------------------------------------------------------------

def conv_dw(in_planes, out_planes, kernel_size, stride=1, dilation=1):
    return nn.Sequential(
        nn.Conv1d(in_planes, in_planes, kernel_size=kernel_size,
                  stride=stride, padding=(kernel_size//2)*dilation, dilation=dilation,
                  groups=in_planes, bias=False),
        nn.Conv1d(in_planes, out_planes, kernel_size=1, bias=False)
    )

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool1d(1)
        self.max = nn.AdaptiveMaxPool1d(1)
        red = max(1, in_planes // ratio)
        self.fc = nn.Sequential(
            nn.Conv1d(in_planes, red, 1, bias=False), nn.ReLU(),
            nn.Conv1d(red, in_planes, 1, bias=False)
        )
        self.sig = nn.Sigmoid()
    def forward(self, x):
        return self.sig(self.fc(self.avg(x)) + self.fc(self.max(x)))

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv1 = nn.Conv1d(2, 1, k, padding=k//2, bias=False)
        self.sig = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sig(self.conv1(torch.cat([avg_out, max_out], dim=1)))

class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, out_planes, k, stride=1, dilation=1, downsample=None):
        super().__init__()
        self.conv1 = conv_dw(in_planes, out_planes, k, stride, dilation)
        self.bn1 = nn.BatchNorm1d(out_planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv_dw(out_planes, out_planes, k)
        self.bn2 = nn.BatchNorm1d(out_planes)
        self.ca = ChannelAttention(out_planes)
        self.sa = SpatialAttention()
        self.downsample = downsample
    def forward(self, x):
        res = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.ca(out)*out
        out = self.sa(out)*out
        if self.downsample is not None:
            res = self.downsample(x)
        return self.relu(out + res)

class DeepILS(nn.Module):
    def __init__(self, num_inputs=1, num_classes=6, block=BasicBlock1D, group_sizes=(2,2,2,2), base=64, k=3):
        super().__init__()
        self.inp = nn.Sequential(
            nn.Conv1d(num_inputs, base, 5, 2, 3, bias=False),
            nn.BatchNorm1d(base),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(3,2,1)
        )
        self.inplanes = base
        self.groups = nn.ModuleList()
        planes = [base*(2**i) for i in range(len(group_sizes))]
        strides = [1] + [2]*(len(group_sizes)-1)
        for i, blocks in enumerate(group_sizes):
            pl = planes[i]
            st = strides[i]
            down = None
            if st != 1 or self.inplanes != pl*block.expansion:
                down = nn.Sequential(
                    nn.Conv1d(self.inplanes, pl*block.expansion, 1, st, bias=False),
                    nn.BatchNorm1d(pl*block.expansion)
                )
            layers = [block(self.inplanes, pl, k, stride=st, downsample=down)]
            self.inplanes = pl*block.expansion
            for _ in range(1, blocks):
                layers.append(block(self.inplanes, pl, k))
            self.groups.append(nn.Sequential(*layers))
        self.groups = nn.Sequential(*self.groups)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(planes[-1]*block.expansion, num_classes)
        )
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
    def forward(self, x):
        x = self.inp(x)
        x = self.groups(x)
        return self.head(x)

class ResNet1D(DeepILS):
    pass

class Swish(nn.Module):
    def forward(self, x): return x*torch.sigmoid(x)

def Conv_3x3(inp, oup, s):
    return nn.Sequential(
        nn.Conv1d(inp,oup,3,s,1,bias=False),
        nn.BatchNorm1d(oup),
        Swish()
    )

def Conv_1x1(inp, oup):
    return nn.Sequential(
        nn.Conv1d(inp,oup,1,1,0,bias=False),
        nn.BatchNorm1d(oup),
        Swish()
    )

class InvertedResidual_Eff(nn.Module):
    def __init__(self, inp, oup, s, t, k):
        super().__init__()
        self.use=(s==1 and inp==oup)
        hid=inp*t
        self.conv=nn.Sequential(
            nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), Swish(),
            nn.Conv1d(hid,hid,k,s,k//2,groups=hid,bias=False), nn.BatchNorm1d(hid), Swish(),
            nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
        )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

class EfficientNetB0_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        setting=[[1,16,1,1,3],[6,24,2,2,3],[6,40,2,2,5],
                 [6,80,3,2,3],[6,112,3,1,5],[6,192,4,2,5],[6,320,1,1,3]]
        in_ch=int(32*wm)
        last=1280 if wm<=1 else int(1280*wm)
        feats=[Conv_3x3(1,in_ch,2)]
        ch=in_ch
        for t,c,n,s,k in setting:
            oc=int(c*wm)
            for i in range(n):
                feats.append(InvertedResidual_Eff(ch, oc, s if i==0 else 1, t, k))
                ch=oc
        feats += [Conv_1x1(ch,last), nn.AdaptiveAvgPool1d(1)]
        self.features=nn.Sequential(*feats)
        self.fc=nn.Linear(last,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        return self.fc(self.features(x).squeeze(-1))

class InvertedResidual_Mnas(nn.Module):
    def __init__(self, inp, oup, s, t, k):
        super().__init__()
        self.use=(s==1 and inp==oup)
        hid=inp*t
        self.conv=nn.Sequential(
            nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
            nn.Conv1d(hid,hid,k,s,k//2,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
            nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
        )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

def SepConv_3x3(inp,oup):
    return nn.Sequential(
        nn.Conv1d(inp,inp,3,1,1,groups=inp,bias=False), nn.BatchNorm1d(inp), nn.ReLU6(inplace=True),
        nn.Conv1d(inp,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
    )

def Conv3_3(inp,oup,s):
    return nn.Sequential(
        nn.Conv1d(inp,oup,3,s,1,bias=False), nn.BatchNorm1d(oup), nn.ReLU6(inplace=True)
    )

def Conv1_1(inp,oup):
    return nn.Sequential(
        nn.Conv1d(inp,oup,1,1,0,bias=False), nn.BatchNorm1d(oup), nn.ReLU6(inplace=True)
    )

class MnasNet_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        setting=[[3,24,3,2,3],[3,40,3,2,5],[6,80,3,2,5],
                 [6,96,2,1,3],[6,192,4,2,5],[6,320,1,1,3]]
        in_ch=int(32*wm)
        last=1280 if wm<=1 else int(1280*wm)
        feats=[Conv3_3(1,in_ch,2), SepConv_3x3(in_ch,16)]
        ch=16
        for t,c,n,s,k in setting:
            oc=int(c*wm)
            for i in range(n):
                feats.append(InvertedResidual_Mnas(ch, oc, s if i==0 else 1, t, k))
                ch=oc
        feats += [Conv1_1(ch,last), nn.AdaptiveAvgPool1d(1)]
        self.features=nn.Sequential(*feats)
        self.fc=nn.Linear(last,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        return self.fc(self.features(x).squeeze(-1))

class DSConv_Mobile(nn.Module):
    def __init__(self, f3, f1, s=1, p=1):
        super().__init__()
        self.seq=nn.Sequential(
            nn.Conv1d(f3,f3,3,s,p,groups=f3,bias=False), nn.BatchNorm1d(f3), nn.ReLU(inplace=True),
            nn.Conv1d(f3,f1,1,1,0,bias=False), nn.BatchNorm1d(f1), nn.ReLU(inplace=True)
        )
    def forward(self,x): return self.seq(x)

class MobileNet_1D(nn.Module):
    def __init__(self, channels=[32,64,128,256,512,1024], wm=1.0, n_classes=6):
        super().__init__()
        ch=[int(c*wm) for c in channels]
        self.conv=nn.Sequential(
            nn.Conv1d(1,ch[0],3,2,1,bias=False), nn.BatchNorm1d(ch[0]), nn.ReLU(inplace=True)
        )
        self.features=nn.Sequential(
            DSConv_Mobile(ch[0],ch[1],1,1),
            DSConv_Mobile(ch[1],ch[2],2,1),
            DSConv_Mobile(ch[2],ch[2],1,1),
            DSConv_Mobile(ch[2],ch[3],2,1),
            DSConv_Mobile(ch[3],ch[3],1,1),
            DSConv_Mobile(ch[3],ch[4],2,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[5],2,1),
            DSConv_Mobile(ch[5],ch[5],1,1),
        )
        self.avg=nn.AdaptiveAvgPool1d(1)
        self.fc=nn.Linear(ch[5],n_classes)
    def forward(self,x):
        x=self.conv(x)
        x=self.features(x)
        x=self.avg(x).squeeze(-1)
        return self.fc(x)

class InvertedResidual_V2(nn.Module):
    def __init__(self, inp, oup, s, t):
        super().__init__()
        hid=int(inp*t)
        self.use=(s==1 and inp==oup)
        if t==1:
            self.conv=nn.Sequential(
                nn.Conv1d(hid,hid,3,s,1,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
            )
        else:
            self.conv=nn.Sequential(
                nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,hid,3,s,1,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
            )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

def make_divisible(x, d=8):
    return int((x + d - 1) // d * d)

class MobileNetV2_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        input_channel=32
        last=1280
        setting=[[1,16,1,1],[6,24,2,2],[6,32,3,2],[6,64,4,2],
                 [6,96,3,1],[6,160,3,2],[6,320,1,1]]
        self.last_ch=make_divisible(last*wm) if wm>1 else last
        feats=[nn.Conv1d(1,input_channel,3,2,1,bias=False), nn.BatchNorm1d(input_channel), nn.ReLU6(inplace=True)]
        ch=input_channel
        for t,c,n,s in setting:
            oc=make_divisible(c*wm) if t>1 else c
            for i in range(n):
                feats.append(InvertedResidual_V2(ch, oc, s if i==0 else 1, t))
                ch=oc
        feats += [nn.Conv1d(ch,self.last_ch,1,1,0,bias=False), nn.BatchNorm1d(self.last_ch), nn.ReLU6(inplace=True)]
        self.features=nn.Sequential(*feats)
        self.pool=nn.AdaptiveAvgPool1d(1)
        self.fc=nn.Linear(self.last_ch,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        x=self.features(x)
        x=self.pool(x).squeeze(-1)
        return self.fc(x)

# --- TSLANet (compact) ---
class TSLANet(nn.Module):
    def __init__(self, C_in, T_in, n_classes=6, emb=128, depth=3, patch=8, drop=0.10):
        super().__init__()
        stride=max(1, patch//2)
        self.proj=nn.Conv1d(C_in,emb,patch,stride)
        N=int((T_in-patch)/stride+1)
        self.pos=nn.Parameter(torch.zeros(1,N,emb))
        self.blocks=nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(emb),
                nn.Identity(),  # spectral placeholder
                nn.LayerNorm(emb),
                nn.Sequential(
                    nn.Linear(emb,int(emb*3)), nn.GELU(),
                    nn.Linear(int(emb*3),emb)
                )
            ) for _ in range(depth)
        ])
        self.drop=drop
        self.head=nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(emb,emb//2),
            nn.GELU(),
            nn.Linear(emb//2,n_classes)
        )
    def forward(self,x):
        x=self.proj(x).flatten(2).transpose(1,2)
        x=x+self.pos
        x=F.dropout(x,p=self.drop,training=self.training)
        for b in self.blocks:
            x=x + b[3](b[1](b[0](x)))
        return self.head(x.mean(1))

# -----------------------------------------------------------------------------
# 9) TRAIN / EVAL HELPERS
# -----------------------------------------------------------------------------

def process_target(y: torch.Tensor) -> torch.Tensor:
    if y.dim() > 1:
        if y.size(1) > 1:
            y = y.argmax(dim=1)
        else:
            y = y.squeeze(1)
    return y.long()

def train_one(model, loader, opt):
    model.train()
    losses = []
    for x, y in loader:
        x = x.to(DEVICE)
        y = process_target(y.to(DEVICE))

        out = model(x)               # logits
        loss = F.cross_entropy(out, y)

        opt.zero_grad()
        loss.backward()
        opt.step()

        losses.append(loss.item())
    return float(np.mean(losses)) if losses else 0.0

def eval_fold(model, loader, num_cls):
    """Evaluate on one fold: return y_true and probabilities."""
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = process_target(y.to(DEVICE))
            out  = model(x)  # logits
            prob = F.softmax(out, dim=1).cpu().numpy()
            ys.append(y.cpu().numpy())
            ps.append(prob)
    ys = np.concatenate(ys)
    ps = np.concatenate(ps)
    return ys, ps

def run_cv_for_model(model_name, model_ctor, num_cls, dataset):
    N = len(dataset)
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=0)

    metrics = {"acc": [], "f1": [], "prec": [], "rec": [], "auc": []}
    loss_hist = []
    all_y = []
    all_probs = []

    print(f"\n====== Running {N_SPLITS}-fold CV for {model_name} ======")

    for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(N)), start=1):
        print(f"\n--- {model_name} Fold {fold}/{N_SPLITS} ---")

        train_loader = DataLoader(
            Subset(dataset, train_idx),
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=4,
            pin_memory=True
        )
        val_loader = DataLoader(
            Subset(dataset, val_idx),
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=4,
            pin_memory=True
        )

        model = model_ctor().to(DEVICE)
        opt   = optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-4)
        sch   = CosineAnnealingLR(opt, T_max=EPOCHS)

        fold_losses = []
        for _ in trange(EPOCHS, desc=f"{model_name}-F{fold}", leave=False):
            loss = train_one(model, train_loader, opt)
            sch.step()
            fold_losses.append(loss)
        loss_hist.append(fold_losses)

        y_fold, p_fold = eval_fold(model, val_loader, num_cls)
        preds = p_fold.argmax(axis=1)

        acc  = accuracy_score(y_fold, preds)
        f1   = f1_score(y_fold, preds, average="weighted")
        prec = precision_score(y_fold, preds, average="weighted")
        rec  = recall_score(y_fold, preds, average="weighted")

        if num_cls == 2:
            auc_val = roc_auc_score(y_fold, p_fold[:,1])
        else:
            y_bin = label_binarize(y_fold, classes=list(range(num_cls)))
            auc_val = roc_auc_score(y_bin, p_fold, average="micro", multi_class="ovo")

        print(f" Fold {fold}: Acc={acc:.4f} F1={f1:.4f} Prec={prec:.4f} Rec={rec:.4f} AUC={auc_val:.4f}")
        for k,v in zip(("acc","f1","prec","rec","auc"), (acc,f1,prec,rec,auc_val)):
            metrics[k].append(v)

        all_y.append(y_fold)
        all_probs.append(p_fold)

        # free model this fold
        del model
        torch.cuda.empty_cache()

    all_y = np.concatenate(all_y)
    all_probs = np.concatenate(all_probs, axis=0)

    # Global ROC for the model
    if num_cls == 2:
        fpr, tpr, _ = roc_curve(all_y, all_probs[:,1], pos_label=1)
        auc_global = roc_auc_score(all_y, all_probs[:,1])
    else:
        y_bin = label_binarize(all_y, classes=list(range(num_cls)))
        fpr, tpr, _ = roc_curve(y_bin.ravel(), all_probs.ravel())
        auc_global = roc_auc_score(y_bin, all_probs, average="micro", multi_class="ovo")

    return metrics, loss_hist, all_y, all_probs, (fpr, tpr, auc_global)

# -----------------------------------------------------------------------------
# 10) MAIN: DATA CHOICE + MODELS + PLOTS
# -----------------------------------------------------------------------------

def main():
    global WINDOW_SIZE, STEP_SIZE

    # -----------------------
    # Dataset choice
    # -----------------------
    if DATASET == "WISDM":
        download_wisdm()
        df = load_wisdm_dataframe(WISDM_PATH)
        X_segments, y_labels = segment_wisdm(df)

        labels_cat = pd.Series(y_labels).astype("category")
        class_names = list(labels_cat.cat.categories)
        y_int = labels_cat.cat.codes.to_numpy().astype(np.int64)

        WINDOW_SIZE = X_segments.shape[1]

    elif DATASET == "UCIHAR":
        download_uci_har()
        X_segments, y_int = load_uci_har_magnitude()

        class_names = [
            "WALKING",
            "WALK_UP",
            "WALK_DOWN",
            "SITTING",
            "STANDING",
            "LAYING",
        ]
        WINDOW_SIZE = X_segments.shape[1]

    elif DATASET == "PAMAP2":
        download_pamap2()
        X_segments, y_int, class_names = load_pamap2_magnitude(
            window_size=PAMAP2_WINDOW, step_size=PAMAP2_STEP
        )
        WINDOW_SIZE = X_segments.shape[1]
        STEP_SIZE = PAMAP2_STEP

    else:
        raise ValueError(f"Unknown DATASET: {DATASET}")

    print("[INFO] Using dataset:", DATASET)
    print("[INFO] Classes:", class_names)

    # Balance dataset
    X_bal, y_bal = balance_by_min_class(X_segments, y_int, seed=42)
    dataset = WISDMDataset(X_bal, y_bal)
    N = len(dataset)
    print(f"[INFO] Balanced dataset size: {N}")

    num_cls = len(class_names)
    seq_len = X_bal.shape[1]
    dims = (1, seq_len)

    # ------------------------
    # Model factories
    # ------------------------
    model_factories = {
        "QTModel":           lambda: QTModel(num_cls=num_cls, in_ch=1),
        "ResNet1D":          lambda: ResNet1D(num_inputs=1, num_classes=num_cls),
        "EfficientNetB0_1D": lambda: EfficientNetB0_1D(n_classes=num_cls),
        "MnasNet_1D":        lambda: MnasNet_1D(n_classes=num_cls),
        "MobileNet_1D":      lambda: MobileNet_1D(n_classes=num_cls),
        "MobileNetV2_1D":    lambda: MobileNetV2_1D(n_classes=num_cls),
        "TSLANet":           lambda: TSLANet(C_in=1, T_in=seq_len, n_classes=num_cls),
    }

    # ------------------------
    # Complexity (params/FLOPs) for all models
    # ------------------------
    complexity = {}
    for name, ctor in model_factories.items():
        model = ctor().to(DEVICE)
        try:
            macs, params = get_model_complexity_info(
                model, dims,
                as_strings=True,
                print_per_layer_stat=False
            )
        except Exception as e:
            print(f"[WARN] ptflops failed on {name}: {e}")
            macs, params = "N/A", "N/A"
        complexity[name] = (params, macs)
        del model
        torch.cuda.empty_cache()
        print(f"[INFO] {name}: params={params}, FLOPs={macs}")

    # ------------------------
    # Run CV for each model
    # ------------------------
    all_metrics  = {}
    all_losses   = {}
    all_yprobs   = {}
    all_rocs     = {}

    for name, ctor in model_factories.items():
        metrics, loss_hist, y_all, p_all, roc_info = run_cv_for_model(
            name, ctor, num_cls, dataset
        )
        all_metrics[name] = metrics
        all_losses[name]  = loss_hist
        all_yprobs[name]  = (y_all, p_all)
        all_rocs[name]    = roc_info

    # Summary table
    gpu_mem = None
    if DEVICE.type == "cuda":
        gpu_mem = (torch.cuda.memory_allocated()/1e6,
                   torch.cuda.memory_reserved()/1e6)

    summary_rows = []
    for name in model_factories.keys():
        m = all_metrics[name]
        params, macs = complexity.get(name, ("N/A", "N/A"))

        row = {
            "model":        name,
            "dataset":      DATASET,
            "acc_mean±sd":  f"{np.mean(m['acc']):.4f}±{np.std(m['acc']):.4f}",
            "f1_mean±sd":   f"{np.mean(m['f1']):.4f}±{np.std(m['f1']):.4f}",
            "prec_mean±sd": f"{np.mean(m['prec']):.4f}±{np.std(m['prec']):.4f}",
            "rec_mean±sd":  f"{np.mean(m['rec']):.4f}±{np.std(m['rec']):.4f}",
            "auc_mean±sd":  f"{np.mean(m['auc']):.4f}±{np.std(m['auc']):.4f}",
            "GPU_mem_MB":   gpu_mem,
            "params":       params,
            "FLOPs":        macs,
        }
        summary_rows.append(row)

    df_sum = pd.DataFrame(summary_rows)
    print("\n[SUMMARY COMPARISON]")
    print(df_sum.to_markdown(index=False))

    # -------------------------------------------------
    # PLOTS
    # -------------------------------------------------

    # 1) Training Loss Curves (mean over folds) per model
    plt.figure()
    for name, loss_hist in all_losses.items():
        loss_arr = np.array(loss_hist)  # [n_folds, epochs]
        mean_loss = loss_arr.mean(axis=0)
        plt.plot(range(1, EPOCHS+1), mean_loss, label=name)
    plt.xlabel("Epoch")
    plt.ylabel("Training Loss")
    plt.title(f"Training Loss Curves (mean over folds) - {DATASET}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # 2) ROC curves per model (Validation)
    plt.figure()
    for name, (fpr, tpr, auc_val) in all_rocs.items():
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})")
    plt.plot([0,1],[0,1],'k--',alpha=0.5)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curves (Validation, all folds aggregated) - {DATASET}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # 3) Classification HeatMaps (Confusion Matrices in %)
    num_cls = len(class_names)
    for name, (y_all, p_all) in all_yprobs.items():
        preds = p_all.argmax(axis=1)
        cm = confusion_matrix(y_all, preds, labels=list(range(num_cls)))
        cm_sum = cm.sum(axis=1, keepdims=True) + 1e-8
        cm_percent = cm.astype(float) / cm_sum * 100.0

        plt.figure(figsize=(6,5))
        im = plt.imshow(cm_percent, interpolation='nearest', cmap="Blues", vmin=0, vmax=100)
        plt.title(f"Confusion Matrix (%) - {name} ({DATASET})")
        plt.colorbar(im, fraction=0.046, pad=0.04)
        tick_marks = np.arange(num_cls)
        plt.xticks(tick_marks, class_names, rotation=45, ha="right")
        plt.yticks(tick_marks, class_names)

        thresh = cm_percent.max() / 2.
        for i in range(num_cls):
            for j in range(num_cls):
                plt.text(j, i, f"{cm_percent[i, j]:.1f}",
                         ha="center", va="center",
                         color="white" if cm_percent[i, j] > thresh else "black",
                         fontsize=8)
        plt.ylabel("True label")
        plt.xlabel("Predicted label")
        plt.tight_layout()
        plt.show()

if __name__ == "__main__":
    main()


### On HHAR

In [ ]:
#!/usr/bin/env python3
"""
HAR with QTModel vs Baseline 1D CNN models on WISDM / UCI HAR / HHAR.

- Added HHAR (Heterogeneity Human Activity Recognition) dataset support.
  * Auto-download from current UCI links (and legacy mirrors).
  * Browser-like User-Agent for reliability.
  * Loads Phones_accelerometer.csv (fallback to Watch_accelerometer.csv).
  * Sliding-window segmentation on accelerometer magnitude.

QTModel:
 - Strong 1D CNN backbone (64-d features)
 - Stacked quantum blocks:
     * Linear -> angles for Q/K/V
     * TorchQuantum QFM for Q, K, V
     * Self-attention after each QFM block (element-wise Q·K gating of V)
     * Transformer-style FFN + LayerNorm with residuals
 - Final quantum gate (extra QFM) on top of last features
 - Fused [features + gate] -> MLP classifier

Baselines:
 - ResNet1D (DeepILS)
 - EfficientNetB0_1D
 - MnasNet_1D
 - MobileNet_1D
 - MobileNetV2_1D
 - TSLANet

Outputs:
 - 6-fold CV metrics for each model
 - Training Loss Curves (mean over folds)
 - ROC curves (Validation, AUC)
 - Classification HeatMaps (confusion matrices in %, per model)
"""

import os
import urllib.request
from pathlib import Path
import zipfile
import warnings
warnings.filterwarnings("ignore")

import psutil
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR

from ptflops import get_model_complexity_info

from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, confusion_matrix
)
from sklearn.preprocessing import label_binarize

import matplotlib.pyplot as plt
from tqdm import trange

import torchquantum as tq

# -----------------------------------------------------------------------------
# GLOBAL CONFIG
# -----------------------------------------------------------------------------

DATASET = "HHAR"  # "WISDM" or "UCIHAR" or "HHAR"

QSize = 4                # number of quantum wires
Q_LAYERS = 4             # depth of quantum feature map
Q_BLOCKS = 4             # number of quantum blocks
EPOCHS = 40              # training epochs per fold
BATCH_SIZE = 256         # batch size
N_SPLITS = 6             # K-fold CV

# Default segmentation; per-dataset loaders may override WINDOW_SIZE
WINDOW_SIZE = 128
STEP_SIZE = 64

DATA_DIR = Path(os.environ.get("DATA_DIR", "../data"))
DATA_DIR.mkdir(exist_ok=True)

# WISDM
WISDM_URL = (
    "https://raw.githubusercontent.com/soham97/WISD_HAR_files/master/"
    "WISDM_ar_v1.1_raw.txt"
)
WISDM_PATH = DATA_DIR / "WISDM_ar_v1.1_raw.txt"

# UCI HAR
UCIHAR_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "00240/UCI%20HAR%20Dataset.zip"
)
UCIHAR_ZIP_PATH = DATA_DIR / "UCI_HAR_Dataset.zip"
UCIHAR_EXTRACT_DIR = DATA_DIR / "UCI_HAR_Dataset"

# --- HHAR (Heterogeneity HAR) -------------------------------------------------
# Current UCI static endpoints (new site) + legacy mirrors (old site).
# We only need "Activity recognition exp.zip" for Phones_*.csv / Watch_*.csv
HHAR_BIG_ZIP_URLS = [
    # New UCI bundle (name sometimes encoded with '+' chars)
    "https://archive.ics.uci.edu/static/public/344/heterogeneity%2Bactivity%2Brecognition.zip",
]
HHAR_COMPONENT_ZIPS = [
    # Preferred component zip (contains Phones_accelerometer.csv etc.)
    "https://archive.ics.uci.edu/static/public/344/activity%20recognition%20exp.zip",
    "https://archive.ics.uci.edu/static/public/344/activity%2Brecognition%2Bexp.zip",
    # Legacy mirrors (old ml/ path)
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00342/Activity%20recognition%20exp.zip",
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00342/Activity%20Recognition%20from%20Smartphones%20and%20Smartwatches.zip",
]
HHAR_ZIP_PATH = DATA_DIR / "HHAR.zip"                   # if the single big zip is used
HHAR_AR_ZIP_PATH = DATA_DIR / "HHAR_activity_exp.zip"   # component zip
HHAR_EXTRACT_DIR = DATA_DIR / "HHAR"

# -----------------------------------------------------------------------------
# 1) DEVICE SELECTION (fixed cuda:0 if available)
# -----------------------------------------------------------------------------

if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
    print("Using device: cuda:0")
else:
    DEVICE = torch.device("cpu")
    print("CUDA not available, using CPU")

ram = psutil.virtual_memory()
print(f"RAM Usage: {(ram.total-ram.available)/1e9:.1f}/{ram.total/1e9:.1f} GiB ({ram.percent}%)\n")

# -----------------------------------------------------------------------------
# 2) UTILITIES
# -----------------------------------------------------------------------------

def _urlretrieve_with_headers(url: str, dst: Path):
    """UCI sometimes returns 400 unless a browser-like UA is sent."""
    req = urllib.request.Request(
        url,
        headers={
            "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) "
                          "AppleWebKit/537.36 (KHTML, like Gecko) "
                          "Chrome/120.0.0.0 Safari/537.36"
        },
    )
    with urllib.request.urlopen(req) as resp, open(dst, "wb") as f:
        f.write(resp.read())

def _download_first_ok(urls, dst_path: Path) -> bool:
    for u in urls:
        try:
            print(f"[INFO] Trying {u}")
            _urlretrieve_with_headers(u, dst_path)
            print(f"[INFO] Downloaded to {dst_path}")
            return True
        except Exception as e:
            print(f"[WARN] Failed: {e}")
    return False

def _ensure_unzip(zip_path: Path, out_dir: Path):
    print(f"[INFO] Extracting {zip_path} -> {out_dir}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(out_dir)
    print("[INFO] Extraction complete.")

# -----------------------------------------------------------------------------
# 3) WISDM & UCI HAR LOADING / PREPROCESSING
# -----------------------------------------------------------------------------

def download_wisdm():
    if WISDM_PATH.exists():
        print(f"[INFO] Found WISDM file at {WISDM_PATH}")
        return
    print(f"[INFO] Downloading WISDM from {WISDM_URL} ...")
    urllib.request.urlretrieve(WISDM_URL, WISDM_PATH)
    print("[INFO] Download complete.")

def load_wisdm_dataframe(path: Path) -> pd.DataFrame:
    """Load and clean WISDM dataset."""
    print("[INFO] Loading WISDM data into DataFrame...")
    columns = ["user", "activity", "timestamp", "x", "y", "z"]
    df = pd.read_csv(
        path,
        header=None,
        names=columns,
        engine="python",
        on_bad_lines="skip",
    )

    df = df.dropna()
    df["z"] = df["z"].astype(str).str.replace(";", "", regex=False)
    df["z"] = df["z"].astype(float)
    df = df[df["timestamp"] != 0]
    df = df.sort_values(by=["user", "timestamp"], ignore_index=True)

    print("[INFO] Cleaned WISDM shape:", df.shape)
    return df

def segment_wisdm(df: pd.DataFrame,
                  window_size: int = WINDOW_SIZE,
                  step_size: int = STEP_SIZE):
    """Segment accelerometer data into windows and compute magnitude."""
    print("[INFO] Segmenting WISDM time series into windows...")
    X_segments = []
    y_labels = []

    users = df["user"].unique()
    for user in users:
        df_user = df[df["user"] == user]
        activities = df_user["activity"].unique()

        for act in activities:
            df_ua = df_user[df_user["activity"] == act]
            signal = df_ua[["x", "y", "z"]].values  # [T, 3]

            for start in range(0, len(signal) - window_size, step_size):
                end = start + window_size
                window = signal[start:end]  # [window_size, 3]
                mag = np.sqrt((window ** 2).sum(axis=1))  # [window_size]
                X_segments.append(mag)
                y_labels.append(act)

    X_segments = np.array(X_segments, dtype=np.float32)
    y_labels = np.array(y_labels)
    print("[INFO] WISDM Segmented X shape:", X_segments.shape)
    print("[INFO] WISDM Label distribution (raw):\n", pd.Series(y_labels).value_counts())
    return X_segments, y_labels

def download_uci_har():
    if UCIHAR_EXTRACT_DIR.exists():
        print(f"[INFO] Found UCI HAR at {UCIHAR_EXTRACT_DIR}")
        return
    if not UCIHAR_ZIP_PATH.exists():
        print(f"[INFO] Downloading UCI HAR from {UCIHAR_URL} ...")
        urllib.request.urlretrieve(UCIHAR_URL, UCIHAR_ZIP_PATH)
        print("[INFO] Download complete.")
    _ensure_unzip(UCIHAR_ZIP_PATH, UCIHAR_EXTRACT_DIR)

def load_uci_har_magnitude():
    """
    Use body_acc_x/y/z from UCI HAR to build magnitude windows:
      - Each window length = 128
      - Combine train + test
      - Labels: 6 activities (0..5)
    """
    base = UCIHAR_EXTRACT_DIR / "UCI HAR Dataset"

    def load_split(split: str):
        split_dir = base / split
        sig_dir = split_dir / "Inertial Signals"

        # Each: [N_windows, 128]
        x = np.loadtxt(sig_dir / f"body_acc_x_{split}.txt")
        y = np.loadtxt(sig_dir / f"body_acc_y_{split}.txt")
        z = np.loadtxt(sig_dir / f"body_acc_z_{split}.txt")

        mag = np.sqrt(x**2 + y**2 + z**2).astype(np.float32)  # [N, 128]

        # Labels are 1..6 -> 0..5
        y_lab = np.loadtxt(split_dir / f"y_{split}.txt").astype(int) - 1
        return mag, y_lab

    X_train, y_train = load_split("train")
    X_test,  y_test  = load_split("test")

    X_all = np.concatenate([X_train, X_test], axis=0)
    y_all = np.concatenate([y_train, y_test], axis=0)

    print("[INFO] UCI HAR magnitude shape:", X_all.shape)
    print("[INFO] UCI HAR label distribution:",
          dict(zip(*np.unique(y_all, return_counts=True))))
    return X_all, y_all

# -----------------------------------------------------------------------------
# 4) HHAR DOWNLOAD & LOADING
# -----------------------------------------------------------------------------

def ensure_hhar():
    """
    Ensure HHAR is present locally under HHAR_EXTRACT_DIR.
    Strategy:
      1) If already extracted & files exist -> return
      2) Try component 'Activity recognition exp.zip' (new UCI static link, then legacy)
      3) As a fallback, try the big combined zip (new UCI static link)
    After extraction, verify Phones_accelerometer.csv / Watch_accelerometer.csv.
    """
    # Already extracted and contains target files?
    for candidate_root in [HHAR_EXTRACT_DIR, DATA_DIR]:
        phones = list(candidate_root.rglob("Phones_accelerometer.csv"))
        watches = list(candidate_root.rglob("Watch_accelerometer.csv"))
        if phones or watches:
            print(f"[INFO] Found HHAR in {candidate_root}")
            return

    HHAR_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    # 1) Try component zip first (smaller; contains the needed files)
    if not HHAR_AR_ZIP_PATH.exists():
        print("[INFO] Downloading HHAR 'Activity recognition exp.zip' ...")
        ok = _download_first_ok(HHAR_COMPONENT_ZIPS, HHAR_AR_ZIP_PATH)
        if not ok:
            print("[WARN] Could not get component zip. Will try big bundle next.")

    if HHAR_AR_ZIP_PATH.exists():
        _ensure_unzip(HHAR_AR_ZIP_PATH, HHAR_EXTRACT_DIR)

    # 2) Fallback: try big bundle if files still not found
    phones = list(HHAR_EXTRACT_DIR.rglob("Phones_accelerometer.csv"))
    watches = list(HHAR_EXTRACT_DIR.rglob("Watch_accelerometer.csv"))
    if not phones and not watches:
        if not HHAR_ZIP_PATH.exists():
            print("[INFO] Downloading HHAR 'heterogeneity+activity+recognition.zip' (bundle) ...")
            ok = _download_first_ok(HHAR_BIG_ZIP_URLS, HHAR_ZIP_PATH)
            if not ok:
                raise FileNotFoundError(
                    "Could not download HHAR automatically from the new UCI site.\n"
                    "Manual fallback: place either\n"
                    "  1) '../data/HHAR_activity_exp.zip' (Activity recognition exp.zip) or\n"
                    "  2) '../data/HHAR.zip' (full bundle)\n"
                    "and re-run."
                )
        _ensure_unzip(HHAR_ZIP_PATH, HHAR_EXTRACT_DIR)

    # Final verification
    phones = list(HHAR_EXTRACT_DIR.rglob("Phones_accelerometer.csv"))
    watches = list(HHAR_EXTRACT_DIR.rglob("Watch_accelerometer.csv"))
    if not phones and not watches:
        raise FileNotFoundError(
            "Extracted HHAR but could not find Phones_accelerometer.csv or Watch_accelerometer.csv.\n"
            f"Checked under: {HHAR_EXTRACT_DIR}\n"
            "If your zip extracts into a differently named folder, move contents under ../data/HHAR/"
        )

def _parse_hhar_timestamp(s: pd.Series) -> pd.Series:
    """
    HHAR timestamps are often epoch millis. If values look large, parse with unit='ms',
    else let pandas infer.
    """
    s_clean = pd.to_numeric(s, errors="coerce")
    if s_clean.notna().any() and s_clean.dropna().median() > 1e11:
        return pd.to_datetime(s_clean, unit="ms", utc=True)
    # fallback: generic parse
    return pd.to_datetime(s, errors="coerce", utc=True)

def _read_hhar_csv(prefer_watch=False) -> pd.DataFrame:
    """
    Read HHAR accelerometer CSV (Phones_* preferred; fallback to Watch_*).
    Returns a tidy DataFrame with columns:
      ['user','activity','timestamp','x','y','z','device']
    """
    # Search both extracted root and data dir for flexibility
    roots = [HHAR_EXTRACT_DIR, DATA_DIR]
    phones_paths, watch_paths = [], []
    for r in roots:
        phones_paths += list(r.rglob("Phones_accelerometer.csv"))
        watch_paths  += list(r.rglob("Watch_accelerometer.csv"))

    use_watch = prefer_watch or (not phones_paths and watch_paths)
    if use_watch and not watch_paths:
        raise FileNotFoundError("Watch_accelerometer.csv not found")
    if not use_watch and not phones_paths:
        raise FileNotFoundError("Phones_accelerometer.csv not found")

    path = (watch_paths[0] if use_watch else phones_paths[0])
    print(f"[INFO] Loading HHAR from: {path.name} ({path.parent})")

    df = pd.read_csv(path)
    # Typical HHAR column names: Index, Arrival_Time, Creation_Time, x, y, z, User, Model, Device, gt
    cand = {c.lower(): c for c in df.columns}
    def pick(*names):
        for n in names:
            if n in cand: return cand[n]
        return None
    user_c = pick("user","subject","uid")
    act_c  = pick("gt","activity","act","label")   # HHAR usually uses 'gt'
    t_c    = pick("arrival_time","creation_time","timestamp","time","datetime")
    x_c    = pick("x","x_acc","acc_x")
    y_c    = pick("y","y_acc","acc_y")
    z_c    = pick("z","z_acc","acc_z")
    d_c    = pick("device","model","dev","brand")

    required = [user_c, act_c, t_c, x_c, y_c, z_c]
    if any(v is None for v in required):
        raise ValueError(f"Unrecognized HHAR columns. Found: {list(df.columns)}")

    out = pd.DataFrame({
        "user": df[user_c].astype(str),
        "activity": df[act_c].astype(str),
        "timestamp": _parse_hhar_timestamp(df[t_c]),
        "x": pd.to_numeric(df[x_c], errors="coerce"),
        "y": pd.to_numeric(df[y_c], errors="coerce"),
        "z": pd.to_numeric(df[z_c], errors="coerce"),
        "device": (df[d_c].astype(str) if d_c else ("watch" if use_watch else "phone"))
    }).dropna(subset=["timestamp","x","y","z"]).sort_values(
        ["user","activity","timestamp"], kind="mergesort", ignore_index=True
    )
    print("[INFO] HHAR tidy shape:", out.shape)
    return out

def segment_hhar(df: pd.DataFrame,
                 window_size: int = 128,
                 step_size: int = 64):
    """
    Sliding-window segmentation per (user, activity, device) for HHAR accelerometer.
    Uses magnitude = sqrt(x^2 + y^2 + z^2). Returns X_windows, y_labels (activity string).
    """
    print("[INFO] Segmenting HHAR into windows...")
    X_segments, y_labels = [], []

    # Within each contiguous (user, activity, device) block, slide windows
    for (user, act, dev), grp in df.groupby(["user","activity","device"], sort=False):
        sig = grp[["x","y","z"]].to_numpy(dtype=np.float32)
        n = sig.shape[0]
        if n < window_size:
            continue
        for start in range(0, n - window_size + 1, step_size):
            end = start + window_size
            win = sig[start:end]
            mag = np.sqrt((win**2).sum(axis=1))
            X_segments.append(mag)
            y_labels.append(act)

    X = np.asarray(X_segments, dtype=np.float32)
    y = np.asarray(y_labels, dtype=object)
    print("[INFO] HHAR segmented windows:", X.shape)
    print("[INFO] HHAR raw label counts:\n", pd.Series(y).value_counts())
    return X, y

# -----------------------------------------------------------------------------
# 5) BALANCING & DATASET CLASS
# -----------------------------------------------------------------------------

def balance_by_min_class(X: np.ndarray, y_int: np.ndarray, seed: int = 42):
    """
    Downsample all classes to the smallest class size.
    """
    rng = np.random.default_rng(seed)
    unique_classes = np.unique(y_int)
    indices_per_class = {c: np.where(y_int == c)[0] for c in unique_classes}
    class_sizes = {c: len(idx) for c, idx in indices_per_class.items()}
    min_count = min(class_sizes.values())

    print("[INFO] Class sizes before balancing:", class_sizes)
    print("[INFO] Using min_count =", min_count, "samples per class")

    balanced_indices_list = []
    for c in unique_classes:
        idxs = indices_per_class[c]
        if len(idxs) > min_count:
            chosen = rng.choice(idxs, size=min_count, replace=False)
        else:
            chosen = idxs
        balanced_indices_list.append(chosen)

    balanced_indices = np.concatenate(balanced_indices_list)
    rng.shuffle(balanced_indices)

    balanced_counts = np.bincount(y_int[balanced_indices])
    print("[INFO] Class sizes after balancing:",
          dict(zip(unique_classes, balanced_counts)))
    return X[balanced_indices], y_int[balanced_indices]

class WISDMDataset(Dataset):
    """
    Generic window-level dataset for time-series magnitude signals.
    Each sample: x -> [1, L], y -> integer label.
    """

    def __init__(self, X_windows: np.ndarray, y_int: np.ndarray):
        X_tensor = torch.tensor(X_windows, dtype=torch.float32)
        y_tensor = torch.tensor(y_int, dtype=torch.long)

        # Per-window normalization
        means = X_tensor.mean(dim=1, keepdim=True)
        stds = X_tensor.std(dim=1, keepdim=True) + 1e-8
        X_tensor = (X_tensor - means) / stds

        self.X = X_tensor
        self.y = y_tensor

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx].unsqueeze(0)  # [1, L]
        y = self.y[idx]
        return x, y

# -----------------------------------------------------------------------------
# 6) MODELS (CNN/QTModel + Baselines)
# -----------------------------------------------------------------------------

class EnhancedCNN1D(nn.Module):
    """
    Strong 1D CNN backbone for QTModel.
    Input:  [B, 1, L]
    Output: [B, 64]
    """
    def __init__(self, in_ch: int = 1):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv1d(in_ch, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2)   # L -> L/2
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2)   # L/2 -> L/4
        )
        self.block3 = nn.Sequential(
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)  # [B,64,1]
        )
        self.dropout = nn.Dropout(p=0.2)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.dropout(x)
        return x.flatten(1)  # [B, 64]

class QuantumFeatureMapTQ(tq.QuantumModule):
    """
    TorchQuantum-based feature map:
     - Angle embedding via RY
     - Ring entangling CNOT chain
     - Measure all Z
    """
    def __init__(self, n_wires=4, n_layers=4):
        super().__init__()
        self.n_wires  = n_wires
        self.n_layers = n_layers
        self.measure  = tq.MeasureAll(tq.PauliZ)

    def forward(self, qdev, angles):
        # angles: (batch, n_wires)
        for w in range(self.n_wires):
            qdev.ry(wires=w, params=angles[:, w], static=self.static_mode)
        for _ in range(self.n_layers):
            for i in range(self.n_wires - 1):
                qdev.cnot(wires=[i, i+1])
            qdev.cnot(wires=[self.n_wires-1, 0])
        return self.measure(qdev)  # (batch, n_wires)

class QBlockQT(tq.QuantumModule):
    """
    Single quantum block for QTModel (QFM + element-wise attention + FFN).
    """
    def __init__(self, dim=64, n_wires=4, n_layers=4, ff_mult=2, dropout=0.1):
        super().__init__()
        self.dim     = dim
        self.n_wires = n_wires

        self.q_fm = QuantumFeatureMapTQ(n_wires, n_layers)
        self.k_fm = QuantumFeatureMapTQ(n_wires, n_layers)
        self.v_fm = QuantumFeatureMapTQ(n_wires, n_layers)

        self.reduce = nn.Linear(dim, 3 * n_wires)

        self.q_proj = nn.Linear(n_wires, dim)
        self.k_proj = nn.Linear(n_wires, dim)
        self.v_proj = nn.Linear(n_wires, dim)

        self.ff = nn.Sequential(
            nn.Linear(dim, ff_mult * dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_mult * dim, dim),
        )

        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        B, D = x.shape
        r = self.reduce(x)
        q_ang, k_ang, v_ang = r.chunk(3, dim=1)

        qdev = tq.QuantumDevice(n_wires=self.n_wires, bsz=B, device=x.device)

        q_out = self.q_fm(qdev, q_ang)
        k_out = self.k_fm(qdev, k_ang)
        v_out = self.v_fm(qdev, v_ang)

        q = self.q_proj(q_out)
        k = self.k_proj(k_out)
        v = self.v_proj(v_out)

        scale = float(D) ** 0.5
        scores = torch.softmax((q * k) / scale, dim=-1)
        attn_out = scores * v

        z = x + attn_out
        z_norm = self.norm1(z)
        ff_out = self.ff(z_norm)
        y = self.norm2(z_norm + ff_out)
        return y

class QTModel(nn.Module):
    """
    CNN -> stacked QBlocks -> final quantum gate -> classifier
    """
    def __init__(self, num_cls, in_ch=1,
                 n_blocks=Q_BLOCKS, n_wires=QSize, n_layers=Q_LAYERS,
                 dim=64, ff_mult=2, dropout=0.1):
        super().__init__()

        self.dim      = dim
        self.n_wires  = n_wires
        self.cnn      = EnhancedCNN1D(in_ch)

        self.blocks = nn.ModuleList([
            QBlockQT(dim=dim, n_wires=n_wires, n_layers=n_layers,
                     ff_mult=ff_mult, dropout=dropout)
            for _ in range(n_blocks)
        ])

        self.gate_linear = nn.Linear(dim, n_wires)
        self.gate_qfm    = QuantumFeatureMapTQ(n_wires, n_layers)

        self.cls_head = nn.Sequential(
            nn.LayerNorm(dim + n_wires),
            nn.Linear(dim + n_wires, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, num_cls),
        )

    def forward(self, x):
        e = self.cnn(x)
        for block in self.blocks:
            e = block(e)

        angles = self.gate_linear(e)
        qdev = tq.QuantumDevice(n_wires=self.n_wires, bsz=angles.size(0), device=angles.device)
        q_gate = self.gate_qfm(qdev, angles)

        fused = torch.cat([e, q_gate], dim=1)
        logits = self.cls_head(fused)
        return logits

# --- Baselines (ResNet1D, EfficientNetB0_1D, MnasNet_1D, MobileNet_1D, MobileNetV2_1D, TSLANet)
def conv_dw(in_planes, out_planes, kernel_size, stride=1, dilation=1):
    return nn.Sequential(
        nn.Conv1d(in_planes, in_planes, kernel_size=kernel_size,
                  stride=stride, padding=(kernel_size//2)*dilation, dilation=dilation,
                  groups=in_planes, bias=False),
        nn.Conv1d(in_planes, out_planes, kernel_size=1, bias=False)
    )

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool1d(1)
        self.max = nn.AdaptiveMaxPool1d(1)
        red = max(1, in_planes // ratio)
        self.fc = nn.Sequential(
            nn.Conv1d(in_planes, red, 1, bias=False), nn.ReLU(),
            nn.Conv1d(red, in_planes, 1, bias=False)
        )
        self.sig = nn.Sigmoid()
    def forward(self, x):
        return self.sig(self.fc(self.avg(x)) + self.fc(self.max(x)))

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv1 = nn.Conv1d(2, 1, k, padding=k//2, bias=False)
        self.sig = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sig(self.conv1(torch.cat([avg_out, max_out], dim=1)))

class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, out_planes, k, stride=1, dilation=1, downsample=None):
        super().__init__()
        self.conv1 = conv_dw(in_planes, out_planes, k, stride, dilation)
        self.bn1 = nn.BatchNorm1d(out_planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv_dw(out_planes, out_planes, k)
        self.bn2 = nn.BatchNorm1d(out_planes)
        self.ca = ChannelAttention(out_planes)
        self.sa = SpatialAttention()
        self.downsample = downsample
    def forward(self, x):
        res = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.ca(out)*out
        out = self.sa(out)*out
        if self.downsample is not None:
            res = self.downsample(x)
        return self.relu(out + res)

class DeepILS(nn.Module):
    def __init__(self, num_inputs=1, num_classes=6, block=BasicBlock1D, group_sizes=(2,2,2,2), base=64, k=3):
        super().__init__()
        self.inp = nn.Sequential(
            nn.Conv1d(num_inputs, base, 5, 2, 3, bias=False),
            nn.BatchNorm1d(base),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(3,2,1)
        )
        self.inplanes = base
        self.groups = nn.ModuleList()
        planes = [base*(2**i) for i in range(len(group_sizes))]
        strides = [1] + [2]*(len(group_sizes)-1)
        for i, blocks in enumerate(group_sizes):
            pl = planes[i]
            st = strides[i]
            down = None
            if st != 1 or self.inplanes != pl*block.expansion:
                down = nn.Sequential(
                    nn.Conv1d(self.inplanes, pl*block.expansion, 1, st, bias=False),
                    nn.BatchNorm1d(pl*block.expansion)
                )
            layers = [block(self.inplanes, pl, k, stride=st, downsample=down)]
            self.inplanes = pl*block.expansion
            for _ in range(1, blocks):
                layers.append(block(self.inplanes, pl, k))
            self.groups.append(nn.Sequential(*layers))
        self.groups = nn.Sequential(*self.groups)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(planes[-1]*block.expansion, num_classes)
        )
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
    def forward(self, x):
        x = self.inp(x)
        x = self.groups(x)
        return self.head(x)

class ResNet1D(DeepILS):
    pass

class Swish(nn.Module):
    def forward(self, x): return x*torch.sigmoid(x)

def Conv_3x3(inp, oup, s):
    return nn.Sequential(
        nn.Conv1d(inp,oup,3,s,1,bias=False),
        nn.BatchNorm1d(oup),
        Swish()
    )

def Conv_1x1(inp, oup):
    return nn.Sequential(
        nn.Conv1d(inp,oup,1,1,0,bias=False),
        nn.BatchNorm1d(oup),
        Swish()
    )

class InvertedResidual_Eff(nn.Module):
    def __init__(self, inp, oup, s, t, k):
        super().__init__()
        self.use=(s==1 and inp==oup)
        hid=inp*t
        self.conv=nn.Sequential(
            nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), Swish(),
            nn.Conv1d(hid,hid,k,s,k//2,groups=hid,bias=False), nn.BatchNorm1d(hid), Swish(),
            nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
        )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

class EfficientNetB0_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        setting=[[1,16,1,1,3],[6,24,2,2,3],[6,40,2,2,5],
                 [6,80,3,2,3],[6,112,3,1,5],[6,192,4,2,5],[6,320,1,1,3]]
        in_ch=int(32*wm)
        last=1280 if wm<=1 else int(1280*wm)
        feats=[Conv_3x3(1,in_ch,2)]
        ch=in_ch
        for t,c,n,s,k in setting:
            oc=int(c*wm)
            for i in range(n):
                feats.append(InvertedResidual_Eff(ch, oc, s if i==0 else 1, t, k))
                ch=oc
        feats += [Conv_1x1(ch,last), nn.AdaptiveAvgPool1d(1)]
        self.features=nn.Sequential(*feats)
        self.fc=nn.Linear(last,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        return self.fc(self.features(x).squeeze(-1))

class InvertedResidual_Mnas(nn.Module):
    def __init__(self, inp, oup, s, t, k):
        super().__init__()
        self.use=(s==1 and inp==oup)
        hid=inp*t
        self.conv=nn.Sequential(
            nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
            nn.Conv1d(hid,hid,k,s,k//2,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
            nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
        )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

def SepConv_3x3(inp,oup):
    return nn.Sequential(
        nn.Conv1d(inp,inp,3,1,1,groups=inp,bias=False), nn.BatchNorm1d(inp), nn.ReLU6(inplace=True),
        nn.Conv1d(inp,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
    )

def Conv3_3(inp,oup,s):
    return nn.Sequential(
        nn.Conv1d(inp,oup,3,s,1,bias=False), nn.BatchNorm1d(oup), nn.ReLU6(inplace=True)
    )

def Conv1_1(inp,oup):
    return nn.Sequential(
        nn.Conv1d(inp,oup,1,1,0,bias=False), nn.BatchNorm1d(oup), nn.ReLU6(inplace=True)
    )

class MnasNet_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        setting=[[3,24,3,2,3],[3,40,3,2,5],[6,80,3,2,5],
                 [6,96,2,1,3],[6,192,4,2,5],[6,320,1,1,3]]
        in_ch=int(32*wm)
        last=1280 if wm<=1 else int(1280*wm)
        feats=[Conv3_3(1,in_ch,2), SepConv_3x3(in_ch,16)]
        ch=16
        for t,c,n,s,k in setting:
            oc=int(c*wm)
            for i in range(n):
                feats.append(InvertedResidual_Mnas(ch, oc, s if i==0 else 1, t, k))
                ch=oc
        feats += [Conv1_1(ch,last), nn.AdaptiveAvgPool1d(1)]
        self.features=nn.Sequential(*feats)
        self.fc=nn.Linear(last,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        return self.fc(self.features(x).squeeze(-1))

class DSConv_Mobile(nn.Module):
    def __init__(self, f3, f1, s=1, p=1):
        super().__init__()
        self.seq=nn.Sequential(
            nn.Conv1d(f3,f3,3,s,p,groups=f3,bias=False), nn.BatchNorm1d(f3), nn.ReLU(inplace=True),
            nn.Conv1d(f3,f1,1,1,0,bias=False), nn.BatchNorm1d(f1), nn.ReLU(inplace=True)
        )
    def forward(self,x): return self.seq(x)

class MobileNet_1D(nn.Module):
    def __init__(self, channels=[32,64,128,256,512,1024], wm=1.0, n_classes=6):
        super().__init__()
        ch=[int(c*wm) for c in channels]
        self.conv=nn.Sequential(
            nn.Conv1d(1,ch[0],3,2,1,bias=False), nn.BatchNorm1d(ch[0]), nn.ReLU(inplace=True)
        )
        self.features=nn.Sequential(
            DSConv_Mobile(ch[0],ch[1],1,1),
            DSConv_Mobile(ch[1],ch[2],2,1),
            DSConv_Mobile(ch[2],ch[2],1,1),
            DSConv_Mobile(ch[2],ch[3],2,1),
            DSConv_Mobile(ch[3],ch[3],1,1),
            DSConv_Mobile(ch[3],ch[4],2,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[5],2,1),
            DSConv_Mobile(ch[5],ch[5],1,1),
        )
        self.avg=nn.AdaptiveAvgPool1d(1)
        self.fc=nn.Linear(ch[5],n_classes)
    def forward(self,x):
        x=self.conv(x)
        x=self.features(x)
        x=self.avg(x).squeeze(-1)
        return self.fc(x)

class InvertedResidual_V2(nn.Module):
    def __init__(self, inp, oup, s, t):
        super().__init__()
        hid=int(inp*t)
        self.use=(s==1 and inp==oup)
        if t==1:
            self.conv=nn.Sequential(
                nn.Conv1d(hid,hid,3,s,1,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
            )
        else:
            self.conv=nn.Sequential(
                nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,hid,3,s,1,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
            )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

def make_divisible(x, d=8):
    return int((x + d - 1) // d * d)

class MobileNetV2_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        input_channel=32
        last=1280
        setting=[[1,16,1,1],[6,24,2,2],[6,32,3,2],[6,64,4,2],
                 [6,96,3,1],[6,160,3,2],[6,320,1,1]]
        self.last_ch=make_divisible(last*wm) if wm>1 else last
        feats=[nn.Conv1d(1,input_channel,3,2,1,bias=False), nn.BatchNorm1d(input_channel), nn.ReLU6(inplace=True)]
        ch=input_channel
        for t,c,n,s in setting:
            oc=make_divisible(c*wm) if t>1 else c
            for i in range(n):
                feats.append(InvertedResidual_V2(ch, oc, s if i==0 else 1, t))
                ch=oc
        feats += [nn.Conv1d(ch,self.last_ch,1,1,0,bias=False), nn.BatchNorm1d(self.last_ch), nn.ReLU6(inplace=True)]
        self.features=nn.Sequential(*feats)
        self.pool=nn.AdaptiveAvgPool1d(1)
        self.fc=nn.Linear(self.last_ch,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        x=self.features(x)
        x=self.pool(x).squeeze(-1)
        return self.fc(x)

class TSLANet(nn.Module):
    def __init__(self, C_in, T_in, n_classes=6, emb=128, depth=3, patch=8, drop=0.10):
        super().__init__()
        stride=max(1, patch//2)
        self.proj=nn.Conv1d(C_in,emb,patch,stride)
        N=int((T_in-patch)/stride+1)
        self.pos=nn.Parameter(torch.zeros(1,N,emb))
        self.blocks=nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(emb),
                nn.Identity(),  # spectral placeholder
                nn.LayerNorm(emb),
                nn.Sequential(
                    nn.Linear(emb,int(emb*3)), nn.GELU(),
                    nn.Linear(int(emb*3),emb)
                )
            ) for _ in range(depth)
        ])
        self.drop=drop
        self.head=nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(emb,emb//2),
            nn.GELU(),
            nn.Linear(emb//2,n_classes)
        )
    def forward(self,x):
        x=self.proj(x).flatten(2).transpose(1,2)
        x=x+self.pos
        x=F.dropout(x,p=self.drop,training=self.training)
        for b in self.blocks:
            x=x + b[3](b[1](b[0](x)))
        return self.head(x.mean(1))

# -----------------------------------------------------------------------------
# 7) TRAIN / EVAL HELPERS
# -----------------------------------------------------------------------------

def process_target(y: torch.Tensor) -> torch.Tensor:
    if y.dim() > 1:
        if y.size(1) > 1:
            y = y.argmax(dim=1)
        else:
            y = y.squeeze(1)
    return y.long()

def train_one(model, loader, opt):
    model.train()
    losses = []
    for x, y in loader:
        x = x.to(DEVICE)
        y = process_target(y.to(DEVICE))

        out = model(x)               # logits
        loss = F.cross_entropy(out, y)

        opt.zero_grad()
        loss.backward()
        opt.step()

        losses.append(loss.item())
    return float(np.mean(losses)) if losses else 0.0

def eval_fold(model, loader, num_cls):
    """Evaluate on one fold: return y_true and probabilities."""
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = process_target(y.to(DEVICE))
            out  = model(x)  # logits
            prob = F.softmax(out, dim=1).cpu().numpy()
            ys.append(y.cpu().numpy())
            ps.append(prob)
    ys = np.concatenate(ys)
    ps = np.concatenate(ps)
    return ys, ps

def run_cv_for_model(model_name, model_ctor, num_cls, dataset):
    N = len(dataset)
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=0)

    metrics = {"acc": [], "f1": [], "prec": [], "rec": [], "auc": []}
    loss_hist = []
    all_y = []
    all_probs = []

    print(f"\n====== Running {N_SPLITS}-fold CV for {model_name} ======")

    for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(N)), start=1):
        print(f"\n--- {model_name} Fold {fold}/{N_SPLITS} ---")

        train_loader = DataLoader(
            Subset(dataset, train_idx),
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=4,
            pin_memory=True
        )
        val_loader = DataLoader(
            Subset(dataset, val_idx),
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=4,
            pin_memory=True
        )

        model = model_ctor().to(DEVICE)
        opt   = optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-4)
        sch   = CosineAnnealingLR(opt, T_max=EPOCHS)

        fold_losses = []
        for _ in trange(EPOCHS, desc=f"{model_name}-F{fold}", leave=False):
            loss = train_one(model, train_loader, opt)
            sch.step()
            fold_losses.append(loss)
        loss_hist.append(fold_losses)

        y_fold, p_fold = eval_fold(model, val_loader, num_cls)
        preds = p_fold.argmax(axis=1)

        acc  = accuracy_score(y_fold, preds)
        f1   = f1_score(y_fold, preds, average="weighted")
        prec = precision_score(y_fold, preds, average="weighted")
        rec  = recall_score(y_fold, preds, average="weighted")

        if num_cls == 2:
            auc_val = roc_auc_score(y_fold, p_fold[:,1])
        else:
            y_bin = label_binarize(y_fold, classes=list(range(num_cls)))
            auc_val = roc_auc_score(y_bin, p_fold, average="micro", multi_class="ovo")

        print(f" Fold {fold}: Acc={acc:.4f} F1={f1:.4f} Prec={prec:.4f} Rec={rec:.4f} AUC={auc_val:.4f}")
        for k,v in zip(("acc","f1","prec","rec","auc"), (acc,f1,prec,rec,auc_val)):
            metrics[k].append(v)

        all_y.append(y_fold)
        all_probs.append(p_fold)

        del model
        torch.cuda.empty_cache()

    all_y = np.concatenate(all_y)
    all_probs = np.concatenate(all_probs, axis=0)

    # Global ROC for the model
    if num_cls == 2:
        fpr, tpr, _ = roc_curve(all_y, all_probs[:,1], pos_label=1)
        auc_global = roc_auc_score(all_y, all_probs[:,1])
    else:
        y_bin = label_binarize(all_y, classes=list(range(num_cls)))
        fpr, tpr, _ = roc_curve(y_bin.ravel(), all_probs.ravel())
        auc_global = roc_auc_score(y_bin, all_probs, average="micro", multi_class="ovo")

    return metrics, loss_hist, all_y, all_probs, (fpr, tpr, auc_global)

# -----------------------------------------------------------------------------
# 8) MAIN: DATA CHOICE + MODELS + PLOTS
# -----------------------------------------------------------------------------

def main():
    global WINDOW_SIZE, STEP_SIZE

    # -----------------------
    # Dataset choice
    # -----------------------
    if DATASET == "WISDM":
        WINDOW_SIZE, STEP_SIZE = 200, 100
        download_wisdm()
        df = load_wisdm_dataframe(WISDM_PATH)
        X_segments, y_labels = segment_wisdm(df, window_size=WINDOW_SIZE, step_size=STEP_SIZE)
        labels_cat = pd.Series(y_labels).astype("category")
        class_names = list(labels_cat.cat.categories)
        y_int = labels_cat.cat.codes.to_numpy().astype(np.int64)
        WINDOW_SIZE = X_segments.shape[1]

    elif DATASET == "UCIHAR":
        download_uci_har()
        X_segments, y_int = load_uci_har_magnitude()
        class_names = [
            "WALKING",
            "WALK_UP",
            "WALK_DOWN",
            "SITTING",
            "STANDING",
            "LAYING",
        ]
        WINDOW_SIZE = X_segments.shape[1]

    elif DATASET == "HHAR":
        # Reasonable defaults for ~50 Hz; adjust if you want denser windows
        WINDOW_SIZE, STEP_SIZE = 256, 128
        ensure_hhar()
        df = _read_hhar_csv(prefer_watch=False)  # set True to use watch data instead
        X_segments, y_labels = segment_hhar(df, window_size=WINDOW_SIZE, step_size=STEP_SIZE)

        # HHAR activities are strings; keep order stable via category
        labels_cat = pd.Series(y_labels, dtype="category")
        class_names = list(labels_cat.cat.categories)
        y_int = labels_cat.cat.codes.to_numpy().astype(np.int64)
        WINDOW_SIZE = X_segments.shape[1]

    else:
        raise ValueError(f"Unknown DATASET: {DATASET}")

    print("[INFO] Using dataset:", DATASET)
    print("[INFO] Classes:", class_names)

    # Balance dataset
    X_bal, y_bal = balance_by_min_class(X_segments, y_int, seed=42)
    dataset = WISDMDataset(X_bal, y_bal)
    N = len(dataset)
    print(f"[INFO] Balanced dataset size: {N}")

    num_cls = len(class_names)
    seq_len = X_bal.shape[1]
    dims = (1, seq_len)

    # ------------------------
    # Model factories
    # ------------------------
    model_factories = {
        "QTModel":           lambda: QTModel(num_cls=num_cls, in_ch=1),
        "ResNet1D":          lambda: ResNet1D(num_inputs=1, num_classes=num_cls),
        "EfficientNetB0_1D": lambda: EfficientNetB0_1D(n_classes=num_cls),
        "MnasNet_1D":        lambda: MnasNet_1D(n_classes=num_cls),
        "MobileNet_1D":      lambda: MobileNet_1D(n_classes=num_cls),
        "MobileNetV2_1D":    lambda: MobileNetV2_1D(n_classes=num_cls),
        "TSLANet":           lambda: TSLANet(C_in=1, T_in=seq_len, n_classes=num_cls),
    }

    # ------------------------
    # Complexity (params/FLOPs) for all models
    # ------------------------
    complexity = {}
    for name, ctor in model_factories.items():
        model = ctor().to(DEVICE)
        try:
            macs, params = get_model_complexity_info(
                model, dims,
                as_strings=True,
                print_per_layer_stat=False
            )
        except Exception as e:
            print(f"[WARN] ptflops failed on {name}: {e}")
            macs, params = "N/A", "N/A"
        complexity[name] = (params, macs)
        del model
        torch.cuda.empty_cache()
        print(f"[INFO] {name}: params={params}, FLOPs={macs}")

    # ------------------------
    # Run CV for each model
    # ------------------------
    all_metrics  = {}
    all_losses   = {}
    all_yprobs   = {}
    all_rocs     = {}

    for name, ctor in model_factories.items():
        metrics, loss_hist, y_all, p_all, roc_info = run_cv_for_model(
            name, ctor, num_cls, dataset
        )
        all_metrics[name] = metrics
        all_losses[name]  = loss_hist
        all_yprobs[name]  = (y_all, p_all)
        all_rocs[name]    = roc_info

    # Summary table
    gpu_mem = None
    if DEVICE.type == "cuda":
        gpu_mem = (torch.cuda.memory_allocated()/1e6,
                   torch.cuda.memory_reserved()/1e6)

    summary_rows = []
    for name in model_factories.keys():
        m = all_metrics[name]
        params, macs = complexity.get(name, ("N/A", "N/A"))

        row = {
            "model":        name,
            "dataset":      DATASET,
            "acc_mean±sd":  f"{np.mean(m['acc']):.4f}±{np.std(m['acc']):.4f}",
            "f1_mean±sd":   f"{np.mean(m['f1']):.4f}±{np.std(m['f1']):.4f}",
            "prec_mean±sd": f"{np.mean(m['prec']):.4f}±{np.std(m['prec']):.4f}",
            "rec_mean±sd":  f"{np.mean(m['rec']):.4f}±{np.std(m['rec']):.4f}",
            "auc_mean±sd":  f"{np.mean(m['auc']):.4f}±{np.std(m['auc']):.4f}",
            "GPU_mem_MB":   gpu_mem,
            "params":       params,
            "FLOPs":        macs,
        }
        summary_rows.append(row)

    df_sum = pd.DataFrame(summary_rows)
    print("\n[SUMMARY COMPARISON]")
    print(df_sum.to_markdown(index=False))

    # -------------------------------------------------
    # PLOTS
    # -------------------------------------------------

    # 1) Training Loss Curves (mean over folds) per model
    plt.figure()
    for name, loss_hist in all_losses.items():
        loss_arr = np.array(loss_hist)  # [n_folds, epochs]
        mean_loss = loss_arr.mean(axis=0)
        plt.plot(range(1, EPOCHS+1), mean_loss, label=name)
    plt.xlabel("Epoch")
    plt.ylabel("Training Loss")
    plt.title(f"Training Loss Curves (mean over folds) - {DATASET}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # 2) ROC curves per model (Validation)
    plt.figure()
    for name, (fpr, tpr, auc_val) in all_rocs.items():
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})")
    plt.plot([0,1],[0,1],'k--',alpha=0.5)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curves (Validation, all folds aggregated) - {DATASET}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # 3) Classification HeatMaps (Confusion Matrices in %)
    num_cls = len(class_names)
    for name, (y_all, p_all) in all_yprobs.items():
        preds = p_all.argmax(axis=1)
        cm = confusion_matrix(y_all, preds, labels=list(range(num_cls)))
        cm_sum = cm.sum(axis=1, keepdims=True) + 1e-8
        cm_percent = cm.astype(float) / cm_sum * 100.0

        plt.figure(figsize=(6,5))
        im = plt.imshow(cm_percent, interpolation='nearest', cmap="Blues", vmin=0, vmax=100)
        plt.title(f"Confusion Matrix (%) - {name} ({DATASET})")
        plt.colorbar(im, fraction=0.046, pad=0.04)
        tick_marks = np.arange(num_cls)
        plt.xticks(tick_marks, class_names, rotation=45, ha="right")
        plt.yticks(tick_marks, class_names)

        thresh = cm_percent.max() / 2.
        for i in range(num_cls):
            for j in range(num_cls):
                plt.text(j, i, f"{cm_percent[i, j]:.1f}",
                         ha="center", va="center",
                         color="white" if cm_percent[i, j] > thresh else "black",
                         fontsize=8)
        plt.ylabel("True label")
        plt.xlabel("Predicted label")
        plt.tight_layout()
        plt.show()

if __name__ == "__main__":
    main()
